# Modelo B — notebook maestro productivo A9

Orquestación Colab para el entrenamiento productivo congelado de Modelo B. Preflight y diagnóstico no entrenan; la cola requiere una acción explícita.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import importlib

EXPECTED_SHA = "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"
FIX_BRANCH = "fix/model-b-a9-position-diverse-batching"

REPO = Path("/content/model_b_workspace/repo")
SRC = REPO / "src"

print("=" * 80)
print("MODEL B A9 — CHECKOUT DEL FIX")
print("=" * 80)

assert REPO.is_dir(), f"REPO no existe: {REPO}"

def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO), *args],
        check=True,
        text=True,
        capture_output=True,
    )

# ------------------------------------------------------------------
# 1. No permitimos perder cambios locales.
# ------------------------------------------------------------------

status_before = git("status", "--porcelain").stdout.strip()

print("STATUS_BEFORE =", repr(status_before))

assert status_before == "", (
    "El repositorio de Colab tiene cambios locales. "
    "No se realizará checkout."
)

# ------------------------------------------------------------------
# 2. Descargar explícitamente la rama del fix.
# ------------------------------------------------------------------

fetch = git(
    "fetch",
    "origin",
    FIX_BRANCH,
)

print("FETCH = PASS")

# ------------------------------------------------------------------
# 3. Verificar que el SHA existe.
# ------------------------------------------------------------------

resolved = git(
    "rev-parse",
    f"{EXPECTED_SHA}^{{commit}}",
).stdout.strip()

print("RESOLVED_SHA =", resolved)

assert resolved == EXPECTED_SHA

# ------------------------------------------------------------------
# 4. Checkout detached exacto.
# ------------------------------------------------------------------

git(
    "checkout",
    "--detach",
    EXPECTED_SHA,
)

HEAD = git(
    "rev-parse",
    "HEAD",
).stdout.strip()

print("HEAD =", HEAD)

assert HEAD == EXPECTED_SHA, (
    f"Checkout incorrecto: {HEAD}"
)

status_after = git(
    "status",
    "--porcelain",
).stdout.strip()

assert status_after == ""

print("SCIENTIFIC_CHECKOUT = PASS")

# ------------------------------------------------------------------
# 5. Trabajar desde la raíz del repositorio.
# ------------------------------------------------------------------

os.chdir(REPO)

if str(SRC) in sys.path:
    sys.path.remove(str(SRC))

sys.path.insert(0, str(SRC))

# ------------------------------------------------------------------
# 6. Eliminar cualquier módulo gnn_siamese cacheado del commit viejo.
# ------------------------------------------------------------------

purged = []

for name in list(sys.modules):
    if name == "gnn_siamese" or name.startswith("gnn_siamese."):
        purged.append(name)
        del sys.modules[name]

importlib.invalidate_caches()

print("GNN_MODULES_PURGED =", len(purged))

# ------------------------------------------------------------------
# 7. Verificar entorno Python / PyTorch / PyG.
# ------------------------------------------------------------------

import torch
import torch_geometric
import gnn_siamese

print("torch =", torch.__version__)
print("torch_geometric =", torch_geometric.__version__)
print("cuda_available =", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU =",
        torch.cuda.get_device_name(0),
    )

print(
    "gnn_siamese file =",
    gnn_siamese.__file__,
)

expected_init = SRC / "gnn_siamese" / "__init__.py"

assert (
    Path(gnn_siamese.__file__).resolve()
    == expected_init.resolve()
)

print("CLEAN_PROJECT_IMPORT = PASS")

print()
print("=" * 80)
print("CHECKOUT_FIX = PASS")
print("RUN_PRODUCTIVE_QUEUE = False")
print("=" * 80)

MODEL B A9 — CHECKOUT DEL FIX


AssertionError: REPO no existe: /content/model_b_workspace/repo

## 1. Montaje de Drive y validación de locators inmutables

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_project = Path(DRIVE_PROJECT_BASE).resolve()
if not drive_project.is_dir():
    raise FileNotFoundError(f'DRIVE_PROJECT_BASE inexistente: {drive_project}')
if not DRIVE_MUTANTS_HDF5 or not DRIVE_WT_HDF5:
    raise ValueError('Defina los locators Drive reales de ambos HDF5 en PARÁMETROS OPERATIVOS')
for label, raw in (('mutants', DRIVE_MUTANTS_HDF5), ('WT companion', DRIVE_WT_HDF5)):
    path = Path(raw).resolve()
    if not path.is_file() or path.stat().st_size <= 0:
        raise FileNotFoundError(f'HDF5 {label} ausente o vacío: {path}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'DRIVE_PROJECT_BASE' is not defined

# RECOVERY AFTER RUNTIME RESET

In [ ]:
# =============================================================================
# MODEL B A9
# 0.1 — MOUNT DRIVE + FROZEN RECOVERY PARAMETERS
# =============================================================================

from pathlib import Path
import hashlib
import json
import os
import sys
import subprocess

from google.colab import drive


print("=" * 80)
print("MODEL B A9 — 0.1 MOUNT DRIVE + PARAMETERS")
print("=" * 80)


# =============================================================================
# SAFETY LOCK
# =============================================================================

RUN_PRODUCTIVE_QUEUE = False

assert RUN_PRODUCTIVE_QUEUE is False


# =============================================================================
# SCIENTIFIC CONTRACT
# =============================================================================

REPO_URL = "https://github.com/sap15/ViVU_lab.git"

EXPECTED_SHA = (
    "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"
)

FIX_BRANCH = (
    "fix/model-b-a9-position-diverse-batching"
)

PRODUCTIVE_SEEDS = (
    11,
    23,
    37,
    41,
    53,
)

SPLIT_SEED = 42


# =============================================================================
# DRIVE — PERSISTENT STORAGE
# =============================================================================

DRIVE_PROJECT_BASE = (
    "/content/drive/MyDrive/modelos_proyecto_PKP2"
)

DRIVE_MUTANTS_HDF5 = (
    DRIVE_PROJECT_BASE
    + "/model_a/data/proc_483p.hdf5"
)

DRIVE_WT_HDF5 = (
    DRIVE_PROJECT_BASE
    + "/model_a/data/wt_companion.hdf5"
)

MODEL_B_A9_ROOT = Path(
    DRIVE_PROJECT_BASE
) / "model_b/runs/model_b_a9"

MODEL_B_ARCH_ROOT = (
    MODEL_B_A9_ROOT
    / "model_b_graph_level_relational"
)


# =============================================================================
# B11 — IMMUTABLE REFERENCE RUN
# =============================================================================

B11_REFERENCE_RUN = (
    MODEL_B_ARCH_ROOT
    / "run_20260914T143850.128514Z-e382b31e"
)

B11_BEST_PT = (
    B11_REFERENCE_RUN
    / "checkpoints"
    / "best.pt"
)

B11_LAST_PT = (
    B11_REFERENCE_RUN
    / "checkpoints"
    / "last.pt"
)

B11_MANIFEST = (
    B11_REFERENCE_RUN
    / "run_manifest.json"
)

B11_ACCEPTANCE = (
    B11_REFERENCE_RUN
    / "a9_acceptance.json"
)

EXPECTED_B11_ACCEPTANCE_SHA256 = (
    "c9285e995fa069ec7fe9622909918f4f"
    "ac024dda8f3eea7ac6b137c08f2641e2"
)


# =============================================================================
# LOCAL EPHEMERAL WORKSPACE
# =============================================================================

LOCAL_ROOT = Path(
    "/content/model_b_workspace"
)

REPO = (
    LOCAL_ROOT
    / "repo"
)

SRC = (
    REPO
    / "src"
)

STAGING_ROOT = (
    LOCAL_ROOT
    / "staging_a9"
)

LOCAL_MUTANTS_HDF5 = (
    STAGING_ROOT
    / "proc_483p.hdf5"
)

LOCAL_WT_HDF5 = (
    STAGING_ROOT
    / "wt_companion.hdf5"
)

RUNTIME_CONFIG_ROOT = (
    LOCAL_ROOT
    / "runtime_configs"
)


# =============================================================================
# HELPER
# =============================================================================

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# =============================================================================
# MOUNT DRIVE
# =============================================================================

drive.mount(
    "/content/drive",
    force_remount=False,
)


# =============================================================================
# VALIDATE PERSISTENT INPUTS
# =============================================================================

assert Path(
    DRIVE_PROJECT_BASE
).is_dir()

assert Path(
    DRIVE_MUTANTS_HDF5
).is_file()

assert Path(
    DRIVE_WT_HDF5
).is_file()

assert MODEL_B_ARCH_ROOT.is_dir()

assert B11_REFERENCE_RUN.is_dir()

for path in (
    B11_BEST_PT,
    B11_LAST_PT,
    B11_MANIFEST,
    B11_ACCEPTANCE,
):

    assert path.is_file(), (
        f"Missing persistent B11 artifact: {path}"
    )

    assert path.stat().st_size > 0


# =============================================================================
# VERIFY IMMUTABLE B11 ACCEPTANCE
# =============================================================================

acceptance_sha = sha256_file(
    B11_ACCEPTANCE
)

print(
    "B11_ACCEPTANCE_SHA256 =",
    acceptance_sha,
)

assert (
    acceptance_sha
    == EXPECTED_B11_ACCEPTANCE_SHA256
)

acceptance = json.loads(
    B11_ACCEPTANCE.read_text(
        encoding="utf-8"
    )
)

assert (
    acceptance["status"]
    == "accepted"
)

assert (
    int(
        acceptance["run_seed"]
    )
    == 11
)

assert (
    acceptance["architecture"]
    == "model_b_graph_level_relational"
)


print()
print("DRIVE_MOUNT = PASS")
print("PERSISTENT_INPUTS = PASS")
print("B11_REFERENCE_ACCEPTANCE = PASS")
print("PRODUCTIVE_SEEDS =", PRODUCTIVE_SEEDS)
print("SPLIT_SEED =", SPLIT_SEED)
print("RUN_PRODUCTIVE_QUEUE =", RUN_PRODUCTIVE_QUEUE)

print("=" * 80)

MODEL B A9 — 0.1 MOUNT DRIVE + PARAMETERS
Mounted at /content/drive
B11_ACCEPTANCE_SHA256 = c9285e995fa069ec7fe9622909918f4fac024dda8f3eea7ac6b137c08f2641e2

DRIVE_MOUNT = PASS
PERSISTENT_INPUTS = PASS
B11_REFERENCE_ACCEPTANCE = PASS
PRODUCTIVE_SEEDS = (11, 23, 37, 41, 53)
SPLIT_SEED = 42
RUN_PRODUCTIVE_QUEUE = False


In [ ]:
# =============================================================================
# MODEL B A9
# 0.2 — RECOVER EXACT SCIENTIFIC GIT CHECKOUT
# =============================================================================

from pathlib import Path
import subprocess
import datetime
import shutil
import os
import sys
import importlib


print("=" * 80)
print("MODEL B A9 — 0.2 RECOVER SCIENTIFIC CHECKOUT")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False

LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# INSPECT EXISTING PATH
# =============================================================================

print("REPO =", REPO)
print(
    "REPO_EXISTS =",
    REPO.exists(),
)

print(
    "GIT_EXISTS =",
    (REPO / ".git").is_dir(),
)


# =============================================================================
# EXISTING VALID REPOSITORY
# =============================================================================

if (REPO / ".git").is_dir():

    result = subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "status",
            "--porcelain",
        ],
        text=True,
        capture_output=True,
    )

    assert result.returncode == 0

    status = (
        result.stdout.strip()
    )

    print(
        "EXISTING_WORKING_TREE =",
        repr(status),
    )

    assert status == "", (
        "STOP: repository contains local changes."
    )


# =============================================================================
# INVALID /content DIRECTORY
# =============================================================================

elif REPO.exists():

    stamp = (
        datetime.datetime.now()
        .strftime("%Y%m%dT%H%M%S")
    )

    preserved = (
        LOCAL_ROOT
        / f"repo_invalid_{stamp}"
    )

    shutil.move(
        str(REPO),
        str(preserved),
    )

    print(
        "INVALID_REPO_PRESERVED_AS =",
        preserved,
    )


# =============================================================================
# CLONE WHEN NEEDED
# =============================================================================

if not (REPO / ".git").is_dir():

    result = subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO),
        ],
        text=True,
        capture_output=True,
    )

    print(
        "CLONE_RETURN_CODE =",
        result.returncode,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    assert result.returncode == 0

    print("REPO_CLONE = PASS")


# =============================================================================
# FAIL-CLOSED GIT HELPER
# =============================================================================

def git(*args):

    result = subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            *args,
        ],
        text=True,
        capture_output=True,
    )

    if result.returncode != 0:

        print(
            "FAILED GIT COMMAND:",
            "git",
            *args,
        )

        print(
            "STDOUT =",
            result.stdout,
        )

        print(
            "STDERR =",
            result.stderr,
        )

        raise RuntimeError(
            "Git recovery failed."
        )

    return result


# =============================================================================
# FETCH EXACT FIX
# =============================================================================

git(
    "fetch",
    "origin",
    FIX_BRANCH,
)

resolved = git(
    "rev-parse",
    f"{EXPECTED_SHA}^{{commit}}",
).stdout.strip()

print(
    "RESOLVED_SHA =",
    resolved,
)

assert resolved == EXPECTED_SHA


# =============================================================================
# EXACT DETACHED CHECKOUT
# =============================================================================

git(
    "checkout",
    "--detach",
    EXPECTED_SHA,
)

HEAD = git(
    "rev-parse",
    "HEAD",
).stdout.strip()

status = git(
    "status",
    "--porcelain",
).stdout.strip()

print(
    "HEAD =",
    HEAD,
)

print(
    "WORKING_TREE =",
    repr(status),
)

assert HEAD == EXPECTED_SHA
assert status == ""


# =============================================================================
# RESET PYTHON IMPORT PATH
# =============================================================================

os.chdir(
    REPO
)

for candidate in (
    str(REPO),
    str(SRC),
):

    while candidate in sys.path:

        sys.path.remove(
            candidate
        )

sys.path.insert(
    0,
    str(REPO),
)

sys.path.insert(
    0,
    str(SRC),
)


for name in list(
    sys.modules
):

    if (
        name == "gnn_siamese"
        or name.startswith(
            "gnn_siamese."
        )
    ):

        del sys.modules[
            name
        ]


importlib.invalidate_caches()


print()
print("SCIENTIFIC_CHECKOUT = PASS")
print("WORKING_TREE_CLEAN = PASS")
print("RUN_PRODUCTIVE_QUEUE = False")
print("=" * 80)

MODEL B A9 — 0.2 RECOVER SCIENTIFIC CHECKOUT
REPO = /content/model_b_workspace/repo
REPO_EXISTS = False
GIT_EXISTS = False
CLONE_RETURN_CODE = 0
Cloning into '/content/model_b_workspace/repo'...

REPO_CLONE = PASS
RESOLVED_SHA = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
WORKING_TREE = ''

SCIENTIFIC_CHECKOUT = PASS
WORKING_TREE_CLEAN = PASS
RUN_PRODUCTIVE_QUEUE = False


In [ ]:
# =============================================================================
# MODEL B A9
# 0.3 — RESTORE EXACT B11 PYTHON / TORCH / PYG ENVIRONMENT
#
# FINAL FAIL-CLOSED / IDEMPOTENT VERSION
#
# IMPORTANT:
# - NEVER remove torch_geometric from sys.modules.
# - NEVER reload torch_geometric.
# - NEVER manipulate PyTorch DataPipe registries.
# - Detect stale PyG DataPipe registrations BEFORE importing PyG.
# =============================================================================

from pathlib import Path

import json
import platform
import subprocess
import sys
import importlib
import importlib.util
import importlib.metadata

import torch


print("=" * 80)
print("MODEL B A9 — 0.3 RESTORE SOFTWARE ENVIRONMENT")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False


# =============================================================================
# 1. LOAD REFERENCE ENVIRONMENT FROM B11
# =============================================================================

assert B11_MANIFEST.is_file(), (
    f"B11 manifest missing: {B11_MANIFEST}"
)

manifest = json.loads(
    B11_MANIFEST.read_text(
        encoding="utf-8"
    )
)


dependency_blocks = []


def find_dependency_block(obj, path="root"):

    if isinstance(obj, dict):

        if {
            "python",
            "pytorch",
            "torch_geometric",
        }.issubset(obj.keys()):

            dependency_blocks.append(
                (
                    path,
                    obj,
                )
            )

        for key, value in obj.items():

            find_dependency_block(
                value,
                f"{path}.{key}",
            )

    elif isinstance(obj, list):

        for index, value in enumerate(obj):

            find_dependency_block(
                value,
                f"{path}[{index}]",
            )


find_dependency_block(
    manifest
)

assert dependency_blocks, (
    "B11 manifest does not contain "
    "python/pytorch/torch_geometric metadata."
)

dependency_path, recorded = (
    dependency_blocks[0]
)

RECORDED_PYTHON = str(
    recorded["python"]
)

RECORDED_TORCH = str(
    recorded["pytorch"]
)

RECORDED_PYG = str(
    recorded["torch_geometric"]
)


print(
    "REFERENCE_DEPENDENCY_BLOCK =",
    dependency_path,
)

print(
    "B11 Python =",
    RECORDED_PYTHON,
)

print(
    "B11 PyTorch =",
    RECORDED_TORCH,
)

print(
    "B11 PyG =",
    RECORDED_PYG,
)


# =============================================================================
# 2. VALIDATE COLAB BASE RUNTIME EXACTLY
# =============================================================================

CURRENT_PYTHON = (
    platform.python_version()
)

CURRENT_TORCH = (
    torch.__version__
)


print()
print(
    "Current Python =",
    CURRENT_PYTHON,
)

print(
    "Current PyTorch =",
    CURRENT_TORCH,
)

print(
    "Current torch CUDA =",
    torch.version.cuda,
)

print(
    "CUDA available =",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():

    print(
        "GPU =",
        torch.cuda.get_device_name(0),
    )


assert (
    CURRENT_PYTHON
    == RECORDED_PYTHON
), (
    "STOP: Python differs from the B11 environment. "
    f"B11={RECORDED_PYTHON}; current={CURRENT_PYTHON}"
)


assert (
    CURRENT_TORCH
    == RECORDED_TORCH
), (
    "STOP: PyTorch differs from the B11 environment. "
    f"B11={RECORDED_TORCH}; current={CURRENT_TORCH}"
)


print()
print(
    "PYTHON_EXACT_MATCH = PASS"
)

print(
    "TORCH_EXACT_MATCH = PASS"
)


# =============================================================================
# 3. DETECT CURRENT PYG PROCESS STATE
# =============================================================================

PYG_ALREADY_LOADED = (
    "torch_geometric"
    in sys.modules
)


print()
print(
    "PYG_ALREADY_LOADED =",
    PYG_ALREADY_LOADED,
)


# =============================================================================
# 4. IF ALREADY LOADED, NEVER RELOAD IT
# =============================================================================

if PYG_ALREADY_LOADED:

    torch_geometric = (
        sys.modules[
            "torch_geometric"
        ]
    )

    CURRENT_PYG = str(
        torch_geometric.__version__
    )

    print(
        "Current PyG =",
        CURRENT_PYG,
    )

    assert (
        CURRENT_PYG
        == RECORDED_PYG
    ), (
        "STOP: another PyG version is already loaded. "
        "Restart the Colab session."
    )

    print(
        "PYG_REUSED_FROM_CURRENT_PROCESS = YES"
    )


# =============================================================================
# 5. PYG NOT LOADED — CHECK FOR STALE DATAPIPE REGISTRATION
# =============================================================================

else:

    # -------------------------------------------------------------------------
    # This registry survives deleting torch_geometric from sys.modules.
    # If batch_graphs is already present while PyG is NOT loaded, then the
    # process has suffered an unsafe partial unload/re-import.
    # -------------------------------------------------------------------------

    from torch.utils.data.datapipes.datapipe import (
        IterDataPipe,
        MapDataPipe,
    )

    iter_functions = getattr(
        IterDataPipe,
        "functions",
        {},
    )

    map_functions = getattr(
        MapDataPipe,
        "functions",
        {},
    )


    BATCH_GRAPHS_IN_ITER = (
        "batch_graphs"
        in iter_functions
    )

    BATCH_GRAPHS_IN_MAP = (
        "batch_graphs"
        in map_functions
    )

    STALE_PYG_DATAPIPE_REGISTRY = (
        BATCH_GRAPHS_IN_ITER
        or BATCH_GRAPHS_IN_MAP
    )


    print()
    print(
        "BATCH_GRAPHS_IN_ITER_REGISTRY =",
        BATCH_GRAPHS_IN_ITER,
    )

    print(
        "BATCH_GRAPHS_IN_MAP_REGISTRY =",
        BATCH_GRAPHS_IN_MAP,
    )

    print(
        "STALE_PYG_DATAPIPE_REGISTRY =",
        STALE_PYG_DATAPIPE_REGISTRY,
    )


    assert not STALE_PYG_DATAPIPE_REGISTRY, (
        "STOP: stale PyG DataPipe registration detected while "
        "torch_geometric is not loaded. "
        "Restart the Colab session before continuing. "
        "Do NOT remove registry entries manually."
    )


    print(
        "PYG_PROCESS_STATE_CLEAN = PASS"
    )


    # =========================================================================
    # 6. CHECK INSTALLED DISTRIBUTION WITHOUT IMPORTING PYG
    # =========================================================================

    try:

        INSTALLED_PYG = (
            importlib.metadata.version(
                "torch-geometric"
            )
        )

    except (
        importlib.metadata.PackageNotFoundError
    ):

        INSTALLED_PYG = None


    print()
    print(
        "INSTALLED_PYG_DISTRIBUTION =",
        INSTALLED_PYG,
    )

    print(
        "TARGET_PYG_VERSION =",
        RECORDED_PYG,
    )


    # =========================================================================
    # 7. INSTALL EXACT VERSION ONLY IF NECESSARY
    # =========================================================================

    if (
        INSTALLED_PYG
        != RECORDED_PYG
    ):

        print(
            "PYG_INSTALL_REQUIRED = YES"
        )


        command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-deps",
            "--force-reinstall",
            f"torch-geometric=={RECORDED_PYG}",
        ]


        print()
        print("COMMAND:")
        print(
            " ".join(
                command
            )
        )


        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )


        print()
        print(
            "PIP_RETURN_CODE =",
            result.returncode,
        )


        if result.stdout:
            print(
                result.stdout
            )

        if result.stderr:
            print(
                result.stderr
            )


        assert (
            result.returncode
            == 0
        ), (
            "Exact PyG installation failed."
        )


        # Verify distribution metadata WITHOUT importing it yet.

        INSTALLED_PYG = (
            importlib.metadata.version(
                "torch-geometric"
            )
        )


        assert (
            INSTALLED_PYG
            == RECORDED_PYG
        )


    else:

        print(
            "PYG_INSTALL_REQUIRED = NO"
        )


    # =========================================================================
    # 8. FIRST AND ONLY PYG IMPORT FOR THIS PROCESS
    # =========================================================================

    import torch_geometric


    CURRENT_PYG = str(
        torch_geometric.__version__
    )


    print()
    print(
        "Current PyG =",
        CURRENT_PYG,
    )


    assert (
        CURRENT_PYG
        == RECORDED_PYG
    ), (
        "Imported PyG does not match B11."
    )


# =============================================================================
# 9. VERIFY PYTORCH STILL UNCHANGED
# =============================================================================

assert (
    torch.__version__
    == RECORDED_TORCH
)


print()
print(
    "PYG_EXACT_MATCH = PASS"
)

print(
    "TORCH_UNCHANGED = PASS"
)


# =============================================================================
# 10. MINIMAL PYG SANITY
# =============================================================================

from torch_geometric.data import (
    Data,
    Batch,
)


x = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
    ],
    dtype=torch.float32,
)


edge_index = torch.tensor(
    [
        [0, 1],
        [1, 0],
    ],
    dtype=torch.long,
)


test_graph = Data(
    x=x,
    edge_index=edge_index,
)


test_batch = (
    Batch.from_data_list(
        [
            test_graph,
            test_graph,
        ]
    )
)


assert (
    test_batch.num_graphs
    == 2
)

assert (
    int(
        test_batch.x.shape[0]
    )
    == 4
)


print(
    "PYG_MINIMAL_SANITY = PASS"
)


# =============================================================================
# 11. PROJECT IMPORT
#
# Safe:
#   gnn_siamese -> may be purged/reimported.
#
# Unsafe:
#   torch_geometric -> NEVER purge/reload.
# =============================================================================

os.chdir(
    REPO
)


for candidate in (
    str(REPO),
    str(SRC),
):

    while candidate in sys.path:

        sys.path.remove(
            candidate
        )


sys.path.insert(
    0,
    str(REPO),
)

sys.path.insert(
    0,
    str(SRC),
)


for name in list(
    sys.modules
):

    if (
        name == "gnn_siamese"
        or name.startswith(
            "gnn_siamese."
        )
    ):

        del sys.modules[
            name
        ]


importlib.invalidate_caches()


import gnn_siamese


expected_init = (
    SRC
    / "gnn_siamese"
    / "__init__.py"
).resolve()


actual_init = (
    Path(
        gnn_siamese.__file__
    ).resolve()
)


print()
print(
    "gnn_siamese file =",
    actual_init,
)


assert (
    actual_init
    == expected_init
)


print(
    "PROJECT_IMPORT = PASS"
)


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 80)

print(
    "SOFTWARE_ENVIRONMENT_RECOVERY = PASS"
)

print(
    "PYTHON =",
    CURRENT_PYTHON,
)

print(
    "PYTORCH =",
    torch.__version__,
)

print(
    "PYG =",
    torch_geometric.__version__,
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_NEW_RUN_CREATED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)

MODEL B A9 — 0.3 RESTORE SOFTWARE ENVIRONMENT
REFERENCE_DEPENDENCY_BLOCK = root.environment.dependencies
B11 Python = 3.13.15
B11 PyTorch = 2.11.0+cu128
B11 PyG = 2.8.0.post1

Current Python = 3.13.15
Current PyTorch = 2.11.0+cu128
Current torch CUDA = 12.8
CUDA available = True
GPU = Tesla T4

PYTHON_EXACT_MATCH = PASS
TORCH_EXACT_MATCH = PASS

PYG_ALREADY_LOADED = True
Current PyG = 2.8.0.post1
PYG_REUSED_FROM_CURRENT_PROCESS = YES

PYG_EXACT_MATCH = PASS
TORCH_UNCHANGED = PASS
PYG_MINIMAL_SANITY = PASS

gnn_siamese file = /content/model_b_workspace/repo/src/gnn_siamese/__init__.py
PROJECT_IMPORT = PASS

SOFTWARE_ENVIRONMENT_RECOVERY = PASS
PYTHON = 3.13.15
PYTORCH = 2.11.0+cu128
PYG = 2.8.0.post1
NO_TRAINING_EXECUTED = TRUE
NO_NEW_RUN_CREATED = TRUE
RUN_PRODUCTIVE_QUEUE = False


In [ ]:

# =============================================================================
# MODEL B A9
# 0.4 — RESTORE HDF5 STAGING + SHA256
# =============================================================================

from pathlib import Path
import shutil


print("=" * 80)
print("MODEL B A9 — 0.4 RESTORE HDF5 STAGING")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False


EXPECTED_MUTANTS_SHA256 = (
    "92eb242f5565db6194a5e29e3469775f"
    "bf7d8c07280ed2d8cdfd5ab98b5b5631"
)

EXPECTED_WT_SHA256 = (
    "29f68e98ae300207511594e0baf7621b"
    "a9e67c85d8df12ceee812a1dea3aa91a"
)


source_mut = Path(
    DRIVE_MUTANTS_HDF5
).resolve()

source_wt = Path(
    DRIVE_WT_HDF5
).resolve()


# =============================================================================
# VERIFY DRIVE SOURCES FIRST
# =============================================================================

drive_mut_sha = (
    sha256_file(
        source_mut
    )
)

drive_wt_sha = (
    sha256_file(
        source_wt
    )
)

print(
    "DRIVE_MUTANTS_SHA256 =",
    drive_mut_sha,
)

print(
    "DRIVE_WT_SHA256 =",
    drive_wt_sha,
)

assert (
    drive_mut_sha
    == EXPECTED_MUTANTS_SHA256
)

assert (
    drive_wt_sha
    == EXPECTED_WT_SHA256
)

print(
    "DRIVE_HDF5_IDENTITY = PASS"
)


# =============================================================================
# RESTORE LOCAL STAGING ONLY WHEN NECESSARY
# =============================================================================

STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def needs_restore(
    path,
    expected_sha,
):

    if not path.is_file():

        return True

    if path.stat().st_size <= 0:

        return True

    return (
        sha256_file(
            path
        )
        != expected_sha
    )


restore_mut = needs_restore(
    LOCAL_MUTANTS_HDF5,
    EXPECTED_MUTANTS_SHA256,
)

restore_wt = needs_restore(
    LOCAL_WT_HDF5,
    EXPECTED_WT_SHA256,
)


if restore_mut:

    print(
        "RESTORING_MUTANTS_HDF5 = YES"
    )

    shutil.copy2(
        source_mut,
        LOCAL_MUTANTS_HDF5,
    )

else:

    print(
        "RESTORING_MUTANTS_HDF5 = NO"
    )


if restore_wt:

    print(
        "RESTORING_WT_HDF5 = YES"
    )

    shutil.copy2(
        source_wt,
        LOCAL_WT_HDF5,
    )

else:

    print(
        "RESTORING_WT_HDF5 = NO"
    )


# =============================================================================
# VERIFY LOCAL STAGING
# =============================================================================

local_mut_sha = (
    sha256_file(
        LOCAL_MUTANTS_HDF5
    )
)

local_wt_sha = (
    sha256_file(
        LOCAL_WT_HDF5
    )
)

print()
print(
    "LOCAL_MUTANTS_SHA256 =",
    local_mut_sha,
)

print(
    "LOCAL_WT_SHA256 =",
    local_wt_sha,
)

assert (
    local_mut_sha
    == EXPECTED_MUTANTS_SHA256
)

assert (
    local_wt_sha
    == EXPECTED_WT_SHA256
)


print()
print(
    "LOCAL_STAGING_IDENTITY = PASS"
)

print(
    "HDF5_STAGING_RECOVERY = PASS"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)

MODEL B A9 — 0.4 RESTORE HDF5 STAGING
DRIVE_MUTANTS_SHA256 = 92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631
DRIVE_WT_SHA256 = 29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a
DRIVE_HDF5_IDENTITY = PASS
RESTORING_MUTANTS_HDF5 = NO
RESTORING_WT_HDF5 = NO

LOCAL_MUTANTS_SHA256 = 92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631
LOCAL_WT_SHA256 = 29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a

LOCAL_STAGING_IDENTITY = PASS
HDF5_STAGING_RECOVERY = PASS
RUN_PRODUCTIVE_QUEUE = False


In [ ]:
# =============================================================================
# MODEL B A9
# 0.5 — SCAN EXISTING PRODUCTIVE RUNS
# READ-ONLY
# =============================================================================

from pathlib import Path
import json


print("=" * 80)
print("MODEL B A9 — 0.5 SCAN PRODUCTIVE RUNS")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False

assert MODEL_B_ARCH_ROOT.is_dir()


# =============================================================================
# SCAN
# =============================================================================

run_records = []


for run_dir in sorted(
    MODEL_B_ARCH_ROOT.glob(
        "run_*"
    )
):

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    acceptance_path = (
        run_dir
        / "a9_acceptance.json"
    )

    best_path = (
        run_dir
        / "checkpoints"
        / "best.pt"
    )

    last_path = (
        run_dir
        / "checkpoints"
        / "last.pt"
    )


    manifest = None

    if manifest_path.is_file():

        try:

            manifest = json.loads(
                manifest_path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception as exc:

            manifest = {
                "_read_error":
                    str(exc)
            }


    seed = None
    status = None
    epochs_completed = None


    if (
        isinstance(
            manifest,
            dict,
        )
        and
        "_read_error"
        not in manifest
    ):

        seed = (
            manifest
            .get(
                "configuration",
                {},
            )
            .get(
                "seed"
            )
        )

        status = (
            manifest.get(
                "status"
            )
        )

        epochs_completed = (
            manifest
            .get(
                "training",
                {},
            )
            .get(
                "epochs_completed"
            )
        )


    acceptance_status = None

    if acceptance_path.is_file():

        try:

            acceptance_payload = json.loads(
                acceptance_path.read_text(
                    encoding="utf-8"
                )
            )

            acceptance_status = (
                acceptance_payload.get(
                    "status"
                )
            )

        except Exception:

            acceptance_status = (
                "UNREADABLE"
            )


    if (
        acceptance_status
        == "accepted"
    ):

        classification = (
            "CLOSED_ACCEPTED"
        )

    elif (
        status
        == "completed"
    ):

        classification = (
            "COMPLETED_NEEDS_ACCEPTANCE"
        )

    elif (
        status
        in {
            "running",
            "interrupted",
        }
    ):

        classification = (
            "NEEDS_RESUME_INSPECTION"
        )

    elif (
        status
        == "failed"
    ):

        classification = (
            "FAILED_REVIEW_REQUIRED"
        )

    else:

        classification = (
            "UNKNOWN"
        )


    record = {

        "run_dir":
            str(
                run_dir
            ),

        "run_name":
            run_dir.name,

        "seed":
            seed,

        "manifest_status":
            status,

        "epochs_completed":
            epochs_completed,

        "best_pt":
            best_path.is_file(),

        "last_pt":
            last_path.is_file(),

        "acceptance_status":
            acceptance_status,

        "classification":
            classification,
    }


    run_records.append(
        record
    )


# =============================================================================
# DISPLAY
# =============================================================================

print(
    "RUNS_FOUND =",
    len(
        run_records
    ),
)

print()

for record in run_records:

    print(
        "-" * 80
    )

    for key in (
        "run_name",
        "seed",
        "manifest_status",
        "epochs_completed",
        "best_pt",
        "last_pt",
        "acceptance_status",
        "classification",
    ):

        print(
            f"{key} =",
            record[
                key
            ],
        )


# =============================================================================
# PER-SEED SUMMARY
# =============================================================================

print()
print("=" * 80)
print("PER-SEED SUMMARY")
print("=" * 80)


seed_records = {

    seed: [
        record

        for record
        in run_records

        if record[
            "seed"
        ]
        == seed
    ]

    for seed
    in PRODUCTIVE_SEEDS
}


for seed in PRODUCTIVE_SEEDS:

    records = seed_records[
        seed
    ]

    accepted = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "CLOSED_ACCEPTED"
    ]

    completed = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "COMPLETED_NEEDS_ACCEPTANCE"
    ]

    resumable = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "NEEDS_RESUME_INSPECTION"
    ]

    failed = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "FAILED_REVIEW_REQUIRED"
    ]


    if accepted:

        seed_state = (
            "CLOSED_ACCEPTED"
        )

    elif completed:

        seed_state = (
            "COMPLETED_NEEDS_ACCEPTANCE"
        )

    elif resumable:

        seed_state = (
            "NEEDS_RESUME_INSPECTION"
        )

    elif failed:

        seed_state = (
            "FAILED_REVIEW_REQUIRED"
        )

    elif records:

        seed_state = (
            "UNKNOWN_EXISTING_RUN"
        )

    else:

        seed_state = (
            "ABSENT"
        )


    print(
        f"SEED_{seed} =",
        seed_state,
    )


print()
print(
    "PRODUCTIVE_RUN_SCAN = PASS"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)

MODEL B A9 — 0.5 SCAN PRODUCTIVE RUNS
RUNS_FOUND = 6

--------------------------------------------------------------------------------
run_name = run_20260911T120628.981117Z-795be862
seed = 11
manifest_status = failed
epochs_completed = 0
best_pt = False
last_pt = False
acceptance_status = None
classification = FAILED_REVIEW_REQUIRED
--------------------------------------------------------------------------------
run_name = run_20260914T143850.128514Z-e382b31e
seed = 11
manifest_status = completed
epochs_completed = 39
best_pt = True
last_pt = True
acceptance_status = accepted
classification = CLOSED_ACCEPTED
--------------------------------------------------------------------------------
run_name = run_20260915T095945.494670Z-9a96c769
seed = 23
manifest_status = completed
epochs_completed = 53
best_pt = True
last_pt = True
acceptance_status = accepted
classification = CLOSED_ACCEPTED
--------------------------------------------------------------------------------
run_name = run_202609

In [ ]:
# =============================================================================
# MODEL B A9
# 0.6 — FULL RUNTIME RECOVERY GATE
# =============================================================================

import subprocess
import json
import torch
import torch_geometric


print("=" * 80)
print("MODEL B A9 — 0.6 RUNTIME RECOVERY GATE")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False


# =============================================================================
# GIT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

git_status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()


# =============================================================================
# B11 ACCEPTANCE
# =============================================================================

acceptance = json.loads(
    B11_ACCEPTANCE.read_text(
        encoding="utf-8"
    )
)


# =============================================================================
# CHECKS
# =============================================================================

checks = {

    "SCIENTIFIC_COMMIT":
        (
            head
            == EXPECTED_SHA
        ),

    "WORKING_TREE_CLEAN":
        (
            git_status
            == ""
        ),

    "PYTHON_ENVIRONMENT":
        (
            platform.python_version()
            == RECORDED_PYTHON
        ),

    "PYTORCH_ENVIRONMENT":
        (
            torch.__version__
            == RECORDED_TORCH
        ),

    "PYG_ENVIRONMENT":
        (
            torch_geometric.__version__
            == RECORDED_PYG
        ),

    "MUTANTS_STAGED":
        (
            LOCAL_MUTANTS_HDF5.is_file()
        ),

    "WT_STAGED":
        (
            LOCAL_WT_HDF5.is_file()
        ),

    "MUTANTS_SHA":
        (
            sha256_file(
                LOCAL_MUTANTS_HDF5
            )
            == EXPECTED_MUTANTS_SHA256
        ),

    "WT_SHA":
        (
            sha256_file(
                LOCAL_WT_HDF5
            )
            == EXPECTED_WT_SHA256
        ),

    "B11_RUN_PRESENT":
        (
            B11_REFERENCE_RUN.is_dir()
        ),

    "B11_BEST_PRESENT":
        (
            B11_BEST_PT.is_file()
        ),

    "B11_LAST_PRESENT":
        (
            B11_LAST_PT.is_file()
        ),

    "B11_ACCEPTED":
        (
            acceptance.get(
                "status"
            )
            == "accepted"
        ),

    "B11_ACCEPTANCE_SHA":
        (
            sha256_file(
                B11_ACCEPTANCE
            )
            == EXPECTED_B11_ACCEPTANCE_SHA256
        ),

    "PRODUCTIVE_RUN_SCAN_AVAILABLE":
        (
            "run_records"
            in globals()
        ),

    "RUN_PRODUCTIVE_QUEUE_FALSE":
        (
            RUN_PRODUCTIVE_QUEUE
            is False
        ),
}


# =============================================================================
# DISPLAY
# =============================================================================

for key, value in checks.items():

    print(
        f"{key} =",
        (
            "PASS"
            if value
            else "FAIL"
        ),
    )


assert all(
    checks.values()
), (
    "RUNTIME RECOVERY GATE FAILED"
)


print()
print(
    "RUNTIME_RECOVERY_GATE = PASS"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_NEW_RUN_CREATED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)

MODEL B A9 — 0.6 RUNTIME RECOVERY GATE
SCIENTIFIC_COMMIT = PASS
WORKING_TREE_CLEAN = PASS
PYTHON_ENVIRONMENT = PASS
PYTORCH_ENVIRONMENT = PASS
PYG_ENVIRONMENT = PASS
MUTANTS_STAGED = PASS
WT_STAGED = PASS
MUTANTS_SHA = PASS
WT_SHA = PASS
B11_RUN_PRESENT = PASS
B11_BEST_PRESENT = PASS
B11_LAST_PRESENT = PASS
B11_ACCEPTED = PASS
B11_ACCEPTANCE_SHA = PASS
PRODUCTIVE_RUN_SCAN_AVAILABLE = PASS
RUN_PRODUCTIVE_QUEUE_FALSE = PASS

RUNTIME_RECOVERY_GATE = PASS
NO_TRAINING_EXECUTED = TRUE
NO_NEW_RUN_CREATED = TRUE
RUN_PRODUCTIVE_QUEUE = False


In [ ]:
# =============================================================================
# MODEL B A9
# 0.7 — DETERMINE NEXT PRODUCTIVE SEED SAFELY
# NO TRAINING
# =============================================================================

print("=" * 80)
print("MODEL B A9 — 0.7 DETERMINE NEXT SEED")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False

assert (
    "run_records"
    in globals()
)

assert (
    "seed_records"
    in globals()
)


# =============================================================================
# RESOLVE ONE STATE PER SEED
# =============================================================================

seed_state_map = {}


for seed in PRODUCTIVE_SEEDS:

    records = seed_records[
        seed
    ]


    accepted = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "CLOSED_ACCEPTED"
    ]


    completed = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "COMPLETED_NEEDS_ACCEPTANCE"
    ]


    resumable = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "NEEDS_RESUME_INSPECTION"
    ]


    failed = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "FAILED_REVIEW_REQUIRED"
    ]


    unknown = [
        r

        for r
        in records

        if r[
            "classification"
        ]
        == "UNKNOWN"
    ]


    if accepted:

        state = (
            "CLOSED_ACCEPTED"
        )

    elif completed:

        state = (
            "COMPLETED_NEEDS_ACCEPTANCE"
        )

    elif resumable:

        state = (
            "NEEDS_RESUME_INSPECTION"
        )

    elif failed:

        state = (
            "FAILED_REVIEW_REQUIRED"
        )

    elif unknown:

        state = (
            "UNKNOWN_EXISTING_RUN"
        )

    else:

        state = (
            "ABSENT"
        )


    seed_state_map[
        seed
    ] = state


# =============================================================================
# DISPLAY CURRENT STATE
# =============================================================================

for seed in PRODUCTIVE_SEEDS:

    print(
        f"B{seed} =",
        seed_state_map[
            seed
        ],
    )


# =============================================================================
# FIND FIRST NON-ACCEPTED SEED
# =============================================================================

NEXT_SEED = None
NEXT_ACTION = None


for seed in PRODUCTIVE_SEEDS:

    state = (
        seed_state_map[
            seed
        ]
    )


    if (
        state
        == "CLOSED_ACCEPTED"
    ):

        continue


    NEXT_SEED = seed


    if (
        state
        == "ABSENT"
    ):

        NEXT_ACTION = (
            "FRESH_READY"
        )


    elif (
        state
        == "COMPLETED_NEEDS_ACCEPTANCE"
    ):

        NEXT_ACTION = (
            "POSTRUN_ACCEPTANCE_REQUIRED"
        )


    elif (
        state
        == "NEEDS_RESUME_INSPECTION"
    ):

        NEXT_ACTION = (
            "INSPECT_RESUME_CANDIDATE"
        )


    elif (
        state
        == "FAILED_REVIEW_REQUIRED"
    ):

        NEXT_ACTION = (
            "INSPECT_FAILED_RUN"
        )


    else:

        NEXT_ACTION = (
            "BLOCKED_UNKNOWN_STATE"
        )


    break


# =============================================================================
# ALL COMPLETE?
# =============================================================================

if NEXT_SEED is None:

    NEXT_ACTION = (
        "ALL_PRODUCTIVE_SEEDS_ACCEPTED"
    )


# =============================================================================
# OUT-OF-ORDER SAFETY CHECK
# =============================================================================

OUT_OF_ORDER_RUNS = []


if NEXT_SEED is not None:

    found_next = False

    for seed in PRODUCTIVE_SEEDS:

        if seed == NEXT_SEED:

            found_next = True
            continue


        if (
            found_next
            and
            seed_state_map[
                seed
            ]
            != "ABSENT"
        ):

            OUT_OF_ORDER_RUNS.append(
                {
                    "seed":
                        seed,

                    "state":
                        seed_state_map[
                            seed
                        ],
                }
            )


if OUT_OF_ORDER_RUNS:

    NEXT_ACTION = (
        "BLOCKED_OUT_OF_ORDER_RUNS"
    )


# =============================================================================
# FINAL REPORT
# =============================================================================

print()
print("=" * 80)

print(
    "NEXT_SEED =",
    NEXT_SEED,
)

print(
    "NEXT_ACTION =",
    NEXT_ACTION,
)

print(
    "OUT_OF_ORDER_RUNS =",
    OUT_OF_ORDER_RUNS,
)

print(
    "RUN_PRODUCTIVE_QUEUE =",
    RUN_PRODUCTIVE_QUEUE,
)


assert (
    RUN_PRODUCTIVE_QUEUE
    is False
)


if (
    NEXT_ACTION
    == "FRESH_READY"
):

    print()
    print(
        "NEXT_SEED_READY_FOR_EXPLICIT_FRESH_LAUNCH = YES"
    )

else:

    print()
    print(
        "NEXT_SEED_READY_FOR_EXPLICIT_FRESH_LAUNCH = NO"
    )


print()
print(
    "AUTOMATIC_TRAINING_LAUNCHED = FALSE"
)

print(
    "RECOVERY_SEQUENCE = COMPLETE"
)

print("=" * 80)

MODEL B A9 — 0.7 DETERMINE NEXT SEED
B11 = CLOSED_ACCEPTED
B23 = CLOSED_ACCEPTED
B37 = CLOSED_ACCEPTED
B41 = CLOSED_ACCEPTED
B53 = CLOSED_ACCEPTED

NEXT_SEED = None
NEXT_ACTION = ALL_PRODUCTIVE_SEEDS_ACCEPTED
OUT_OF_ORDER_RUNS = []
RUN_PRODUCTIVE_QUEUE = False

NEXT_SEED_READY_FOR_EXPLICIT_FRESH_LAUNCH = NO

AUTOMATIC_TRAINING_LAUNCHED = FALSE
RECOVERY_SEQUENCE = COMPLETE


## 2. Clon controlado y checkout detached del commit congelado

In [ ]:
from pathlib import Path
from collections import Counter
import subprocess
import sys
import os

# ================================================================
# CONTRATO
# ================================================================

EXPECTED_SHA = "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"

EXPECTED_MUTANTS_SHA = (
    "92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631"
)

EXPECTED_WT_SHA = (
    "29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a"
)

EXPECTED_COMBINED_SHA = (
    "9a320862a54566d232e1ef2a4eafa468cf900a19186dc0fd7c92aaf0abe06c68"
)

EXPECTED_SPLIT_FINGERPRINT = (
    "8dcc611a6575b95242fda70d2b29e6bf4917a9c781bbeebd851e95e71391d7dd"
)

EXPECTED_COUNTS = {
    "train": 342,
    "validation": 78,
    "test": 63,
}

EXPECTED_BATCH_COUNTS = {
    "train": 86,
    "validation": 20,
    "test": 16,
}

RUN_PRODUCTIVE_QUEUE = False

# ================================================================
# PATHS
# ================================================================

REPO = Path("/content/model_b_workspace/repo")
SRC = REPO / "src"

CONFIG_PATH = (
    REPO
    / "configs"
    / "model_b_a9.yaml"
)

SCHEMA_PATH = (
    REPO
    / "sample_data"
    / "sample_schema.json"
)

FROZEN_SPLIT = (
    REPO
    / "splits"
    / "leave_position_out_seed_42.json"
)

MUTANTS_HDF5 = Path(
    "/content/drive/MyDrive/"
    "modelos_proyecto_PKP2/"
    "model_a/data/proc_483p.hdf5"
)

WT_HDF5 = Path(
    "/content/drive/MyDrive/"
    "modelos_proyecto_PKP2/"
    "model_a/data/wt_companion.hdf5"
)

FAILED_B11 = Path(
    "/content/drive/MyDrive/"
    "modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9/"
    "model_b_graph_level_relational/"
    "run_20260911T120628.981117Z-795be862"
)

print("=" * 80)
print("MODEL B A9 — REAL DATA BATCHING PREFLIGHT")
print("=" * 80)

print("RUN_PRODUCTIVE_QUEUE =", RUN_PRODUCTIVE_QUEUE)

assert RUN_PRODUCTIVE_QUEUE is False

# ================================================================
# 1. CHECKOUT
# ================================================================

print()
print("=" * 80)
print("1. SCIENTIFIC CHECKOUT")
print("=" * 80)

HEAD = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

print("HEAD =", HEAD)

assert HEAD == EXPECTED_SHA, (
    f"HEAD incorrecto: {HEAD}"
)

print("SCIENTIFIC_CHECKOUT = PASS")

# ================================================================
# 2. INPUTS
# ================================================================

print()
print("=" * 80)
print("2. INPUT FILES")
print("=" * 80)

for label, path in (
    ("CONFIG", CONFIG_PATH),
    ("SCHEMA", SCHEMA_PATH),
    ("MUTANTS_HDF5", MUTANTS_HDF5),
    ("WT_HDF5", WT_HDF5),
    ("FROZEN_SPLIT", FROZEN_SPLIT),
):

    print(label, "=", path)

    assert path.is_file(), (
        f"{label} no existe: {path}"
    )

print("INPUT_FILES = PASS")

# El run fallido se conserva, pero NO se utiliza como input.
print("FAILED_B11 =", FAILED_B11)
print(
    "FAILED_B11_PRESERVED =",
    FAILED_B11.exists(),
)

# ================================================================
# 3. IMPORTS
# ================================================================

print()
print("=" * 80)
print("3. IMPORTS")
print("=" * 80)

os.chdir(REPO)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from gnn_siamese.config import load_config

from gnn_siamese.builders import (
    build_dataset_bundle,
    build_split_bundle,
    build_dataloaders,
)

from gnn_siamese.data.position_batch_sampler import (
    PositionDiverseBatchSampler,
)

from gnn_siamese.losses.false_negative_mask import (
    build_false_negative_mask,
)

from gnn_siamese.utils.fingerprints import (
    fingerprint_hdf5_inputs,
    fingerprint_split_definition,
    resolve_hdf5_dataset_id,
)

print("IMPORTS = PASS")

# ================================================================
# 4. CONFIG CANÓNICO
# ================================================================

print()
print("=" * 80)
print("4. MODEL B A9 CONTRACT")
print("=" * 80)

cfg = load_config(CONFIG_PATH)

# ------------------------------------------------
# Solo adaptación operacional de paths.
# No modificamos parámetros científicos.
# ------------------------------------------------

cfg["paths"]["mutants_hdf5"] = str(
    MUTANTS_HDF5
)

cfg["paths"]["wt_companion_hdf5"] = str(
    WT_HDF5
)

cfg["paths"]["sample_schema"] = str(
    SCHEMA_PATH
)

cfg["split"]["persist_path"] = str(
    FROZEN_SPLIT
)

cfg["split"]["allow_create"] = False

# Verificación explícita del contrato.
assert (
    cfg["model"]["architecture"]
    == "model_b_graph_level_relational"
)

assert cfg["training"]["batch_size"] == 4

assert cfg["split"]["seed"] == 42
assert cfg["split"]["allow_create"] is False

mask_cfg = cfg["loss"]["false_negative_mask"]

assert mask_cfg["enabled"] is True
assert mask_cfg["mode"] == "same_position"
assert mask_cfg["strict"] is True
assert mask_cfg["min_valid_negatives"] == 1

assert (
    float(mask_cfg["min_valid_negative_fraction"])
    == 0.0
)

print(
    "architecture =",
    cfg["model"]["architecture"],
)

print(
    "batch_size =",
    cfg["training"]["batch_size"],
)

print(
    "split_seed =",
    cfg["split"]["seed"],
)

print(
    "false_negative_mask.mode =",
    mask_cfg["mode"],
)

print(
    "false_negative_mask.strict =",
    mask_cfg["strict"],
)

print(
    "min_valid_negatives =",
    mask_cfg["min_valid_negatives"],
)

print("MODEL_B_CONTRACT = PASS")

# ================================================================
# 5. HDF5 FINGERPRINTS
# ================================================================

print()
print("=" * 80)
print("5. HDF5 FINGERPRINTS")
print("=" * 80)

hdf5_fp = fingerprint_hdf5_inputs(
    mutants_path=MUTANTS_HDF5,
    wt_companion_path=WT_HDF5,
    dataset_id=resolve_hdf5_dataset_id(cfg),
)

digest_by_role = {
    item["role"]: item["digest"]
    for item in hdf5_fp["files"]
}

mutants_digest = digest_by_role["mutants"]
wt_digest = digest_by_role["wt_companion"]
combined_digest = hdf5_fp["combined"]["digest"]

print(
    "mutants_sha256 =",
    mutants_digest,
)

print(
    "wt_sha256 =",
    wt_digest,
)

print(
    "combined =",
    combined_digest,
)

assert mutants_digest == EXPECTED_MUTANTS_SHA
assert wt_digest == EXPECTED_WT_SHA
assert combined_digest == EXPECTED_COMBINED_SHA

print("HDF5_FINGERPRINTS = PASS")

# ================================================================
# 6. DATASET
# ================================================================

print()
print("=" * 80)
print("6. DATASET")
print("=" * 80)

dataset_bundle = build_dataset_bundle(
    cfg
)

dataset = dataset_bundle.dataset

print(
    "BIOLOGICAL_VARIANTS =",
    len(dataset.pairs),
)

print(
    "UNIQUE_POSITIONS =",
    len({
        int(pair.position)
        for pair in dataset.pairs
    }),
)

assert len(dataset.pairs) == 483

print("DATASET = PASS")

# ================================================================
# 7. FROZEN SPLIT
# ================================================================

print()
print("=" * 80)
print("7. FROZEN SPLIT")
print("=" * 80)

split_bundle = build_split_bundle(
    cfg,
    dataset,
)

assert split_bundle.created is False

counts = {
    "train": len(
        split_bundle.train_indices
    ),
    "validation": len(
        split_bundle.validation_indices
    ),
    "test": len(
        split_bundle.test_indices
    ),
}

for partition in (
    "train",
    "validation",
    "test",
):
    print(
        partition.upper(),
        "=",
        counts[partition],
    )

    assert (
        counts[partition]
        == EXPECTED_COUNTS[partition]
    )

split_fp = fingerprint_split_definition(
    split_bundle.split
)

print(
    "split_fingerprint =",
    split_fp,
)

assert (
    split_fp
    == EXPECTED_SPLIT_FINGERPRINT
)

# Verificación explícita de ausencia de solape por posición.
partition_positions = {}

for name, indices in (
    (
        "train",
        split_bundle.train_indices,
    ),
    (
        "validation",
        split_bundle.validation_indices,
    ),
    (
        "test",
        split_bundle.test_indices,
    ),
):

    partition_positions[name] = {
        int(dataset.pairs[index].position)
        for index in indices
    }

assert (
    partition_positions["train"]
    .isdisjoint(
        partition_positions["validation"]
    )
)

assert (
    partition_positions["train"]
    .isdisjoint(
        partition_positions["test"]
    )
)

assert (
    partition_positions["validation"]
    .isdisjoint(
        partition_positions["test"]
    )
)

print("POSITION_OVERLAP = 0")
print("FROZEN_SPLIT = PASS")

# ================================================================
# 8. DATALOADERS DEL NUEVO FIX
# ================================================================

print()
print("=" * 80)
print("8. DATALOADERS")
print("=" * 80)

loaders = build_dataloaders(
    cfg,
    dataset,
    split_bundle,
)

samplers = {
    "train":
        loaders.train_loader.batch_sampler,

    "validation":
        loaders.validation_loader.batch_sampler,

    "test":
        loaders.test_loader.batch_sampler,
}

for name, sampler in samplers.items():

    print(
        name,
        "sampler =",
        type(sampler).__name__,
    )

    assert isinstance(
        sampler,
        PositionDiverseBatchSampler,
    )

print(
    "POSITION_DIVERSE_DATALOADERS = PASS"
)

# ================================================================
# 9. AUDITAR TODOS LOS BATCHES
# ================================================================

print()
print("=" * 80)
print("9. COMPLETE BATCH AUDIT")
print("=" * 80)


def audit_sampler(
    name,
    sampler,
):

    # Preservar estado RNG del sampler de train.
    generator_state = None

    if sampler.generator is not None:
        generator_state = (
            sampler.generator
            .get_state()
            .clone()
        )

    batches = list(sampler)

    if (
        sampler.generator is not None
        and generator_state is not None
    ):
        sampler.generator.set_state(
            generator_state
        )

    flat = [
        index
        for batch in batches
        for index in batch
    ]

    # Todos los ejemplos una vez.
    assert len(flat) == len(
        sampler.positions
    )

    assert Counter(flat) == Counter(
        range(len(sampler.positions))
    )

    degenerate_batches = 0
    global_min_valid_negatives = None

    batch_sizes = []

    for batch_index, batch in enumerate(
        batches
    ):

        batch_sizes.append(
            len(batch)
        )

        batch_positions = [
            sampler.positions[index]
            for index in batch
        ]

        if len(set(batch_positions)) < 2:
            degenerate_batches += 1

        # Usamos exactamente la semántica
        # same_position + strict del contrato A9.
        result = build_false_negative_mask(
            batch_size=len(batch),
            mode="same_position",
            positions=batch_positions,
            min_valid_negatives=1,
            min_valid_fraction=0.0,
            strict=True,
        )

        local_min = min(
            item.valid_negatives
            for item
            in result.per_anchor_stats
        )

        if global_min_valid_negatives is None:
            global_min_valid_negatives = (
                local_min
            )
        else:
            global_min_valid_negatives = min(
                global_min_valid_negatives,
                local_min,
            )

    print()
    print(name.upper())
    print(
        "examples =",
        len(sampler.positions),
    )
    print(
        "batches =",
        len(batches),
    )
    print(
        "batch_sizes =",
        dict(Counter(batch_sizes)),
    )
    print(
        "degenerate_batches =",
        degenerate_batches,
    )
    print(
        "min_valid_negatives =",
        global_min_valid_negatives,
    )

    assert (
        len(batches)
        == EXPECTED_BATCH_COUNTS[name]
    )

    assert degenerate_batches == 0

    assert (
        global_min_valid_negatives
        is not None
    )

    assert (
        global_min_valid_negatives >= 1
    )

    return {
        "examples":
            len(sampler.positions),

        "batches":
            len(batches),

        "degenerate_batches":
            degenerate_batches,

        "min_valid_negatives":
            global_min_valid_negatives,
    }


audit_results = {
    name: audit_sampler(
        name,
        sampler,
    )
    for name, sampler
    in samplers.items()
}

print()
print(
    "COMPLETE_BATCH_AUDIT = PASS"
)

# ================================================================
# 10. FINAL GATE
# ================================================================

print()
print("=" * 80)
print("10. FINAL PREFLIGHT GATE")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False

for name in (
    "train",
    "validation",
    "test",
):

    assert (
        audit_results[name]
        ["degenerate_batches"]
        == 0
    )

print("NEW_SHA =", EXPECTED_SHA)
print("BIOLOGICAL_VARIANTS = 483")
print("TRAIN = 342")
print("VALIDATION = 78")
print("TEST = 63")

print(
    "TRAIN_DEGENERATE_BATCHES =",
    audit_results["train"][
        "degenerate_batches"
    ],
)

print(
    "VALIDATION_DEGENERATE_BATCHES =",
    audit_results["validation"][
        "degenerate_batches"
    ],
)

print(
    "TEST_DEGENERATE_BATCHES =",
    audit_results["test"][
        "degenerate_batches"
    ],
)

print()
print("REAL_DATA_BATCHING_PREFLIGHT = PASS")
print("RUN_PRODUCTIVE_QUEUE = False")

print("=" * 80)

MODEL B A9 — REAL DATA BATCHING PREFLIGHT
RUN_PRODUCTIVE_QUEUE = False

1. SCIENTIFIC CHECKOUT
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
SCIENTIFIC_CHECKOUT = PASS

2. INPUT FILES
CONFIG = /content/model_b_workspace/repo/configs/model_b_a9.yaml
SCHEMA = /content/model_b_workspace/repo/sample_data/sample_schema.json
MUTANTS_HDF5 = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/proc_483p.hdf5
WT_HDF5 = /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/wt_companion.hdf5
FROZEN_SPLIT = /content/model_b_workspace/repo/splits/leave_position_out_seed_42.json
INPUT_FILES = PASS
FAILED_B11 = /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260911T120628.981117Z-795be862
FAILED_B11_PRESERVED = True

3. IMPORTS
IMPORTS = PASS

4. MODEL B A9 CONTRACT
architecture = model_b_graph_level_relational
batch_size = 4
split_seed = 42
false_negative_mask.mode = same_position
false_negative_mask.strict = True
min_valid_negativ

In [ ]:
from pathlib import Path
import subprocess
import json

EXPECTED_SHA = "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"

REPO = Path("/content/model_b_workspace/repo")

CONFIG_B = REPO / "configs" / "model_b_a9.yaml"

MUTANTS_HDF5 = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/data/proc_483p.hdf5"
)

WT_HDF5 = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/data/wt_companion.hdf5"
)

B_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9"
)

A_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

ALLOWED_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2"
)

FAILED_B11 = (
    B_OUTPUT_ROOT
    / "model_b_graph_level_relational"
    / "run_20260911T120628.981117Z-795be862"
)

RUN_PRODUCTIVE_QUEUE = False

print("=" * 80)
print("MODEL B A9 — OFFICIAL SEED 11 PREFLIGHT")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False

# ------------------------------------------------------------------
# 1. Commit exacto
# ------------------------------------------------------------------

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

print("HEAD =", head)

assert head == EXPECTED_SHA

dirty = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"],
    text=True,
).strip()

assert dirty == ""

print("SCIENTIFIC_CHECKOUT = PASS")

# ------------------------------------------------------------------
# 2. El B11 fallido debe seguir existiendo y no se reutiliza
# ------------------------------------------------------------------

print()
print("FAILED_B11 =", FAILED_B11)
print("FAILED_B11_EXISTS =", FAILED_B11.is_dir())

assert FAILED_B11.is_dir()

# No utilizamos checkpoint alguno del run fallido.
failed_checkpoints = (
    list((FAILED_B11 / "checkpoints").glob("*"))
    if (FAILED_B11 / "checkpoints").is_dir()
    else []
)

print(
    "FAILED_B11_CHECKPOINT_FILES =",
    [p.name for p in failed_checkpoints],
)

print("FAILED_B11_PRESERVED = PASS")

# ------------------------------------------------------------------
# 3. A9 preflight oficial, seed 11
# ------------------------------------------------------------------

command = [
    "python",
    "scripts/a9_preflight.py",

    "--config",
    str(CONFIG_B),

    "--architecture",
    "model_b_graph_level_relational",

    "--run-seed",
    "11",

    "--mutants-hdf5",
    str(MUTANTS_HDF5),

    "--wt-hdf5",
    str(WT_HDF5),

    "--output-root",
    str(B_OUTPUT_ROOT),

    "--peer-output-root",
    str(A_OUTPUT_ROOT),

    "--allowed-output-root",
    str(ALLOWED_OUTPUT_ROOT),

    "--expected-commit",
    EXPECTED_SHA,

    "--repo-root",
    str(REPO),
]

print()
print("COMMAND:")
print(" ".join(command))

result = subprocess.run(
    command,
    cwd=str(REPO),
    text=True,
    capture_output=True,
)

print()
print("STDOUT:")
print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("RETURN_CODE =", result.returncode)

assert result.returncode == 0, (
    "A9 official preflight failed"
)

payload = json.loads(result.stdout)

assert payload["status"] == "PASS"
assert payload["run_seed"] == 11
assert payload["split_seed"] == 42
assert payload["ab_intersection"] == 483
assert payload["prospective_exists"] is False

prospective = Path(
    payload["prospective_run_dir"]
)

print()
print("PROSPECTIVE_RUN_DIR =", prospective)
print(
    "PROSPECTIVE_RUN_EXISTS =",
    prospective.exists(),
)

assert not prospective.exists()

print()
print("=" * 80)
print("A9_OFFICIAL_PREFLIGHT_SEED11 = PASS")
print("FRESH_RUN_TARGET = PASS")
print("FAILED_B11_PRESERVED = PASS")
print("RESUME_FROM_FAILED_B11 = FALSE")
print("RUN_PRODUCTIVE_QUEUE = False")
print("=" * 80)

MODEL B A9 — OFFICIAL SEED 11 PREFLIGHT
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
SCIENTIFIC_CHECKOUT = PASS

FAILED_B11 = /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260911T120628.981117Z-795be862
FAILED_B11_EXISTS = True
FAILED_B11_CHECKPOINT_FILES = []
FAILED_B11_PRESERVED = PASS

COMMAND:
python scripts/a9_preflight.py --config /content/model_b_workspace/repo/configs/model_b_a9.yaml --architecture model_b_graph_level_relational --run-seed 11 --mutants-hdf5 /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/proc_483p.hdf5 --wt-hdf5 /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/wt_companion.hdf5 --output-root /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9 --peer-output-root /content/drive/MyDrive/modelos_proyecto_PKP2/model_a/runs/model_a_a9 --allowed-output-root /content/drive/MyDrive/modelos_proyecto_PKP2 --expected-commit ed9dc23d7936a77a33bc40b2ed3400d43f1f9111 --repo-root

# **Prueba seed 11**




In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import yaml

EXPECTED_SHA = "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"

EXPECTED_MUTANTS_SHA = (
    "92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631"
)

EXPECTED_WT_SHA = (
    "29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a"
)

REPO = Path("/content/model_b_workspace/repo")
SRC = REPO / "src"

CONFIG_B = REPO / "configs" / "model_b_a9.yaml"
SCHEMA = REPO / "sample_data" / "sample_schema.json"
FROZEN_SPLIT = REPO / "splits" / "leave_position_out_seed_42.json"

DRIVE_MUTANTS = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/data/proc_483p.hdf5"
)

DRIVE_WT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/data/wt_companion.hdf5"
)

B_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9"
)

A_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_a/runs/model_a_a9"
)

ALLOWED_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2"
)

STAGING_ROOT = Path(
    "/content/model_b_workspace/staging_a9"
)

LOCAL_MUTANTS = STAGING_ROOT / "proc_483p.hdf5"
LOCAL_WT = STAGING_ROOT / "wt_companion.hdf5"

RUNTIME_DIR = Path(
    "/content/model_b_workspace/runtime_configs"
)

RUNTIME_CONFIG = (
    RUNTIME_DIR
    / "model_b_a9_seed11_fresh.yaml"
)

RUN_PRODUCTIVE_QUEUE = False
RUN_SEED = 11

print("=" * 80)
print("MODEL B A9 — PREPARACIÓN PRODUCTIVA B11 FRESH")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False

# ------------------------------------------------------------------
# 1. Checkout exacto
# ------------------------------------------------------------------

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

dirty = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"],
    text=True,
).strip()

print("HEAD =", head)

assert head == EXPECTED_SHA
assert dirty == ""

print("SCIENTIFIC_CHECKOUT = PASS")

# ------------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------------

os.chdir(REPO)

for path in (str(REPO), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

from scripts.colab_preflight import (
    stage_file,
    require_free_space,
)

from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
    validate_a9_colab_preflight,
)

from gnn_siamese.config import load_config
from gnn_siamese.training.a9_contract import validate_a9_run_seed

# ------------------------------------------------------------------
# 3. Staging local de HDF5
# ------------------------------------------------------------------

print()
print("=" * 80)
print("STAGING HDF5")
print("=" * 80)

STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

required_bytes = (
    DRIVE_MUTANTS.stat().st_size
    + DRIVE_WT.stat().st_size
)

free = require_free_space(
    STAGING_ROOT,
    required_bytes * 2,
)

print("LOCAL_FREE_BYTES =", free)

mutants_record = stage_file(
    DRIVE_MUTANTS,
    LOCAL_MUTANTS,
    staging_root=STAGING_ROOT,
    role="mutants",
)

wt_record = stage_file(
    DRIVE_WT,
    LOCAL_WT,
    staging_root=STAGING_ROOT,
    role="wt_companion",
)

print("MUTANTS_STAGED =", mutants_record)
print("WT_STAGED =", wt_record)

assert (
    mutants_record["sha256"]
    == EXPECTED_MUTANTS_SHA
)

assert (
    wt_record["sha256"]
    == EXPECTED_WT_SHA
)

print("LOCAL_HDF5_STAGING = PASS")

# ------------------------------------------------------------------
# 4. Resolver configuración A9 oficial para seed 11
# ------------------------------------------------------------------

print()
print("=" * 80)
print("RESOLVE A9 CONFIG")
print("=" * 80)

config_a, config_b = resolve_a9_runtime_configs(
    CONFIG_B,
    architecture="model_b_graph_level_relational",
    run_seed=RUN_SEED,
    mutants_hdf5=LOCAL_MUTANTS,
    wt_hdf5=LOCAL_WT,
    output_root=B_OUTPUT_ROOT,
    peer_output_root=A_OUTPUT_ROOT,
    repo_root=REPO,
)

# Rutas operacionales absolutas.
config_b["paths"]["sample_schema"] = str(
    SCHEMA.resolve()
)

config_b["split"]["persist_path"] = str(
    FROZEN_SPLIT.resolve()
)

config_b["split"]["allow_create"] = False

config_b["training"]["device"] = "cuda"

# No queremos serializar una ruta interna heredada.
config_b.pop("__config_path__", None)

# ------------------------------------------------------------------
# 5. Invariantes científicos
# ------------------------------------------------------------------

assert (
    config_b["model"]["architecture"]
    == "model_b_graph_level_relational"
)

assert config_b["project"]["seed"] == 11

assert config_b["reproducibility"]["seed_python"] == 11
assert config_b["reproducibility"]["seed_numpy"] == 11
assert config_b["reproducibility"]["seed_torch"] == 11
assert config_b["reproducibility"]["seed_cuda"] == 11
assert config_b["reproducibility"]["seed_dataloader"] == 11

assert config_b["training"]["epochs"] == 100
assert config_b["training"]["batch_size"] == 4

assert config_b["split"]["seed"] == 42
assert config_b["split"]["allow_create"] is False

mask = config_b["loss"]["false_negative_mask"]

assert mask["enabled"] is True
assert mask["mode"] == "same_position"
assert mask["strict"] is True
assert mask["min_valid_negatives"] == 1
assert float(
    mask["min_valid_negative_fraction"]
) == 0.0

validate_a9_run_seed(
    config_b,
    RUN_SEED,
)

print("A9_SCIENTIFIC_CONTRACT = PASS")

# ------------------------------------------------------------------
# 6. Guardar config efímero
# ------------------------------------------------------------------

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUNTIME_CONFIG.write_text(
    yaml.safe_dump(
        config_b,
        sort_keys=False,
    ),
    encoding="utf-8",
)

print("RUNTIME_CONFIG =", RUNTIME_CONFIG)

# Reload independiente.
reloaded = load_config(
    RUNTIME_CONFIG
)

assert reloaded["project"]["seed"] == 11
assert reloaded["training"]["epochs"] == 100
assert reloaded["training"]["batch_size"] == 4
assert reloaded["split"]["seed"] == 42
assert reloaded["split"]["allow_create"] is False
assert (
    reloaded["loss"]["false_negative_mask"]["strict"]
    is True
)

print("RUNTIME_CONFIG_RELOAD = PASS")

# ------------------------------------------------------------------
# 7. Preflight oficial otra vez, ahora usando los HDF5 staged
# ------------------------------------------------------------------

print()
print("=" * 80)
print("OFFICIAL PREFLIGHT WITH STAGED HDF5")
print("=" * 80)

preflight = validate_a9_colab_preflight(
    CONFIG_B,
    architecture="model_b_graph_level_relational",
    run_seed=11,
    mutants_hdf5=LOCAL_MUTANTS,
    wt_hdf5=LOCAL_WT,
    output_root=B_OUTPUT_ROOT,
    peer_output_root=A_OUTPUT_ROOT,
    repo_root=REPO,
    expected_commit=EXPECTED_SHA,
    allowed_output_root=ALLOWED_OUTPUT_ROOT,
)

assert preflight["status"] == "PASS"
assert preflight["run_seed"] == 11
assert preflight["split_seed"] == 42
assert preflight["ab_intersection"] == 483
assert preflight["prospective_exists"] is False

print(
    "PROSPECTIVE_RUN_DIR =",
    preflight["prospective_run_dir"],
)

print()
print("=" * 80)
print("B11_FRESH_PREPARATION = PASS")
print("RUN_SEED = 11")
print("TRAINING_EPOCHS = 100")
print("BATCH_SIZE = 4")
print("SPLIT_SEED = 42")
print("RESUME_FROM = NONE")
print("RUN_PRODUCTIVE_QUEUE = False")
print("TRAINING_STARTED = False")
print("=" * 80)

MODEL B A9 — PREPARACIÓN PRODUCTIVA B11 FRESH
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
SCIENTIFIC_CHECKOUT = PASS

STAGING HDF5
LOCAL_FREE_BYTES = 69931171840
MUTANTS_STAGED = {'role': 'mutants', 'drive_locator': '/content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/proc_483p.hdf5', 'local_locator': '/content/model_b_workspace/staging_a9/proc_483p.hdf5', 'sha256': '92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631', 'size_bytes': 33580891, 'reused': False}
WT_STAGED = {'role': 'wt_companion', 'drive_locator': '/content/drive/MyDrive/modelos_proyecto_PKP2/model_a/data/wt_companion.hdf5', 'local_locator': '/content/model_b_workspace/staging_a9/wt_companion.hdf5', 'sha256': '29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a', 'size_bytes': 22961913, 'reused': False}
LOCAL_HDF5_STAGING = PASS

RESOLVE A9 CONFIG
A9_SCIENTIFIC_CONTRACT = PASS
RUNTIME_CONFIG = /content/model_b_workspace/runtime_configs/model_b_a9_seed11_fresh.yaml
RUNTIME_CONFIG_RELOA

In [ ]:
from pathlib import Path
import subprocess

EXPECTED_SHA = "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"

REPO = Path("/content/model_b_workspace/repo")

RUNTIME_CONFIG = Path(
    "/content/model_b_workspace/runtime_configs/"
    "model_b_a9_seed11_fresh.yaml"
)

MODEL_B_RUN_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9/"
    "model_b_graph_level_relational"
)

FAILED_B11_NAME = (
    "run_20260911T120628.981117Z-795be862"
)

RUN_PRODUCTIVE_QUEUE = False
RUN_B11_FRESH = True

print("=" * 80)
print("MODEL B A9 — LAUNCH B11 FRESH")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False
assert RUN_B11_FRESH is True

# ================================================================
# 1. GUARDAS FINALES
# ================================================================

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

dirty = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"],
    text=True,
).strip()

assert head == EXPECTED_SHA
assert dirty == ""
assert RUNTIME_CONFIG.is_file()

failed_b11 = (
    MODEL_B_RUN_ROOT
    / FAILED_B11_NAME
)

assert failed_b11.is_dir()

runs_before = {
    path.name
    for path in MODEL_B_RUN_ROOT.glob("run_*")
    if path.is_dir()
}

print("HEAD =", head)
print("RUNS_BEFORE =", len(runs_before))
print("FAILED_B11_PRESERVED =", failed_b11.is_dir())

# ================================================================
# 2. COMANDO PRODUCTIVO
# ================================================================

command = [
    "python",
    "scripts/train.py",
    "--config",
    str(RUNTIME_CONFIG),
    "--device",
    "cuda",
]

# Fundamental: fresh, nunca resume.
assert "--resume-from" not in command

print()
print("COMMAND:")
print(" ".join(command))

print()
print("RUN_SEED = 11")
print("EXECUTION = FRESH")
print("RESUME_FROM = NONE")
print("QUEUE = DISABLED")
print()

# ================================================================
# 3. ENTRENAMIENTO PRODUCTIVO B11
# ================================================================

result = subprocess.run(
    command,
    cwd=str(REPO),
)

print()
print("TRAIN_RETURN_CODE =", result.returncode)

# ================================================================
# 4. IDENTIFICAR EL ÚNICO RUN NUEVO
# ================================================================

runs_after = {
    path.name
    for path in MODEL_B_RUN_ROOT.glob("run_*")
    if path.is_dir()
}

new_runs = sorted(
    runs_after - runs_before
)

print("NEW_RUNS =", new_runs)

assert len(new_runs) == 1, (
    "Se esperaba exactamente un nuevo run B11"
)

new_run = (
    MODEL_B_RUN_ROOT
    / new_runs[0]
)

print("NEW_B11_RUN =", new_run)

# El B11 fallido histórico debe seguir intacto.
assert failed_b11.is_dir()

print(
    "FAILED_B11_STILL_PRESERVED =",
    failed_b11.is_dir(),
)

# ================================================================
# 5. RESULTADO
# ================================================================

if result.returncode != 0:
    print()
    print("=" * 80)
    print("B11_FRESH = FAILED")
    print("NO_OTHER_SEEDS_LAUNCHED = TRUE")
    print("RUN_PRODUCTIVE_QUEUE = False")
    print("NEW_FAILED_RUN =", new_run)
    print("=" * 80)

    raise RuntimeError(
        f"B11 fresh terminó con código {result.returncode}"
    )

print()
print("=" * 80)
print("B11_FRESH_TRAINING = COMPLETED")
print("NEW_B11_RUN =", new_run)
print("FAILED_B11_PRESERVED = PASS")
print("NO_RESUME = PASS")
print("NO_OTHER_SEEDS_LAUNCHED = TRUE")
print("RUN_PRODUCTIVE_QUEUE = False")
print("=" * 80)

MODEL B A9 — LAUNCH B11 FRESH
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
RUNS_BEFORE = 1
FAILED_B11_PRESERVED = True

COMMAND:
python scripts/train.py --config /content/model_b_workspace/runtime_configs/model_b_a9_seed11_fresh.yaml --device cuda

RUN_SEED = 11
EXECUTION = FRESH
RESUME_FROM = NONE
QUEUE = DISABLED


TRAIN_RETURN_CODE = 0
NEW_RUNS = ['run_20260914T143850.128514Z-e382b31e']
NEW_B11_RUN = /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260914T143850.128514Z-e382b31e
FAILED_B11_STILL_PRESERVED = True

B11_FRESH_TRAINING = COMPLETED
NEW_B11_RUN = /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260914T143850.128514Z-e382b31e
FAILED_B11_PRESERVED = PASS
NO_RESUME = PASS
NO_OTHER_SEEDS_LAUNCHED = TRUE
RUN_PRODUCTIVE_QUEUE = False


In [ ]:
from pathlib import Path
from copy import deepcopy
import subprocess
import json
import math
import sys
import os

# ================================================================
# MODEL B A9 — AUDITORÍA POST-RUN B11
# READ-ONLY
# ================================================================

EXPECTED_SHA = "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"

EXPECTED_MUTANTS_SHA = (
    "92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631"
)

EXPECTED_WT_SHA = (
    "29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a"
)

EXPECTED_COMBINED_SHA = (
    "9a320862a54566d232e1ef2a4eafa468cf900a19186dc0fd7c92aaf0abe06c68"
)

EXPECTED_SPLIT_FP = (
    "8dcc611a6575b95242fda70d2b29e6bf4917a9c781bbeebd851e95e71391d7dd"
)

EXPECTED_ARCHITECTURE = "model_b_graph_level_relational"
EXPECTED_RUN_SEED = 11
EXPECTED_SPLIT_SEED = 42

REPO = Path("/content/model_b_workspace/repo")
SRC = REPO / "src"

RUNTIME_CONFIG = Path(
    "/content/model_b_workspace/runtime_configs/"
    "model_b_a9_seed11_fresh.yaml"
)

RUN_ROOT = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9/"
    "model_b_graph_level_relational"
)

FAILED_B11 = (
    RUN_ROOT
    / "run_20260911T120628.981117Z-795be862"
)

# Si hubiera más de un B11 nuevo, puedes escribir aquí la ruta exacta.
# Normalmente se deja como None.
EXPLICIT_NEW_B11_RUN = None

RUN_PRODUCTIVE_QUEUE = False

print("=" * 80)
print("MODEL B A9 — POST-RUN AUDIT B11")
print("=" * 80)
print("READ_ONLY = True")
print("RUN_PRODUCTIVE_QUEUE =", RUN_PRODUCTIVE_QUEUE)

assert RUN_PRODUCTIVE_QUEUE is False


# ================================================================
# HELPERS
# ================================================================

errors = []

def check(condition, label, detail=None):
    if condition:
        print(f"{label} = PASS")
        return True

    message = label if detail is None else f"{label}: {detail}"
    print(f"{label} = FAIL")
    if detail is not None:
        print("  reason =", detail)
    errors.append(message)
    return False


def load_json(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )


def artifact_path(run_dir, value):
    p = Path(str(value))
    if p.is_absolute():
        return p.resolve()
    return (run_dir / p).resolve()


def finite_tree(value):
    if isinstance(value, dict):
        return all(finite_tree(v) for v in value.values())
    if isinstance(value, list):
        return all(finite_tree(v) for v in value)
    if isinstance(value, bool) or value is None or isinstance(value, str):
        return True
    if isinstance(value, (int, float)):
        return math.isfinite(float(value))
    return True


def dotted(mapping, path):
    value = mapping
    for part in path.split("."):
        value = value[part]
    return value


# ================================================================
# 1. CHECKOUT / ENTORNO
# ================================================================

print()
print("=" * 80)
print("1. SCIENTIFIC CHECKOUT")
print("=" * 80)

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

dirty = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"],
    text=True,
).strip()

print("HEAD =", head)

check(
    head == EXPECTED_SHA,
    "SCIENTIFIC_COMMIT",
    head,
)

check(
    dirty == "",
    "WORKING_TREE_CLEAN",
    repr(dirty),
)


# ================================================================
# 2. PRESERVACIÓN DEL B11 FALLIDO ORIGINAL
# ================================================================

print()
print("=" * 80)
print("2. HISTORICAL FAILED B11")
print("=" * 80)

check(
    FAILED_B11.is_dir(),
    "FAILED_B11_PRESERVED",
    str(FAILED_B11),
)

historical_checkpoints = []

if (FAILED_B11 / "checkpoints").is_dir():
    historical_checkpoints = [
        p
        for p in (FAILED_B11 / "checkpoints").iterdir()
        if p.is_file()
    ]

print(
    "FAILED_B11_CHECKPOINT_FILES =",
    [p.name for p in historical_checkpoints],
)

check(
    len(historical_checkpoints) == 0,
    "FAILED_B11_REMAINS_NO_CHECKPOINTS",
    [p.name for p in historical_checkpoints],
)

old_manifest = FAILED_B11 / "run_manifest.json"

if old_manifest.is_file():
    old_payload = load_json(old_manifest)

    print(
        "FAILED_B11_MANIFEST_STATUS =",
        old_payload.get("status"),
    )

    check(
        old_payload.get("status") != "completed",
        "FAILED_B11_NON_CANONICAL",
        old_payload.get("status"),
    )


# ================================================================
# 3. LOCALIZAR EL NUEVO B11
# ================================================================

print()
print("=" * 80)
print("3. IDENTIFY NEW B11")
print("=" * 80)

if EXPLICIT_NEW_B11_RUN is not None:
    NEW_RUN = Path(
        EXPLICIT_NEW_B11_RUN
    ).resolve()

else:
    candidates = []

    for run_dir in sorted(RUN_ROOT.glob("run_*")):

        if not run_dir.is_dir():
            continue

        if run_dir.resolve() == FAILED_B11.resolve():
            continue

        manifest_path = run_dir / "run_manifest.json"

        if not manifest_path.is_file():
            continue

        try:
            payload = load_json(manifest_path)
        except Exception:
            continue

        architecture = payload.get("architecture")
        seed = (
            payload.get("configuration", {})
            .get("seed")
        )

        if (
            architecture == EXPECTED_ARCHITECTURE
            and seed == EXPECTED_RUN_SEED
        ):
            candidates.append(run_dir.resolve())

    print(
        "B11_CANDIDATES =",
        [str(p) for p in candidates],
    )

    if len(candidates) != 1:
        raise RuntimeError(
            "No se puede identificar inequívocamente el nuevo B11. "
            f"Candidatos encontrados: {len(candidates)}. "
            "Define EXPLICIT_NEW_B11_RUN con la ruta exacta."
        )

    NEW_RUN = candidates[0]


print("NEW_B11_RUN =", NEW_RUN)

check(
    NEW_RUN.is_dir(),
    "NEW_B11_RUN_EXISTS",
    str(NEW_RUN),
)

check(
    NEW_RUN.resolve() != FAILED_B11.resolve(),
    "NEW_B11_IS_DISTINCT_FROM_FAILED_RUN",
)


# ================================================================
# 4. IMPORTS VERSIONADOS
# ================================================================

print()
print("=" * 80)
print("4. VERSIONED AUDIT CODE")
print("=" * 80)

os.chdir(REPO)

for p in (str(REPO), str(SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

from gnn_siamese.config import load_config
from gnn_siamese.training import load_checkpoint

from gnn_siamese.training.a9_contract import (
    validate_a9_run_seed,
    validate_a9_run_acceptance,
)

print("AUDIT_IMPORTS = PASS")


# ================================================================
# 5. RUNTIME CONFIG
# ================================================================

print()
print("=" * 80)
print("5. RUNTIME CONFIG")
print("=" * 80)

check(
    RUNTIME_CONFIG.is_file(),
    "RUNTIME_CONFIG_EXISTS",
    str(RUNTIME_CONFIG),
)

config = load_config(RUNTIME_CONFIG)

validate_a9_run_seed(
    config,
    EXPECTED_RUN_SEED,
)

check(
    config["model"]["architecture"] == EXPECTED_ARCHITECTURE,
    "CONFIG_ARCHITECTURE",
    config["model"]["architecture"],
)

check(
    config["project"]["seed"] == 11,
    "CONFIG_RUN_SEED",
    config["project"]["seed"],
)

check(
    config["training"]["epochs"] == 100,
    "CONFIG_EPOCHS",
    config["training"]["epochs"],
)

check(
    config["training"]["batch_size"] == 4,
    "CONFIG_BATCH_SIZE",
    config["training"]["batch_size"],
)

check(
    config["split"]["seed"] == 42,
    "CONFIG_SPLIT_SEED",
    config["split"]["seed"],
)

check(
    config["split"]["allow_create"] is False,
    "CONFIG_SPLIT_FROZEN",
)

mask_cfg = config["loss"]["false_negative_mask"]

check(
    mask_cfg["enabled"] is True
    and mask_cfg["mode"] == "same_position"
    and mask_cfg["strict"] is True
    and mask_cfg["min_valid_negatives"] == 1
    and float(mask_cfg["min_valid_negative_fraction"]) == 0.0,
    "CONFIG_FALSE_NEGATIVE_MASK",
    mask_cfg,
)


# ================================================================
# 6. ARTEFACTOS OBLIGATORIOS
# ================================================================

print()
print("=" * 80)
print("6. REQUIRED ARTIFACTS")
print("=" * 80)

required = {
    "manifest":
        NEW_RUN / "run_manifest.json",

    "resolved_config":
        NEW_RUN / "config_resolved.yaml",

    "split":
        NEW_RUN / "split.json",

    "gradient_audit":
        NEW_RUN / "gradient_audit.json",

    "best_checkpoint":
        NEW_RUN / "checkpoints" / "best.pt",

    "last_checkpoint":
        NEW_RUN / "checkpoints" / "last.pt",
}

for label, path in required.items():

    exists = (
        path.is_file()
        and path.stat().st_size > 0
    )

    print(
        label,
        "=",
        path,
        "bytes=",
        path.stat().st_size if path.exists() else None,
    )

    check(
        exists,
        f"ARTIFACT_{label.upper()}",
        str(path),
    )


# Si faltan artefactos estructurales, no seguimos con lecturas inconsistentes.
if errors:
    print()
    print("=" * 80)
    print("B11_POSTRUN_AUDIT = FAIL")
    print("EARLY_STRUCTURAL_FAILURE = TRUE")
    print("=" * 80)

    for item in errors:
        print(" -", item)

    raise RuntimeError(
        "La auditoría estructural B11 ha fallado."
    )


# ================================================================
# 7. MANIFEST
# ================================================================

print()
print("=" * 80)
print("7. RUN MANIFEST")
print("=" * 80)

manifest = load_json(
    required["manifest"]
)

status = manifest.get("status")
architecture = manifest.get("architecture")

configuration = manifest.get(
    "configuration",
    {},
)

training = manifest.get(
    "training",
    {},
)

data = manifest.get(
    "data",
    {},
)

losses = manifest.get(
    "losses",
    {},
)

print("status =", status)
print("architecture =", architecture)
print(
    "run_seed =",
    configuration.get("seed"),
)
print(
    "split_seed =",
    configuration.get(
        "seed_bundle",
        {},
    ).get("split"),
)

print(
    "epochs_planned =",
    training.get("epochs_planned"),
)

print(
    "epochs_completed =",
    training.get("epochs_completed"),
)

print(
    "stopped_early =",
    training.get("stopped_early"),
)

print(
    "stop_reason =",
    training.get("stop_reason"),
)

print(
    "resume_from =",
    training.get("resume_from"),
)

check(
    status == "completed",
    "MANIFEST_STATUS_COMPLETED",
    status,
)

check(
    architecture == EXPECTED_ARCHITECTURE,
    "MANIFEST_ARCHITECTURE",
    architecture,
)

check(
    configuration.get("seed") == 11,
    "MANIFEST_RUN_SEED",
    configuration.get("seed"),
)

check(
    configuration.get(
        "seed_bundle",
        {},
    ).get("split") == 42,
    "MANIFEST_SPLIT_SEED",
)

check(
    training.get("resume_from") in (None, ""),
    "FRESH_NO_RESUME",
    training.get("resume_from"),
)

check(
    training.get("epochs_planned") == 100,
    "EPOCHS_PLANNED",
    training.get("epochs_planned"),
)

epochs_completed = int(
    training.get(
        "epochs_completed",
        0,
    )
)

check(
    1 <= epochs_completed <= 100,
    "EPOCHS_COMPLETED_VALID",
    epochs_completed,
)

if epochs_completed < 100:

    check(
        training.get("stopped_early") is True,
        "EARLY_STOPPING_CONSISTENCY",
        {
            "epochs_completed": epochs_completed,
            "stopped_early": training.get("stopped_early"),
            "stop_reason": training.get("stop_reason"),
        },
    )


# ================================================================
# 8. IDENTIDAD DE DATOS Y SPLIT
# ================================================================

print()
print("=" * 80)
print("8. DATA / SPLIT IDENTITY")
print("=" * 80)

fingerprints = data.get(
    "hdf5_content_fingerprint",
    {},
)

files_fp = fingerprints.get(
    "files",
    [],
)

by_role = {
    item.get("role"): item.get("digest")
    for item in files_fp
    if isinstance(item, dict)
}

combined = (
    fingerprints.get(
        "combined",
        {},
    ).get("digest")
)

split_fp = data.get(
    "split_fingerprint"
)

inventory = data.get(
    "inventory",
    {},
)

print(
    "mutants_sha256 =",
    by_role.get("mutants"),
)

print(
    "wt_sha256 =",
    by_role.get("wt_companion"),
)

print(
    "combined_sha256 =",
    combined,
)

print(
    "split_fingerprint =",
    split_fp,
)

print(
    "biological_variants =",
    inventory.get("biological_variants"),
)

print(
    "native_wt_controls =",
    inventory.get("native_wt_controls"),
)

check(
    by_role.get("mutants")
    == EXPECTED_MUTANTS_SHA,
    "MUTANTS_FINGERPRINT",
)

check(
    by_role.get("wt_companion")
    == EXPECTED_WT_SHA,
    "WT_FINGERPRINT",
)

check(
    combined
    == EXPECTED_COMBINED_SHA,
    "COMBINED_FINGERPRINT",
)

check(
    split_fp
    == EXPECTED_SPLIT_FP,
    "SPLIT_FINGERPRINT",
)

check(
    inventory.get("biological_variants")
    == 483,
    "BIOLOGICAL_VARIANTS_483",
)

check(
    inventory.get("native_wt_controls")
    == 1,
    "NATIVE_WT_CONTROLS_1",
)


# ================================================================
# 9. CONFIG RESUELTO DEL RUN
# ================================================================

print()
print("=" * 80)
print("9. RESOLVED RUN CONFIG")
print("=" * 80)

run_config = load_config(
    required["resolved_config"]
)

validate_a9_run_seed(
    run_config,
    11,
)

scientific_paths = [
    "model.architecture",
    "project.seed",

    "training.epochs",
    "training.batch_size",
    "training.optimizer",
    "training.learning_rate",
    "training.weight_decay",
    "training.scheduler",

    "training.early_stopping.enabled",
    "training.early_stopping.monitor",
    "training.early_stopping.mode",
    "training.early_stopping.patience",
    "training.early_stopping.min_delta",

    "loss.main",
    "loss.temperature",

    "loss.false_negative_mask.enabled",
    "loss.false_negative_mask.mode",
    "loss.false_negative_mask.strict",
    "loss.false_negative_mask.min_valid_negatives",
    "loss.false_negative_mask.min_valid_negative_fraction",

    "split.seed",
    "split.allow_create",
]

config_mismatches = {}

for path in scientific_paths:

    expected = dotted(
        config,
        path,
    )

    actual = dotted(
        run_config,
        path,
    )

    if actual != expected:
        config_mismatches[path] = {
            "expected": expected,
            "actual": actual,
        }

print(
    "SCIENTIFIC_CONFIG_MISMATCHES =",
    config_mismatches,
)

check(
    not config_mismatches,
    "RUN_CONFIG_SCIENTIFIC_IDENTITY",
    config_mismatches,
)


# ================================================================
# 10. CHECKPOINTS
# ================================================================

print()
print("=" * 80)
print("10. CHECKPOINTS")
print("=" * 80)

best_checkpoint = load_checkpoint(
    required["best_checkpoint"]
)

last_checkpoint = load_checkpoint(
    required["last_checkpoint"]
)

print(
    "last.epoch_completed =",
    last_checkpoint.get("epoch_completed"),
)

print(
    "best.epoch_completed =",
    best_checkpoint.get("epoch_completed"),
)

print(
    "last.global_step =",
    last_checkpoint.get("global_step"),
)

check(
    last_checkpoint.get(
        "epoch_completed"
    ) == epochs_completed,
    "LAST_CHECKPOINT_EPOCH",
    last_checkpoint.get("epoch_completed"),
)

best_epoch = int(
    best_checkpoint.get(
        "epoch_completed",
        0,
    )
)

check(
    1 <= best_epoch <= epochs_completed,
    "BEST_CHECKPOINT_EPOCH",
    best_epoch,
)

check(
    last_checkpoint.get(
        "split_fingerprint"
    ) == EXPECTED_SPLIT_FP,
    "LAST_CHECKPOINT_SPLIT_FP",
)

check(
    best_checkpoint.get(
        "split_fingerprint"
    ) == EXPECTED_SPLIT_FP,
    "BEST_CHECKPOINT_SPLIT_FP",
)


# ================================================================
# 11. MÉTRICAS
# ================================================================

print()
print("=" * 80)
print("11. METRICS")
print("=" * 80)

metrics_raw = (
    manifest
    .get("artifacts", {})
    .get("metrics")
)

check(
    metrics_raw is not None,
    "MANIFEST_METRICS_REFERENCE",
    metrics_raw,
)

if metrics_raw is not None:

    metrics_path = artifact_path(
        NEW_RUN,
        metrics_raw,
    )

    print(
        "METRICS_PATH =",
        metrics_path,
    )

    check(
        metrics_path.is_file()
        and metrics_path.stat().st_size > 0,
        "METRICS_FILE",
        str(metrics_path),
    )

    if metrics_path.is_file():

        metric_rows = [
            json.loads(line)
            for line in metrics_path.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        print(
            "METRIC_ROWS =",
            len(metric_rows),
        )

        check(
            len(metric_rows)
            == epochs_completed,
            "METRICS_EPOCH_COUNT",
            {
                "rows": len(metric_rows),
                "epochs_completed": epochs_completed,
            },
        )

        check(
            finite_tree(metric_rows),
            "METRICS_FINITE",
        )

        if metric_rows:
            print(
                "FIRST_METRIC_ROW =",
                metric_rows[0],
            )

            print(
                "LAST_METRIC_ROW =",
                metric_rows[-1],
            )


# ================================================================
# 12. GRADIENT AUDIT + A9 CORE ACCEPTANCE
# ================================================================

print()
print("=" * 80)
print("12. CORE A9 ACCEPTANCE")
print("=" * 80)

gradient_payload = load_json(
    required["gradient_audit"]
)

print(
    "GRADIENT_MODULES =",
    sorted(gradient_payload),
)

# El validador oficial incluye todas las comprobaciones de:
# - status completed
# - HDF5
# - split
# - 483 + WT
# - contrato de entrenamiento
# - módulos activos
# - gradientes finitos/no cero
# - cambio de pesos
#
# Para esta auditoría inmediata pos-entrenamiento dejamos
# las representaciones para la fase posterior específica.

core_config = deepcopy(config)

required_representations = (
    core_config
    .get("a9", {})
    .get("acceptance", {})
    .get(
        "required_pair_representations",
        [],
    )
)

print(
    "REPRESENTATIONS_REQUIRED_LATER =",
    required_representations,
)

core_config.setdefault(
    "a9",
    {},
).setdefault(
    "acceptance",
    {},
)["required_pair_representations"] = []

try:

    core_acceptance = (
        validate_a9_run_acceptance(
            NEW_RUN,
            core_config,
            representations=None,
        )
    )

    print(
        "CORE_ACCEPTANCE_RESULT =",
        core_acceptance,
    )

    check(
        core_acceptance.get("status")
        == "accepted",
        "CORE_A9_ACCEPTANCE",
        core_acceptance,
    )

except Exception as exc:

    check(
        False,
        "CORE_A9_ACCEPTANCE",
        f"{type(exc).__name__}: {exc}",
    )


# ================================================================
# 13. ASEGURAR QUE NO SE LANZARON OTRAS SEEDS
# ================================================================

print()
print("=" * 80)
print("13. OTHER PRODUCTIVE SEEDS")
print("=" * 80)

other_productive_runs = []

for run_dir in sorted(
    RUN_ROOT.glob("run_*")
):

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    if not manifest_path.is_file():
        continue

    try:
        payload = load_json(
            manifest_path
        )
    except Exception:
        continue

    seed = (
        payload
        .get(
            "configuration",
            {},
        )
        .get("seed")
    )

    if seed in {23, 37, 41, 53}:

        other_productive_runs.append(
            {
                "seed": seed,
                "run": str(run_dir),
                "status": payload.get("status"),
            }
        )

print(
    "OTHER_PRODUCTIVE_RUNS =",
    other_productive_runs,
)

check(
    len(other_productive_runs) == 0,
    "NO_OTHER_SEEDS_LAUNCHED",
    other_productive_runs,
)


# ================================================================
# 14. NO TEMP FILES
# ================================================================

print()
print("=" * 80)
print("14. RUN DIRECTORY SANITY")
print("=" * 80)

tmp_files = list(
    NEW_RUN.rglob(".*.tmp")
)

print(
    "TEMP_FILES =",
    [str(p) for p in tmp_files],
)

check(
    len(tmp_files) == 0,
    "NO_ATOMIC_TEMP_FILES",
)


# ================================================================
# 15. RESULTADO
# ================================================================

print()
print("=" * 80)
print("15. FINAL RESULT")
print("=" * 80)

print("NEW_B11_RUN =", NEW_RUN)
print("FAILED_B11 =", FAILED_B11)
print("RUN_PRODUCTIVE_QUEUE = False")

if required_representations:

    print(
        "REPRESENTATION_ACCEPTANCE = DEFERRED"
    )

    print(
        "REPRESENTATIONS_PENDING =",
        required_representations,
    )

else:

    print(
        "REPRESENTATION_ACCEPTANCE = NOT_REQUIRED"
    )


if errors:

    print()
    print("B11_POSTRUN_AUDIT = FAIL")
    print("FAILURES =", len(errors))

    for index, error in enumerate(
        errors,
        start=1,
    ):
        print(
            f"{index}. {error}"
        )

    print("=" * 80)

    raise RuntimeError(
        "B11 post-run audit failed."
    )


print()
print("B11_CORE_TRAINING_ACCEPTANCE = PASS")
print("FAILED_B11_PRESERVED = PASS")
print("FRESH_NO_RESUME = PASS")
print("NO_OTHER_SEEDS_LAUNCHED = PASS")
print("RUN_PRODUCTIVE_QUEUE = False")

if required_representations:
    print(
        "FINAL_A9_ACCEPTANCE = "
        "PENDING_REPRESENTATION_AUDIT"
    )
else:
    print(
        "FINAL_A9_ACCEPTANCE = PASS"
    )

print("B11_POSTRUN_AUDIT = PASS")
print("=" * 80)

MODEL B A9 — POST-RUN AUDIT B11
READ_ONLY = True
RUN_PRODUCTIVE_QUEUE = False

1. SCIENTIFIC CHECKOUT
HEAD = 4af06bd729b5ef04111852482aebe63c9363f543
SCIENTIFIC_COMMIT = FAIL
  reason = 4af06bd729b5ef04111852482aebe63c9363f543
WORKING_TREE_CLEAN = PASS

2. HISTORICAL FAILED B11
FAILED_B11_PRESERVED = PASS
FAILED_B11_CHECKPOINT_FILES = []
FAILED_B11_REMAINS_NO_CHECKPOINTS = PASS
FAILED_B11_MANIFEST_STATUS = failed
FAILED_B11_NON_CANONICAL = PASS

3. IDENTIFY NEW B11
B11_CANDIDATES = ['/content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260914T143850.128514Z-e382b31e']
NEW_B11_RUN = /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260914T143850.128514Z-e382b31e
NEW_B11_RUN_EXISTS = PASS
NEW_B11_IS_DISTINCT_FROM_FAILED_RUN = PASS

4. VERSIONED AUDIT CODE
AUDIT_IMPORTS = PASS

5. RUNTIME CONFIG
RUNTIME_CONFIG_EXISTS = FAIL
  reason = /content/model_b_workspace/runtime_configs/mode

ConfigError: Configuration file does not exist: /content/model_b_workspace/runtime_configs/model_b_a9_seed11_fresh.yaml

In [ ]:
# =============================================================================
# MODEL B A9 — B11 REPRESENTATION AUDIT
# READ-ONLY / BEST CHECKPOINT / NO OTHER SEEDS
# =============================================================================

from pathlib import Path
import os
import sys
import json
import subprocess

import numpy as np
import torch
from torch.utils.data import DataLoader


# =============================================================================
# 0. CONSTANTES
# =============================================================================

EXPECTED_COMMIT = (
    "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"
)

REPO_DIR = Path(
    "/content/model_b_workspace/repo"
)

B11_RUN = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9/model_b_graph_level_relational/"
    "run_20260914T143850.128514Z-e382b31e"
)

BEST_PT = (
    B11_RUN
    / "checkpoints"
    / "best.pt"
)

LAST_PT = (
    B11_RUN
    / "checkpoints"
    / "last.pt"
)

RESOLVED_CONFIG = (
    B11_RUN
    / "config_resolved.yaml"
)

MANIFEST_PATH = (
    B11_RUN
    / "run_manifest.json"
)

REQUIRED_REPRESENTATIONS = [
    "r_delta",
    "z_delta",
    "z_instance_pair",
]

RUN_PRODUCTIVE_QUEUE = False


# =============================================================================
# 1. FAIL-CLOSED PRECHECK
# =============================================================================

print("=" * 80)
print("MODEL B A9 — B11 REPRESENTATION AUDIT")
print("=" * 80)

print("READ_ONLY = True")
print(
    "RUN_PRODUCTIVE_QUEUE =",
    RUN_PRODUCTIVE_QUEUE,
)

assert RUN_PRODUCTIVE_QUEUE is False

assert REPO_DIR.is_dir()
assert B11_RUN.is_dir()

for p in [
    BEST_PT,
    LAST_PT,
    RESOLVED_CONFIG,
    MANIFEST_PATH,
]:

    assert p.is_file(), (
        f"Missing artifact: {p}"
    )

    assert p.stat().st_size > 0

os.chdir(REPO_DIR)

head = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "status",
        "--short",
    ],
    text=True,
).strip()

print()
print("HEAD =", head)
print(
    "WORKING_TREE_STATUS =",
    repr(status),
)

assert head == EXPECTED_COMMIT
assert status == ""

print("SCIENTIFIC_CHECKOUT = PASS")


# =============================================================================
# 2. IMPORTS VERSIONADOS
# =============================================================================

src = REPO_DIR / "src"

for candidate in [
    str(REPO_DIR),
    str(src),
]:

    if candidate not in sys.path:
        sys.path.insert(
            0,
            candidate,
        )

from gnn_siamese.config import (
    load_config,
)

from gnn_siamese.builders import (
    build_dataset_bundle,
    build_model,
)

from gnn_siamese.data import (
    collate_mut_wt_pairs,
)

from gnn_siamese.training import (
    load_checkpoint,
)

from gnn_siamese.training.a9_contract import (
    validate_a9_run_acceptance,
)

print("AUDIT_IMPORTS = PASS")


# =============================================================================
# 3. MANIFEST + CHECKPOINT IDENTITY
# =============================================================================

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

best_ckpt = load_checkpoint(
    BEST_PT
)

last_ckpt = load_checkpoint(
    LAST_PT
)

print()
print("=" * 80)
print("CHECKPOINT IDENTITY")
print("=" * 80)

print(
    "manifest.status =",
    manifest["status"],
)

print(
    "manifest.architecture =",
    manifest["architecture"],
)

print(
    "best.epoch_completed =",
    best_ckpt["epoch_completed"],
)

print(
    "last.epoch_completed =",
    last_ckpt["epoch_completed"],
)

print(
    "best.split_fingerprint =",
    best_ckpt["split_fingerprint"],
)

print(
    "last.split_fingerprint =",
    last_ckpt["split_fingerprint"],
)

assert (
    manifest["status"]
    == "completed"
)

assert (
    manifest["architecture"]
    == "model_b_graph_level_relational"
)

assert (
    int(
        best_ckpt["epoch_completed"]
    )
    == 24
)

assert (
    int(
        last_ckpt["epoch_completed"]
    )
    == 39
)

assert (
    best_ckpt["split_fingerprint"]
    == last_ckpt["split_fingerprint"]
)

print(
    "BEST_CHECKPOINT_SELECTION = PASS"
)

print(
    "REPRESENTATION_CHECKPOINT = best.pt"
)

print(
    "REPRESENTATION_EPOCH = 24"
)


# =============================================================================
# 4. CONFIG EXACTA DEL RUN
# =============================================================================

config = load_config(
    RESOLVED_CONFIG
)

assert (
    config["model"]["architecture"]
    == "model_b_graph_level_relational"
)

assert (
    int(
        config["project"]["seed"]
    )
    == 11
)

assert (
    int(
        config["split"]["seed"]
    )
    == 42
)

assert (
    config["split"]["allow_create"]
    is False
)

required_from_config = list(
    config
    .get("a9", {})
    .get("acceptance", {})
    .get(
        "required_pair_representations",
        [],
    )
)

print()
print("=" * 80)
print("RESOLVED CONFIG")
print("=" * 80)

print(
    "architecture =",
    config["model"]["architecture"],
)

print(
    "run_seed =",
    config["project"]["seed"],
)

print(
    "split_seed =",
    config["split"]["seed"],
)

print(
    "required_pair_representations =",
    required_from_config,
)

assert (
    required_from_config
    == REQUIRED_REPRESENTATIONS
)

print("RESOLVED_CONFIG = PASS")


# =============================================================================
# 5. RECONSTRUIR DATASET + MODEL
# =============================================================================

print()
print("=" * 80)
print("REBUILD MODEL / DATASET")
print("=" * 80)

dataset_bundle = build_dataset_bundle(
    config
)

dataset = dataset_bundle.dataset

print(
    "BIOLOGICAL_DATASET_SIZE =",
    len(dataset),
)

assert len(dataset) == 483

model = build_model(
    config,
    dataset,
)

assert (
    getattr(
        model,
        "architecture_name",
        None,
    )
    == "model_b_graph_level_relational"
)

assert hasattr(
    model,
    "siamese_model",
)

print(
    "MODEL_CLASS =",
    model.__class__.__name__,
)

print(
    "MODEL_ARCHITECTURE =",
    model.architecture_name,
)

print("MODEL_REBUILD = PASS")


# =============================================================================
# 6. LOAD BEST.PT STRICT
# =============================================================================

missing, unexpected = (
    model.load_state_dict(
        best_ckpt["model_state_dict"],
        strict=True,
    )
)

print()
print("=" * 80)
print("LOAD BEST CHECKPOINT")
print("=" * 80)

print(
    "MISSING_STATE_KEYS =",
    list(missing),
)

print(
    "UNEXPECTED_STATE_KEYS =",
    list(unexpected),
)

assert len(missing) == 0
assert len(unexpected) == 0

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(
    device
)

model.eval()

print(
    "DEVICE =",
    device,
)

print(
    "STRICT_STATE_DICT_LOAD = PASS"
)


# =============================================================================
# 7. DETERMINISTIC LOADER — ALL 483
# =============================================================================

representation_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_mut_wt_pairs,
)

print()
print("=" * 80)
print("REPRESENTATION LOADER")
print("=" * 80)

print(
    "NUM_VARIANTS =",
    len(dataset),
)

print(
    "NUM_BATCHES =",
    len(representation_loader),
)

print("SHUFFLE = False")
print("AUGMENTATION = False")
print(
    "REPRESENTATION_LOADER = PASS"
)


# =============================================================================
# 8. EXTRACT REPRESENTATIONS
# =============================================================================

collected = {
    name: []
    for name
    in REQUIRED_REPRESENTATIONS
}

variant_ids = []

with torch.inference_mode():

    for batch_index, batch in enumerate(
        representation_loader,
        start=1,
    ):

        batch = batch.to(
            device
        )

        output = (
            model.siamese_model(
                graph_mut=batch.graph_mut,
                graph_wt=batch.graph_wt,
                allow_trainable_z_delta=True,
            )
        )

        assert (
            output.r_delta
            is not None
        )

        assert (
            output.z_delta
            is not None
        )

        assert (
            output.z_instance_pair
            is not None
        )

        tensors = {
            "r_delta":
                output.r_delta,

            "z_delta":
                output.z_delta,

            "z_instance_pair":
                output.z_instance_pair,
        }

        for name, tensor in (
            tensors.items()
        ):

            assert tensor.ndim == 2

            assert (
                tensor.shape[0]
                == batch.batch_size
            )

            assert (
                torch.isfinite(
                    tensor
                ).all()
            ), (
                f"{name} contains NaN/Inf "
                f"in batch {batch_index}"
            )

            collected[
                name
            ].append(
                tensor
                .detach()
                .cpu()
            )

        variant_ids.extend(
            list(
                batch.variant_ids
            )
        )


# =============================================================================
# 9. CONCATENATE + BASIC QC
# =============================================================================

representations = {
    name:
        torch.cat(
            parts,
            dim=0,
        ).numpy()

    for name, parts
    in collected.items()
}

print()
print("=" * 80)
print("EXTRACTED REPRESENTATIONS")
print("=" * 80)

print(
    "VARIANT_IDS =",
    len(variant_ids),
)

print(
    "UNIQUE_VARIANT_IDS =",
    len(
        set(
            variant_ids
        )
    ),
)

assert (
    len(variant_ids)
    == 483
)

assert (
    len(
        set(
            variant_ids
        )
    )
    == 483
)

for name in (
    REQUIRED_REPRESENTATIONS
):

    matrix = representations[
        name
    ]

    print()
    print(
        name,
        "shape =",
        matrix.shape,
    )

    print(
        name,
        "dtype =",
        matrix.dtype,
    )

    print(
        name,
        "finite =",
        bool(
            np.isfinite(
                matrix
            ).all()
        ),
    )

    unique_rows = (
        np.unique(
            matrix,
            axis=0,
        ).shape[0]
    )

    print(
        name,
        "unique_rows =",
        unique_rows,
    )

    print(
        name,
        "mean =",
        float(
            matrix.mean()
        ),
    )

    print(
        name,
        "std =",
        float(
            matrix.std()
        ),
    )

    assert matrix.ndim == 2
    assert matrix.shape[0] == 483
    assert matrix.shape[1] >= 1

    assert (
        np.isfinite(
            matrix
        ).all()
    )

    assert unique_rows > 1

print()
print(
    "REPRESENTATION_EXTRACTION = PASS"
)

print(
    "REPRESENTATION_BASIC_QC = PASS"
)


# =============================================================================
# 10. OFFICIAL A9 ACCEPTANCE IN MEMORY
# =============================================================================

print()
print("=" * 80)
print(
    "OFFICIAL A9 REPRESENTATION VALIDATION"
)
print("=" * 80)

acceptance = (
    validate_a9_run_acceptance(
        B11_RUN,
        config,
        representations=representations,
    )
)

print(
    json.dumps(
        acceptance,
        indent=2,
        sort_keys=True,
    )
)

assert (
    acceptance["status"]
    == "accepted"
)

assert (
    acceptance[
        "automatic_rejection_checks"
    ]
    == "PASS"
)

metrics = (
    acceptance[
        "representation_metrics"
    ]
)

assert (
    set(metrics)
    == set(
        REQUIRED_REPRESENTATIONS
    )
)

print()
print(
    "REPRESENTATION_ACCEPTANCE = PASS"
)

print(
    "B11_FINAL_A9_READ_ONLY_ACCEPTANCE = PASS"
)


# =============================================================================
# 11. CONFIRM NO OTHER PRODUCTIVE SEEDS
# =============================================================================

runs_root = B11_RUN.parent

other_productive = []

for run_dir in sorted(
    runs_root.glob(
        "run_*"
    )
):

    if run_dir == B11_RUN:
        continue

    manifest_file = (
        run_dir
        / "run_manifest.json"
    )

    if not manifest_file.is_file():
        continue

    try:

        payload = json.loads(
            manifest_file.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue

    seed = (
        payload
        .get(
            "configuration",
            {},
        )
        .get(
            "seed"
        )
    )

    status_value = (
        payload.get(
            "status"
        )
    )

    if (
        seed
        in {
            23,
            37,
            41,
            53,
        }
        and
        status_value
        in {
            "running",
            "completed",
        }
    ):

        other_productive.append(
            {
                "run":
                    str(
                        run_dir
                    ),

                "seed":
                    seed,

                "status":
                    status_value,
            }
        )

print()
print("=" * 80)
print("QUEUE / OTHER SEEDS")
print("=" * 80)

print(
    "OTHER_PRODUCTIVE_RUNS =",
    other_productive,
)

assert (
    other_productive
    == []
)

print(
    "NO_OTHER_SEEDS_LAUNCHED = PASS"
)

print(
    "RUN_PRODUCTIVE_QUEUE =",
    RUN_PRODUCTIVE_QUEUE,
)


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 80)
print(
    "B11 REPRESENTATION AUDIT — FINAL"
)
print("=" * 80)

print(
    "RUN =",
    B11_RUN,
)

print(
    "CHECKPOINT = best.pt"
)

print(
    "CHECKPOINT_EPOCH = 24"
)

print(
    "VARIANTS = 483"
)

for name in (
    REQUIRED_REPRESENTATIONS
):

    print(
        f"{name} =",
        representations[
            name
        ].shape,
    )

print()

print(
    "REPRESENTATION_EXTRACTION = PASS"
)

print(
    "REPRESENTATION_BASIC_QC = PASS"
)

print(
    "REPRESENTATION_ACCEPTANCE = PASS"
)

print(
    "B11_FINAL_A9_READ_ONLY_ACCEPTANCE = PASS"
)

print(
    "NO_OTHER_SEEDS_LAUNCHED = PASS"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print(
    "FILES_WRITTEN = 0"
)

print("=" * 80)

MODEL B A9 — B11 REPRESENTATION AUDIT
READ_ONLY = True
RUN_PRODUCTIVE_QUEUE = False

HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
WORKING_TREE_STATUS = ''
SCIENTIFIC_CHECKOUT = PASS
AUDIT_IMPORTS = PASS

CHECKPOINT IDENTITY
manifest.status = completed
manifest.architecture = model_b_graph_level_relational
best.epoch_completed = 24
last.epoch_completed = 39
best.split_fingerprint = 8dcc611a6575b95242fda70d2b29e6bf4917a9c781bbeebd851e95e71391d7dd
last.split_fingerprint = 8dcc611a6575b95242fda70d2b29e6bf4917a9c781bbeebd851e95e71391d7dd
BEST_CHECKPOINT_SELECTION = PASS
REPRESENTATION_CHECKPOINT = best.pt
REPRESENTATION_EPOCH = 24

RESOLVED CONFIG
architecture = model_b_graph_level_relational
run_seed = 11
split_seed = 42
required_pair_representations = ['r_delta', 'z_delta', 'z_instance_pair']
RESOLVED_CONFIG = PASS

REBUILD MODEL / DATASET
BIOLOGICAL_DATASET_SIZE = 483
MODEL_CLASS = ModelBGraphLevelRelationalContrastive
MODEL_ARCHITECTURE = model_b_graph_level_relational
MODEL_REBUILD 

In [ ]:
# =============================================================================
# MODEL B A9 — MATERIALIZE OFFICIAL B11 ACCEPTANCE
# Uses already-validated in-memory representations
# NO TRAINING / NO OTHER SEEDS
# =============================================================================

from pathlib import Path
import hashlib
import json
import subprocess
import sys
import numpy as np

print("=" * 80)
print("MODEL B A9 — MATERIALIZE B11 OFFICIAL ACCEPTANCE")
print("=" * 80)

EXPECTED_SHA = "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"

REPO = Path("/content/model_b_workspace/repo")

B11_RUN = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2/"
    "model_b/runs/model_b_a9/model_b_graph_level_relational/"
    "run_20260914T143850.128514Z-e382b31e"
)

RESOLVED_CONFIG = (
    B11_RUN
    / "config_resolved.yaml"
)

ACCEPTANCE_PATH = (
    B11_RUN
    / "a9_acceptance.json"
)

ACCEPTANCE_SCRIPT = (
    REPO
    / "scripts"
    / "a9_acceptance.py"
)

TEMP_DIR = Path(
    "/content/model_b_workspace/a9_acceptance_inputs"
)

TEMP_NPZ = (
    TEMP_DIR
    / "b11_best_representations.npz"
)

REQUIRED_REPRESENTATIONS = [
    "r_delta",
    "z_delta",
    "z_instance_pair",
]

RUN_PRODUCTIVE_QUEUE = False


# =============================================================================
# 1. FAIL-CLOSED PRECHECK
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False

assert REPO.is_dir()
assert B11_RUN.is_dir()
assert RESOLVED_CONFIG.is_file()
assert ACCEPTANCE_SCRIPT.is_file()

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()

print("HEAD =", head)
print("WORKING_TREE =", repr(status))

assert head == EXPECTED_SHA
assert status == ""

print("SCIENTIFIC_CHECKOUT = PASS")


# =============================================================================
# 2. VERIFY THE ALREADY EXTRACTED REPRESENTATIONS
# =============================================================================

assert "representations" in globals(), (
    "No existe la variable `representations`. "
    "No regenere aceptación sin volver a ejecutar primero "
    "B11 REPRESENTATION AUDIT."
)

assert set(
    REQUIRED_REPRESENTATIONS
).issubset(
    representations.keys()
)

for name in REQUIRED_REPRESENTATIONS:

    matrix = np.asarray(
        representations[name]
    )

    print(
        name,
        "shape =",
        matrix.shape,
    )

    assert matrix.ndim == 2
    assert matrix.shape[0] == 483
    assert matrix.shape[1] >= 1

    assert np.isfinite(
        matrix
    ).all()

    assert (
        np.unique(
            matrix,
            axis=0,
        ).shape[0]
        == 483
    )

print(
    "IN_MEMORY_REPRESENTATIONS = PASS"
)


# =============================================================================
# 3. PROTECT AGAINST ACCIDENTAL OVERWRITE
# =============================================================================

if ACCEPTANCE_PATH.exists():

    print()
    print(
        "A9_ACCEPTANCE_ALREADY_EXISTS =",
        ACCEPTANCE_PATH,
    )

    existing = json.loads(
        ACCEPTANCE_PATH.read_text(
            encoding="utf-8"
        )
    )

    print(
        json.dumps(
            existing,
            indent=2,
            sort_keys=True,
        )
    )

    assert (
        existing.get("status")
        == "accepted"
    ), (
        "Existe a9_acceptance.json pero no está accepted. "
        "No se sobrescribe automáticamente."
    )

    assert (
        int(
            existing.get(
                "run_seed",
                -1,
            )
        )
        == 11
    )

    assert (
        existing.get(
            "architecture"
        )
        == "model_b_graph_level_relational"
    )

    assert (
        existing.get(
            "automatic_rejection_checks"
        )
        == "PASS"
    )

    assert (
        set(
            existing
            .get(
                "representation_metrics",
                {},
            )
            .keys()
        )
        == set(
            REQUIRED_REPRESENTATIONS
        )
    )

    MATERIALIZATION_NEEDED = False

    print(
        "EXISTING_ACCEPTANCE_VALID = PASS"
    )

else:

    MATERIALIZATION_NEEDED = True

    print()
    print(
        "A9_ACCEPTANCE_ALREADY_EXISTS = NO"
    )


# =============================================================================
# 4. TEMPORARY NPZ INPUT FOR OFFICIAL VERSIONED SCRIPT
# =============================================================================

if MATERIALIZATION_NEEDED:

    TEMP_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        TEMP_NPZ,
        **{
            name:
                np.asarray(
                    representations[name]
                )

            for name
            in REQUIRED_REPRESENTATIONS
        }
    )

    assert TEMP_NPZ.is_file()
    assert TEMP_NPZ.stat().st_size > 0


    def sha256(path):
        digest = hashlib.sha256()

        with Path(path).open("rb") as handle:

            while True:

                chunk = handle.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                digest.update(chunk)

        return digest.hexdigest()


    print()
    print(
        "TEMP_REPRESENTATIONS_NPZ =",
        TEMP_NPZ,
    )

    print(
        "TEMP_REPRESENTATIONS_SHA256 =",
        sha256(TEMP_NPZ),
    )


# =============================================================================
# 5. OFFICIAL VERSIONED ACCEPTANCE SCRIPT
# =============================================================================

if MATERIALIZATION_NEEDED:

    command = [
        sys.executable,
        str(ACCEPTANCE_SCRIPT),

        "--run-dir",
        str(B11_RUN),

        "--config",
        str(RESOLVED_CONFIG),

        "--pair-representations",
        str(TEMP_NPZ),
    ]

    print()
    print("COMMAND:")
    print(
        " ".join(command)
    )

    result = subprocess.run(
        command,
        cwd=REPO,
        text=True,
        capture_output=True,
    )

    print()
    print(
        "ACCEPTANCE_RETURN_CODE =",
        result.returncode,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    assert result.returncode == 0, (
        "Official A9 acceptance script failed."
    )


# =============================================================================
# 6. VERIFY PERSISTED ACCEPTANCE
# =============================================================================

assert ACCEPTANCE_PATH.is_file()
assert ACCEPTANCE_PATH.stat().st_size > 0

acceptance = json.loads(
    ACCEPTANCE_PATH.read_text(
        encoding="utf-8"
    )
)

print()
print("=" * 80)
print("PERSISTED A9 ACCEPTANCE")
print("=" * 80)

print(
    json.dumps(
        acceptance,
        indent=2,
        sort_keys=True,
    )
)

assert (
    acceptance["status"]
    == "accepted"
)

assert (
    int(
        acceptance["run_seed"]
    )
    == 11
)

assert (
    acceptance["architecture"]
    == "model_b_graph_level_relational"
)

assert (
    acceptance[
        "automatic_rejection_checks"
    ]
    == "PASS"
)

assert (
    set(
        acceptance[
            "representation_metrics"
        ]
    )
    == set(
        REQUIRED_REPRESENTATIONS
    )
)

for name in REQUIRED_REPRESENTATIONS:

    metrics = (
        acceptance[
            "representation_metrics"
        ][name]
    )

    assert metrics["finite"] is True

    assert (
        metrics[
            "exact_total_collapse"
        ]
        is False
    )

    assert (
        int(
            metrics[
                "exact_unique_rows"
            ]
        )
        == 483
    )

    assert (
        int(
            metrics["shape"][0]
        )
        == 483
    )

print()
print(
    "PERSISTED_ACCEPTANCE_CONTENT = PASS"
)


# =============================================================================
# 7. SHA256 OF OFFICIAL RESULT
# =============================================================================

def file_sha256(path):

    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


acceptance_sha = file_sha256(
    ACCEPTANCE_PATH
)

print(
    "A9_ACCEPTANCE_PATH =",
    ACCEPTANCE_PATH,
)

print(
    "A9_ACCEPTANCE_SHA256 =",
    acceptance_sha,
)


# =============================================================================
# 8. REMOVE ONLY THE EPHEMERAL NPZ
# =============================================================================

if TEMP_NPZ.exists():

    TEMP_NPZ.unlink()

assert not TEMP_NPZ.exists()

print(
    "TEMP_NPZ_REMOVED = PASS"
)


# =============================================================================
# 9. VERIFY NO B23/B37/B41/B53 PRODUCTIVE RUNS
# =============================================================================

other_productive = []

runs_root = B11_RUN.parent

for run_dir in sorted(
    runs_root.glob(
        "run_*"
    )
):

    if run_dir == B11_RUN:
        continue

    manifest_file = (
        run_dir
        / "run_manifest.json"
    )

    if not manifest_file.is_file():
        continue

    try:

        payload = json.loads(
            manifest_file.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue

    seed = (
        payload
        .get(
            "configuration",
            {},
        )
        .get(
            "seed"
        )
    )

    run_status = payload.get(
        "status"
    )

    if (
        seed
        in {
            23,
            37,
            41,
            53,
        }
        and
        run_status
        in {
            "running",
            "completed",
        }
    ):

        other_productive.append(
            {
                "run":
                    str(run_dir),

                "seed":
                    seed,

                "status":
                    run_status,
            }
        )

print()
print(
    "OTHER_PRODUCTIVE_RUNS =",
    other_productive,
)

assert other_productive == []

print(
    "NO_OTHER_SEEDS_LAUNCHED = PASS"
)


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 80)

print(
    "B11_OFFICIAL_A9_ACCEPTANCE = PASS"
)

print(
    "B11_A9_STATUS = CLOSED_ACCEPTED"
)

print(
    "READY_FOR_B23 = YES"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_NEW_RUN_CREATED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)

MODEL B A9 — MATERIALIZE B11 OFFICIAL ACCEPTANCE
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
WORKING_TREE = ''
SCIENTIFIC_CHECKOUT = PASS
r_delta shape = (483, 640)
z_delta shape = (483, 128)
z_instance_pair shape = (483, 64)
IN_MEMORY_REPRESENTATIONS = PASS

A9_ACCEPTANCE_ALREADY_EXISTS = NO

TEMP_REPRESENTATIONS_NPZ = /content/model_b_workspace/a9_acceptance_inputs/b11_best_representations.npz
TEMP_REPRESENTATIONS_SHA256 = aca8ff746537fc0f6e1c2eeece02a11a356785bf6ad4cc66a3f7eccf881b652b

COMMAND:
/usr/bin/python3 /content/model_b_workspace/repo/scripts/a9_acceptance.py --run-dir /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260914T143850.128514Z-e382b31e --config /content/drive/MyDrive/modelos_proyecto_PKP2/model_b/runs/model_b_a9/model_b_graph_level_relational/run_20260914T143850.128514Z-e382b31e/config_resolved.yaml --pair-representations /content/model_b_workspace/a9_acceptance_inputs/b11_best_representations.npz

ACC

# Prueba seeds b23-B53

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B1 — PREPARE + AUDIT CONFIGS FOR B23 / B37 / B41 / B53
#
# NO TRAINING
# NO RUN CREATION
# WRITES ONLY EPHEMERAL YAML FILES UNDER /content
# =============================================================================

from pathlib import Path
from copy import deepcopy
import subprocess
import json

from scripts.a9_preflight import resolve_a9_runtime_configs
from gnn_siamese.config import save_config, load_config


print("=" * 80)
print("MODEL B A9 — FAST-B1 PREPARE REMAINING PRODUCTIVE SEEDS")
print("=" * 80)

# =============================================================================
# 0. SAFETY
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False

REMAINING_SEEDS = (23, 37, 41, 53)

assert NEXT_SEED == 23
assert NEXT_ACTION == "FRESH_READY"

# B11 must already be closed/accepted.
assert seed_state_map[11] == "CLOSED_ACCEPTED"

# The four remaining productive seeds must still be absent.
for seed in REMAINING_SEEDS:
    assert seed_state_map[seed] == "ABSENT", (
        f"Seed {seed} is not ABSENT: {seed_state_map[seed]}"
    )

print("B11_CLOSED_ACCEPTED = PASS")
print("REMAINING_SEEDS_ABSENT = PASS")


# =============================================================================
# 1. PATHS
# =============================================================================

BASE_B_CONFIG = (
    REPO
    / "configs"
    / "model_b_a9.yaml"
)

FROZEN_SPLIT = (
    REPO
    / "splits"
    / "leave_position_out_seed_42.json"
)

MODEL_A_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_a"
    / "runs"
    / "model_a_a9"
)

MODEL_B_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
)

FAST_CONFIG_ROOT = (
    LOCAL_ROOT
    / "runtime_configs"
    / "fast_remaining_b"
)

FAST_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert BASE_B_CONFIG.is_file()
assert FROZEN_SPLIT.is_file()
assert LOCAL_MUTANTS_HDF5.is_file()
assert LOCAL_WT_HDF5.is_file()

print("INPUT_PATHS = PASS")


# =============================================================================
# 2. GIT IDENTITY
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()

assert head == EXPECTED_SHA
assert status == ""

print("SCIENTIFIC_CHECKOUT = PASS")


# =============================================================================
# 3. GENERATE 4 OFFICIAL RUNTIME CONFIGS
# =============================================================================

runtime_configs = {}
runtime_paths = {}

for seed in REMAINING_SEEDS:

    config_a, config_b = resolve_a9_runtime_configs(
        BASE_B_CONFIG,
        architecture="model_b_graph_level_relational",
        run_seed=seed,
        mutants_hdf5=LOCAL_MUTANTS_HDF5,
        wt_hdf5=LOCAL_WT_HDF5,
        output_root=MODEL_B_A9_ROOT,
        peer_output_root=MODEL_A_A9_ROOT,
        repo_root=REPO,
    )

    # Pin exact frozen split in the ephemeral runtime config.
    config_b["split"]["persist_path"] = str(
        FROZEN_SPLIT.resolve()
    )

    assert config_b["split"]["allow_create"] is False

    # Remove internal runtime-only metadata before saving.
    clean = {
        key: value
        for key, value in config_b.items()
        if not str(key).startswith("__")
    }

    target = (
        FAST_CONFIG_ROOT
        / f"model_b_a9_seed{seed}_fresh.yaml"
    )

    if target.exists():

        existing = load_config(target)

        assert existing == clean, (
            f"Existing config for seed {seed} differs "
            "from the expected runtime config."
        )

        print(
            f"B{seed}_CONFIG_ALREADY_EXISTS = VALID"
        )

    else:

        save_config(
            clean,
            target,
        )

        print(
            f"B{seed}_CONFIG_CREATED = PASS"
        )

    # Reload exactly what train.py would consume.
    loaded = load_config(target)

    runtime_configs[seed] = loaded
    runtime_paths[seed] = target


# =============================================================================
# 4. PER-SEED CONTRACT AUDIT
# =============================================================================

print()
print("=" * 80)
print("PER-SEED CONFIG AUDIT")
print("=" * 80)

expected_seed_fields = (
    "seed_python",
    "seed_numpy",
    "seed_torch",
    "seed_cuda",
    "seed_dataloader",
)

for seed in REMAINING_SEEDS:

    cfg = runtime_configs[seed]

    print()
    print("-" * 80)
    print("SEED =", seed)
    print("CONFIG =", runtime_paths[seed])

    # Architecture
    assert (
        cfg["model"]["architecture"]
        == "model_b_graph_level_relational"
    )

    # Project seed
    assert int(
        cfg["project"]["seed"]
    ) == seed

    # All reproducibility seeds must match run seed.
    for field in expected_seed_fields:

        assert int(
            cfg["reproducibility"][field]
        ) == seed

    # Frozen split
    assert int(
        cfg["split"]["seed"]
    ) == 42

    assert (
        cfg["split"]["allow_create"]
        is False
    )

    assert (
        Path(
            cfg["split"]["persist_path"]
        ).resolve()
        == FROZEN_SPLIT.resolve()
    )

    # HDF5 runtime paths
    assert (
        Path(
            cfg["paths"]["mutants_hdf5"]
        ).resolve()
        == LOCAL_MUTANTS_HDF5.resolve()
    )

    assert (
        Path(
            cfg["paths"]["wt_companion_hdf5"]
        ).resolve()
        == LOCAL_WT_HDF5.resolve()
    )

    # Output root
    assert (
        Path(
            cfg["outputs"]["root_dir"]
        ).resolve()
        == MODEL_B_A9_ROOT.resolve()
    )

    assert (
        cfg["outputs"]["model_name"]
        == "model_b_graph_level_relational"
    )

    # Training contract
    assert int(
        cfg["training"]["epochs"]
    ) == 100

    assert int(
        cfg["training"]["batch_size"]
    ) == 4

    assert (
        cfg["training"]["optimizer"]
        == "adamw"
    )

    assert float(
        cfg["training"]["learning_rate"]
    ) == 0.001

    assert float(
        cfg["training"]["weight_decay"]
    ) == 0.0001

    assert (
        cfg["training"]["scheduler"]
        == "cosine"
    )

    assert (
        cfg["training"]["device"]
        == "cuda"
    )

    assert (
        cfg["training"]["mixed_precision"]["enabled"]
        is False
    )

    # Early stopping
    early = cfg["training"]["early_stopping"]

    assert early["enabled"] is True
    assert early["monitor"] == "validation_loss"
    assert early["mode"] == "min"
    assert int(early["patience"]) == 15
    assert float(early["min_delta"]) == 0.0

    # Loss contract
    assert cfg["loss"]["main"] == "nt_xent"
    assert float(cfg["loss"]["temperature"]) == 0.2

    mask = cfg["loss"]["false_negative_mask"]

    assert mask["enabled"] is True
    assert mask["mode"] == "same_position"
    assert mask["same_position"] is True
    assert mask["strict"] is True
    assert int(mask["min_valid_negatives"]) == 1

    # Relational Model B
    assert (
        cfg["model"]["projection_instance"]["enabled"]
        is False
    )

    assert (
        cfg["model"]["mlp_delta"]["enabled"]
        is True
    )

    assert (
        cfg["model"]["projection_pair"]["enabled"]
        is True
    )

    assert (
        cfg["model"]["projection_pair"]["input"]
        == "z_delta"
    )

    # A9 representations
    assert (
        cfg["a9"]["acceptance"][
            "required_pair_representations"
        ]
        == [
            "r_delta",
            "z_delta",
            "z_instance_pair",
        ]
    )

    # Productive seed list remains frozen.
    assert (
        list(
            cfg["a9"]["run_seeds"]
        )
        == [
            11,
            23,
            37,
            41,
            53,
        ]
    )

    # Fingerprints remain frozen.
    assert (
        cfg["a9"][
            "expected_hdf5_fingerprints"
        ]["mutants"]
        == EXPECTED_MUTANTS_SHA256
    )

    assert (
        cfg["a9"][
            "expected_hdf5_fingerprints"
        ]["wt_companion"]
        == EXPECTED_WT_SHA256
    )

    assert (
        cfg["a9"][
            "expected_split_fingerprint"
        ]
        == "8dcc611a6575b95242fda70d2b29e6bf4917a9c781bbeebd851e95e71391d7dd"
    )

    print(
        f"B{seed}_SCIENTIFIC_CONTRACT = PASS"
    )


# =============================================================================
# 5. CROSS-SEED AUDIT
#
# Remove ONLY run-seed fields.
# Everything else must then be exactly identical.
# =============================================================================

def normalize_seed_fields(config):

    normalized = deepcopy(config)

    normalized["project"]["seed"] = "<RUN_SEED>"

    for field in expected_seed_fields:

        normalized["reproducibility"][field] = (
            "<RUN_SEED>"
        )

    return normalized


reference_seed = REMAINING_SEEDS[0]

reference_normalized = (
    normalize_seed_fields(
        runtime_configs[
            reference_seed
        ]
    )
)


for seed in REMAINING_SEEDS[1:]:

    candidate_normalized = (
        normalize_seed_fields(
            runtime_configs[
                seed
            ]
        )
    )

    assert (
        candidate_normalized
        == reference_normalized
    ), (
        f"B{seed} differs from B{reference_seed} "
        "outside the permitted run-seed fields."
    )


print()
print(
    "CROSS_SEED_ONLY_RUN_SEED_DIFFERS = PASS"
)


# =============================================================================
# 6. VERIFY NO PRODUCTIVE RUNS WERE CREATED
# =============================================================================

existing_remaining_runs = []

for record in run_records:

    if record["seed"] in REMAINING_SEEDS:

        existing_remaining_runs.append(
            record
        )

print(
    "EXISTING_REMAINING_PRODUCTIVE_RUNS =",
    existing_remaining_runs,
)

assert existing_remaining_runs == []

print(
    "NO_REMAINING_RUNS_CREATED = PASS"
)


# =============================================================================
# 7. FINAL REPORT
# =============================================================================

print()
print("=" * 80)
print("FAST-B1 — FINAL")
print("=" * 80)

for seed in REMAINING_SEEDS:

    print(
        f"B{seed}_CONFIG =",
        runtime_paths[seed],
    )

print()
print(
    "FAST_B1_CONFIG_GENERATION = PASS"
)

print(
    "FAST_B1_PER_SEED_AUDIT = PASS"
)

print(
    "FAST_B1_CROSS_SEED_IDENTITY = PASS"
)

print(
    "REMAINING_SEEDS =",
    REMAINING_SEEDS,
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_NEW_RUN_CREATED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print(
    "READY_FOR_FAST_B2_PREFLIGHT = YES"
)

print("=" * 80)

MODEL B A9 — FAST-B1 PREPARE REMAINING PRODUCTIVE SEEDS
B11_CLOSED_ACCEPTED = PASS
REMAINING_SEEDS_ABSENT = PASS
INPUT_PATHS = PASS
SCIENTIFIC_CHECKOUT = PASS
B23_CONFIG_CREATED = PASS
B37_CONFIG_CREATED = PASS
B41_CONFIG_CREATED = PASS
B53_CONFIG_CREATED = PASS

PER-SEED CONFIG AUDIT

--------------------------------------------------------------------------------
SEED = 23
CONFIG = /content/model_b_workspace/runtime_configs/fast_remaining_b/model_b_a9_seed23_fresh.yaml
B23_SCIENTIFIC_CONTRACT = PASS

--------------------------------------------------------------------------------
SEED = 37
CONFIG = /content/model_b_workspace/runtime_configs/fast_remaining_b/model_b_a9_seed37_fresh.yaml
B37_SCIENTIFIC_CONTRACT = PASS

--------------------------------------------------------------------------------
SEED = 41
CONFIG = /content/model_b_workspace/runtime_configs/fast_remaining_b/model_b_a9_seed41_fresh.yaml
B41_SCIENTIFIC_CONTRACT = PASS

--------------------------------------------------

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B1.1 — FIX CANONICAL SPLIT PATH IN REMAINING CONFIGS
#
# CORRECTS ONLY:
# split.persist_path:
#   /content/.../splits/leave_position_out_seed_42.json
# ->
#   splits/leave_position_out_seed_42.json
#
# NO TRAINING
# NO PRODUCTIVE RUN CREATION
# =============================================================================

from pathlib import Path
from copy import deepcopy
import subprocess

from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
)

from gnn_siamese.config import (
    save_config,
    load_config,
)


print("=" * 80)
print("MODEL B A9 — FAST-B1.1 FIX CANONICAL SPLIT PATH")
print("=" * 80)

assert RUN_PRODUCTIVE_QUEUE is False

REMAINING_SEEDS = (
    23,
    37,
    41,
    53,
)

CANONICAL_SPLIT_PATH = (
    "splits/leave_position_out_seed_42.json"
)

WRONG_ABSOLUTE_SPLIT_PATH = str(
    (
        REPO
        / CANONICAL_SPLIT_PATH
    ).resolve()
)


# =============================================================================
# 1. SAFETY
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()

assert head == EXPECTED_SHA
assert status == ""

assert (
    seed_state_map[11]
    == "CLOSED_ACCEPTED"
)

for seed in REMAINING_SEEDS:

    assert (
        seed_state_map[seed]
        == "ABSENT"
    )

print("SCIENTIFIC_CHECKOUT = PASS")
print("B11_CLOSED_ACCEPTED = PASS")
print("REMAINING_SEEDS_ABSENT = PASS")


# =============================================================================
# 2. INSPECT THE FOUR EXISTING TEMPORARY CONFIGS
# =============================================================================

print()
print("=" * 80)
print("CURRENT CONFIGS")
print("=" * 80)

for seed in REMAINING_SEEDS:

    path = runtime_paths[seed]

    assert path.is_file()

    current = load_config(
        path
    )

    actual_split_path = (
        current["split"]["persist_path"]
    )

    print(
        f"B{seed}_CURRENT_SPLIT_PATH =",
        actual_split_path,
    )

    # We only accept the known incorrect value,
    # or an already-correct canonical value.
    assert actual_split_path in {
        WRONG_ABSOLUTE_SPLIT_PATH,
        CANONICAL_SPLIT_PATH,
    }, (
        f"Unexpected split.persist_path in B{seed}: "
        f"{actual_split_path}"
    )


# =============================================================================
# 3. REGENERATE FROM OFFICIAL A9 RESOLVER
#
# IMPORTANT:
# DO NOT convert split.persist_path to absolute.
# =============================================================================

corrected_configs = {}


for seed in REMAINING_SEEDS:

    config_a, config_b = (
        resolve_a9_runtime_configs(
            BASE_B_CONFIG,
            architecture=(
                "model_b_graph_level_relational"
            ),
            run_seed=seed,
            mutants_hdf5=(
                LOCAL_MUTANTS_HDF5
            ),
            wt_hdf5=(
                LOCAL_WT_HDF5
            ),
            output_root=(
                MODEL_B_A9_ROOT
            ),
            peer_output_root=(
                MODEL_A_A9_ROOT
            ),
            repo_root=REPO,
        )
    )

    # The resolver must preserve the canonical
    # A9 split path from the versioned config.
    assert (
        config_b["split"]["persist_path"]
        == CANONICAL_SPLIT_PATH
    )

    assert (
        config_b["split"]["allow_create"]
        is False
    )

    assert (
        int(
            config_b["split"]["seed"]
        )
        == 42
    )

    clean = {
        key: value
        for key, value
        in config_b.items()
        if not str(key).startswith("__")
    }

    corrected_configs[
        seed
    ] = clean


print()
print(
    "OFFICIAL_REGENERATION = PASS"
)


# =============================================================================
# 4. VERIFY ONLY THE KNOWN SPLIT-PATH CORRECTION
# =============================================================================

for seed in REMAINING_SEEDS:

    old = load_config(
        runtime_paths[seed]
    )

    new = deepcopy(
        corrected_configs[seed]
    )

    old_normalized = deepcopy(
        old
    )

    # Normalize only the field we intentionally correct.
    old_normalized[
        "split"
    ][
        "persist_path"
    ] = CANONICAL_SPLIT_PATH

    assert (
        old_normalized
        == new
    ), (
        f"B{seed}: regeneration differs in fields "
        "other than split.persist_path."
    )

    print(
        f"B{seed}_ONLY_SPLIT_PATH_CORRECTION = PASS"
    )


# =============================================================================
# 5. REPLACE ONLY THE EPHEMERAL YAML FILES
# =============================================================================

for seed in REMAINING_SEEDS:

    save_config(
        corrected_configs[seed],
        runtime_paths[seed],
    )

    reloaded = load_config(
        runtime_paths[seed]
    )

    assert (
        reloaded["split"]["persist_path"]
        == CANONICAL_SPLIT_PATH
    )

    assert (
        reloaded
        == corrected_configs[seed]
    )

    runtime_configs[
        seed
    ] = reloaded

    print(
        f"B{seed}_CONFIG_CORRECTED = PASS"
    )


# =============================================================================
# 6. CROSS-SEED IDENTITY AGAIN
# =============================================================================

expected_seed_fields = (
    "seed_python",
    "seed_numpy",
    "seed_torch",
    "seed_cuda",
    "seed_dataloader",
)


def normalize_run_seed(config):

    x = deepcopy(
        config
    )

    x["project"]["seed"] = (
        "<RUN_SEED>"
    )

    for field in expected_seed_fields:

        x["reproducibility"][
            field
        ] = "<RUN_SEED>"

    return x


reference = normalize_run_seed(
    runtime_configs[23]
)


for seed in (
    37,
    41,
    53,
):

    assert (
        normalize_run_seed(
            runtime_configs[seed]
        )
        == reference
    )


print()
print(
    "CROSS_SEED_ONLY_RUN_SEED_DIFFERS = PASS"
)


# =============================================================================
# 7. VERIFY CANONICAL CONTRACT
# =============================================================================

for seed in REMAINING_SEEDS:

    cfg = runtime_configs[
        seed
    ]

    assert (
        cfg["split"]["persist_path"]
        == CANONICAL_SPLIT_PATH
    )

    assert (
        int(
            cfg["split"]["seed"]
        )
        == 42
    )

    assert (
        cfg["split"]["allow_create"]
        is False
    )

    print(
        f"B{seed}_CANONICAL_SPLIT_CONTRACT = PASS"
    )


# =============================================================================
# 8. VERIFY NO NEW RUNS
# =============================================================================

current_runs = sorted(
    p.name
    for p
    in MODEL_B_ARCH_ROOT.glob(
        "run_*"
    )
    if p.is_dir()
)

print()
print(
    "PRODUCTIVE_RUN_DIRS =",
    current_runs,
)

assert current_runs == [
    "run_20260911T120628.981117Z-795be862",
    "run_20260914T143850.128514Z-e382b31e",
]

print(
    "NO_PRODUCTIVE_RUN_CREATED = PASS"
)


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 80)
print("FAST-B1.1 — FINAL")
print("=" * 80)

print(
    "CANONICAL_SPLIT_PATH =",
    CANONICAL_SPLIT_PATH,
)

print(
    "FAST_B1_SPLIT_PATH_FIX = PASS"
)

print(
    "ONLY_INTENDED_FIELD_CORRECTED = PASS"
)

print(
    "FAST_B1_CROSS_SEED_IDENTITY = PASS"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_NEW_RUN_CREATED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print(
    "READY_TO_RERUN_FAST_B2 = YES"
)

print("=" * 80)

MODEL B A9 — FAST-B1.1 FIX CANONICAL SPLIT PATH
SCIENTIFIC_CHECKOUT = PASS
B11_CLOSED_ACCEPTED = PASS
REMAINING_SEEDS_ABSENT = PASS

CURRENT CONFIGS
B23_CURRENT_SPLIT_PATH = /content/model_b_workspace/repo/splits/leave_position_out_seed_42.json
B37_CURRENT_SPLIT_PATH = /content/model_b_workspace/repo/splits/leave_position_out_seed_42.json
B41_CURRENT_SPLIT_PATH = /content/model_b_workspace/repo/splits/leave_position_out_seed_42.json
B53_CURRENT_SPLIT_PATH = /content/model_b_workspace/repo/splits/leave_position_out_seed_42.json

OFFICIAL_REGENERATION = PASS
B23_ONLY_SPLIT_PATH_CORRECTION = PASS
B37_ONLY_SPLIT_PATH_CORRECTION = PASS
B41_ONLY_SPLIT_PATH_CORRECTION = PASS
B53_ONLY_SPLIT_PATH_CORRECTION = PASS
B23_CONFIG_CORRECTED = PASS
B37_CONFIG_CORRECTED = PASS
B41_CONFIG_CORRECTED = PASS
B53_CONFIG_CORRECTED = PASS

CROSS_SEED_ONLY_RUN_SEED_DIFFERS = PASS
B23_CANONICAL_SPLIT_CONTRACT = PASS
B37_CANONICAL_SPLIT_CONTRACT = PASS
B41_CANONICAL_SPLIT_CONTRACT = PASS
B53_CANONICAL_SPLIT_CONT

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B2 — OFFICIAL JOINT PREFLIGHT FOR B23 / B37 / B41 / B53
#
# NO TRAINING
# NO PRODUCTIVE RUN CREATION
# =============================================================================

from pathlib import Path
import json
import subprocess

from scripts.a9_preflight import (
    validate_a9_colab_preflight,
)


print("=" * 80)
print("MODEL B A9 — FAST-B2 JOINT OFFICIAL PREFLIGHT")
print("=" * 80)


# =============================================================================
# 0. SAFETY
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False

REMAINING_SEEDS = (
    23,
    37,
    41,
    53,
)

assert tuple(
    runtime_configs.keys()
) == REMAINING_SEEDS

assert (
    seed_state_map[11]
    == "CLOSED_ACCEPTED"
)

for seed in REMAINING_SEEDS:

    assert (
        seed_state_map[seed]
        == "ABSENT"
    )


print("B11_CLOSED_ACCEPTED = PASS")
print("REMAINING_SEEDS_ABSENT = PASS")


# =============================================================================
# 1. PATHS
# =============================================================================

MODEL_A_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_a"
    / "runs"
    / "model_a_a9"
)

MODEL_B_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
)

ALLOWED_OUTPUT_ROOT = Path(
    DRIVE_PROJECT_BASE
)

assert LOCAL_MUTANTS_HDF5.is_file()
assert LOCAL_WT_HDF5.is_file()

assert MODEL_B_A9_ROOT.is_dir()

print("INPUT_PATHS = PASS")


# =============================================================================
# 2. SCIENTIFIC CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()

assert head == EXPECTED_SHA
assert status == ""

print("SCIENTIFIC_CHECKOUT = PASS")


# =============================================================================
# 3. GPU
# =============================================================================

import torch

assert torch.cuda.is_available(), (
    "FAST-B2 requires CUDA."
)

print(
    "GPU =",
    torch.cuda.get_device_name(0),
)

print(
    "CUDA_AVAILABLE =",
    torch.cuda.is_available(),
)

print("GPU_PREFLIGHT = PASS")


# =============================================================================
# 4. SNAPSHOT PRODUCTIVE RUN DIRECTORIES BEFORE PREFLIGHT
# =============================================================================

runs_before = sorted(
    p.name
    for p in MODEL_B_ARCH_ROOT.glob(
        "run_*"
    )
    if p.is_dir()
)

print()
print(
    "PRODUCTIVE_RUN_DIRS_BEFORE =",
    runs_before,
)


# =============================================================================
# 5. RUN OFFICIAL PREFLIGHT FOR ALL 4 SEEDS
# =============================================================================

preflight_results = {}
preflight_failures = {}


for seed in REMAINING_SEEDS:

    print()
    print("=" * 80)
    print(
        f"OFFICIAL PREFLIGHT — B{seed}"
    )
    print("=" * 80)

    config_path = (
        runtime_paths[
            seed
        ]
    )

    assert config_path.is_file()

    try:

        result = (
            validate_a9_colab_preflight(
                config_path,
                architecture=(
                    "model_b_graph_level_relational"
                ),
                run_seed=seed,
                mutants_hdf5=(
                    LOCAL_MUTANTS_HDF5
                ),
                wt_hdf5=(
                    LOCAL_WT_HDF5
                ),
                output_root=(
                    MODEL_B_A9_ROOT
                ),
                peer_output_root=(
                    MODEL_A_A9_ROOT
                ),
                repo_root=REPO,
                expected_commit=(
                    EXPECTED_SHA
                ),
                allowed_output_root=(
                    ALLOWED_OUTPUT_ROOT
                ),
            )
        )

        preflight_results[
            seed
        ] = result

        print(
            json.dumps(
                result,
                indent=2,
                sort_keys=True,
                default=str,
            )
        )

        print()
        print(
            f"B{seed}_OFFICIAL_PREFLIGHT = PASS"
        )

    except Exception as exc:

        preflight_failures[
            seed
        ] = repr(
            exc
        )

        print(
            f"B{seed}_OFFICIAL_PREFLIGHT = FAIL"
        )

        print(
            "ERROR =",
            repr(exc),
        )


# =============================================================================
# 6. FAIL CLOSED IF ANY PREFLIGHT FAILED
# =============================================================================

print()
print("=" * 80)
print("PREFLIGHT SUMMARY")
print("=" * 80)

print(
    "PREFLIGHT_FAILURES =",
    preflight_failures,
)

assert (
    preflight_failures
    == {}
), (
    "At least one remaining seed failed official A9 preflight."
)

assert (
    set(
        preflight_results
    )
    == set(
        REMAINING_SEEDS
    )
)


# =============================================================================
# 7. COMMON SCIENTIFIC CHECKS
# =============================================================================

prospective_dirs = []


for seed in REMAINING_SEEDS:

    result = (
        preflight_results[
            seed
        ]
    )

    assert (
        result["status"]
        == "PASS"
    )

    assert (
        int(
            result["run_seed"]
        )
        == seed
    )

    assert (
        int(
            result["split_seed"]
        )
        == 42
    )

    assert (
        int(
            result[
                "ab_intersection"
            ]
        )
        == 483
    )


    # -------------------------------------------------------------------------
    # Model A inventory
    # -------------------------------------------------------------------------

    inv_a = (
        result[
            "inventories"
        ]["model_a"]
    )

    assert (
        int(
            inv_a[
                "biological_variants"
            ]
        )
        == 483
    )

    assert (
        int(
            inv_a[
                "native_wt_controls"
            ]
        )
        == 1
    )

    assert (
        inv_a[
            "partitions"
        ]
        == {
            "train": 342,
            "validation": 78,
            "test": 63,
        }
    )

    assert (
        inv_a[
            "split_fingerprint"
        ]
        == (
            "8dcc611a6575b95242fda70d2b29e6b"
            "f4917a9c781bbeebd851e95e71391d7dd"
        )
    )


    # -------------------------------------------------------------------------
    # Model B inventory
    # -------------------------------------------------------------------------

    inv_b = (
        result[
            "inventories"
        ]["model_b"]
    )

    assert (
        int(
            inv_b[
                "biological_variants"
            ]
        )
        == 483
    )

    assert (
        int(
            inv_b[
                "native_wt_controls"
            ]
        )
        == 1
    )

    assert (
        inv_b[
            "partitions"
        ]
        == {
            "train": 342,
            "validation": 78,
            "test": 63,
        }
    )

    assert (
        inv_b[
            "split_fingerprint"
        ]
        == (
            "8dcc611a6575b95242fda70d2b29e6b"
            "f4917a9c781bbeebd851e95e71391d7dd"
        )
    )


    # -------------------------------------------------------------------------
    # Future fresh run
    # -------------------------------------------------------------------------

    prospective = Path(
        result[
            "prospective_run_dir"
        ]
    )

    assert (
        result[
            "prospective_exists"
        ]
        is False
    )

    assert not prospective.exists()

    prospective_dirs.append(
        str(
            prospective
        )
    )


    print()
    print(
        f"B{seed}_INVENTORY_483_PLUS_1 = PASS"
    )

    print(
        f"B{seed}_SPLIT_342_78_63 = PASS"
    )

    print(
        f"B{seed}_SPLIT_FINGERPRINT = PASS"
    )

    print(
        f"B{seed}_AB_INTERSECTION_483 = PASS"
    )

    print(
        f"B{seed}_FRESH_TARGET_ABSENT = PASS"
    )


# =============================================================================
# 8. PROSPECTIVE RUN DIRECTORIES MUST BE DISTINCT
# =============================================================================

assert (
    len(
        set(
            prospective_dirs
        )
    )
    == len(
        REMAINING_SEEDS
    )
)

print()
print(
    "PROSPECTIVE_RUN_DIRS_DISTINCT = PASS"
)


# =============================================================================
# 9. VERIFY PREFLIGHT DID NOT CREATE PRODUCTIVE RUNS
# =============================================================================

runs_after = sorted(
    p.name
    for p in MODEL_B_ARCH_ROOT.glob(
        "run_*"
    )
    if p.is_dir()
)

print(
    "PRODUCTIVE_RUN_DIRS_AFTER =",
    runs_after,
)

assert (
    runs_after
    == runs_before
), (
    "Official preflight unexpectedly created a productive run directory."
)

print(
    "NO_PRODUCTIVE_RUN_CREATED = PASS"
)


# =============================================================================
# 10. VERIFY NO LEFTOVER PREFLIGHT TEMP FILES
# =============================================================================

temp_files = []

for root in (
    MODEL_A_A9_ROOT,
    MODEL_B_A9_ROOT,
):

    if root.exists():

        temp_files.extend(
            str(p)
            for p in root.glob(
                ".colab-preflight-*"
            )
        )


print(
    "LEFTOVER_PREFLIGHT_TEMP_FILES =",
    temp_files,
)

assert temp_files == []

print(
    "PREFLIGHT_TEMP_CLEANUP = PASS"
)


# =============================================================================
# 11. FAST TRACK AUTHORIZATION
# =============================================================================

FAST_B2_ALL_PASS = True

FAST_B3_AUTHORIZED_SEEDS = (
    23,
    37,
    41,
    53,
)

assert (
    FAST_B3_AUTHORIZED_SEEDS
    == REMAINING_SEEDS
)


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 80)
print("FAST-B2 — FINAL")
print("=" * 80)

for seed in REMAINING_SEEDS:

    print(
        f"B{seed}_OFFICIAL_PREFLIGHT = PASS"
    )

print()
print(
    "FAST_B2_ALL_PREFLIGHTS = PASS"
)

print(
    "FAST_B2_SCIENTIFIC_IDENTITY = PASS"
)

print(
    "FAST_B2_FRESH_TARGETS = PASS"
)

print(
    "FAST_B3_AUTHORIZED_SEEDS =",
    FAST_B3_AUTHORIZED_SEEDS,
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_NEW_RUN_CREATED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print(
    "READY_FOR_PARALLEL_TRAINING = YES"
)

print("=" * 80)

MODEL B A9 — FAST-B2 JOINT OFFICIAL PREFLIGHT
B11_CLOSED_ACCEPTED = PASS
REMAINING_SEEDS_ABSENT = PASS
INPUT_PATHS = PASS
SCIENTIFIC_CHECKOUT = PASS
GPU = Tesla T4
CUDA_AVAILABLE = True
GPU_PREFLIGHT = PASS

PRODUCTIVE_RUN_DIRS_BEFORE = ['run_20260911T120628.981117Z-795be862', 'run_20260914T143850.128514Z-e382b31e']

OFFICIAL PREFLIGHT — B23
{
  "ab_intersection": 483,
  "git": {
    "commit": "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111",
    "expected_commit": "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111",
    "working_tree": "clean"
  },
  "inventories": {
    "model_a": {
      "biological_variants": 483,
      "dataset_identity": {
        "actual_combined_fingerprint": "9a320862a54566d232e1ef2a4eafa468cf900a19186dc0fd7c92aaf0abe06c68",
        "actual_mutant_sha256": "92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631",
        "actual_wt_sha256": "29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a",
        "dataset_identity_status": "PASS",
        "expect

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B3 — INDEPENDENT PRODUCTIVE WORKER
#
# CHANGE ONLY:
#     WORKER_SEED = 23 / 37 / 41 / 53
#
# FRESH TRAINING
# NO RESUME
# ONE SEED PER RUNTIME
# =============================================================================

from pathlib import Path
import json
import subprocess
import sys
import torch

from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
)

from gnn_siamese.config import (
    save_config,
    load_config,
)


# =============================================================================
# EDIT ONLY THIS VALUE
# =============================================================================

WORKER_SEED = 23


# =============================================================================
# FROZEN WORKER CONTRACT
# =============================================================================

ALLOWED_WORKER_SEEDS = {
    23,
    37,
    41,
    53,
}

RUN_PRODUCTIVE_QUEUE = False

CANONICAL_SPLIT_PATH = (
    "splits/leave_position_out_seed_42.json"
)

EXPECTED_SPLIT_FINGERPRINT = (
    "8dcc611a6575b95242fda70d2b29e6b"
    "f4917a9c781bbeebd851e95e71391d7dd"
)


print("=" * 80)
print(
    f"MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B{WORKER_SEED}"
)
print("=" * 80)


# =============================================================================
# 1. FAIL-CLOSED SAFETY
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False

assert WORKER_SEED in ALLOWED_WORKER_SEEDS, (
    f"Invalid worker seed: {WORKER_SEED}"
)

assert torch.cuda.is_available(), (
    "CUDA GPU required."
)


print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "GPU =",
    torch.cuda.get_device_name(0),
)

print(
    "RUN_PRODUCTIVE_QUEUE =",
    RUN_PRODUCTIVE_QUEUE,
)


# =============================================================================
# 2. REQUIRE RECOVERY CELLS 0.1–0.4
# =============================================================================

required_names = [
    "REPO",
    "SRC",
    "EXPECTED_SHA",
    "LOCAL_ROOT",
    "LOCAL_MUTANTS_HDF5",
    "LOCAL_WT_HDF5",
    "DRIVE_PROJECT_BASE",
    "B11_REFERENCE_RUN",
    "B11_ACCEPTANCE",
    "EXPECTED_MUTANTS_SHA256",
    "EXPECTED_WT_SHA256",
    "sha256_file",
]

for name in required_names:

    assert name in globals(), (
        f"Missing recovery variable {name}. "
        "Run cells 0.1–0.4 first."
    )


assert REPO.is_dir()
assert LOCAL_MUTANTS_HDF5.is_file()
assert LOCAL_WT_HDF5.is_file()
assert B11_ACCEPTANCE.is_file()

print(
    "RECOVERY_PREREQUISITES = PASS"
)


# =============================================================================
# 3. EXACT GIT CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()


print(
    "HEAD =",
    head,
)

print(
    "WORKING_TREE =",
    repr(status),
)


assert head == EXPECTED_SHA
assert status == ""

print(
    "SCIENTIFIC_CHECKOUT = PASS"
)


# =============================================================================
# 4. EXACT DATA IDENTITY
# =============================================================================

mut_sha = sha256_file(
    LOCAL_MUTANTS_HDF5
)

wt_sha = sha256_file(
    LOCAL_WT_HDF5
)


assert (
    mut_sha
    == EXPECTED_MUTANTS_SHA256
)

assert (
    wt_sha
    == EXPECTED_WT_SHA256
)


print(
    "MUTANTS_SHA256 =",
    mut_sha,
)

print(
    "WT_SHA256 =",
    wt_sha,
)

print(
    "HDF5_IDENTITY = PASS"
)


# =============================================================================
# 5. B11 REFERENCE MUST REMAIN ACCEPTED
# =============================================================================

b11_acceptance = json.loads(
    B11_ACCEPTANCE.read_text(
        encoding="utf-8"
    )
)


assert (
    b11_acceptance["status"]
    == "accepted"
)

assert (
    int(
        b11_acceptance["run_seed"]
    )
    == 11
)


print(
    "B11_REFERENCE_ACCEPTED = PASS"
)


# =============================================================================
# 6. PRODUCTIVE ROOTS
# =============================================================================

BASE_B_CONFIG = (
    REPO
    / "configs"
    / "model_b_a9.yaml"
)

MODEL_A_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_a"
    / "runs"
    / "model_a_a9"
)

MODEL_B_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
)

MODEL_B_ARCH_ROOT = (
    MODEL_B_A9_ROOT
    / "model_b_graph_level_relational"
)

WORKER_CONFIG_ROOT = (
    LOCAL_ROOT
    / "runtime_configs"
    / "fast_workers"
)

WORKER_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

WORKER_CONFIG = (
    WORKER_CONFIG_ROOT
    / f"model_b_a9_seed{WORKER_SEED}_fresh.yaml"
)


assert BASE_B_CONFIG.is_file()
assert MODEL_B_ARCH_ROOT.is_dir()


# =============================================================================
# 7. SCAN FOR AN EXISTING RUN OF THIS SEED
#
# Important for concurrent workers:
# inspect manifests by SEED, not by directory count.
# =============================================================================

def find_seed_runs(seed):

    results = []

    for run_dir in sorted(
        MODEL_B_ARCH_ROOT.glob(
            "run_*"
        )
    ):

        manifest_path = (
            run_dir
            / "run_manifest.json"
        )

        if not manifest_path.is_file():
            continue

        try:

            payload = json.loads(
                manifest_path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:
            continue


        manifest_seed = (
            payload
            .get(
                "configuration",
                {}
            )
            .get(
                "seed"
            )
        )


        if manifest_seed == seed:

            results.append(
                {
                    "run_dir":
                        run_dir,

                    "status":
                        payload.get(
                            "status"
                        ),

                    "manifest":
                        payload,
                }
            )

    return results


seed_runs_before = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_BEFORE =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_before
    ],
)


assert seed_runs_before == [], (
    f"STOP: seed {WORKER_SEED} already has a run. "
    "Do not launch fresh automatically."
)


print(
    f"B{WORKER_SEED}_ABSENT = PASS"
)


# =============================================================================
# 8. GENERATE OFFICIAL RUNTIME CONFIG
# =============================================================================

config_a, config_b = (
    resolve_a9_runtime_configs(
        BASE_B_CONFIG,
        architecture=(
            "model_b_graph_level_relational"
        ),
        run_seed=WORKER_SEED,
        mutants_hdf5=(
            LOCAL_MUTANTS_HDF5
        ),
        wt_hdf5=(
            LOCAL_WT_HDF5
        ),
        output_root=(
            MODEL_B_A9_ROOT
        ),
        peer_output_root=(
            MODEL_A_A9_ROOT
        ),
        repo_root=REPO,
    )
)


# DO NOT convert this to an absolute path.
assert (
    config_b["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    int(
        config_b["split"]["seed"]
    )
    == 42
)

assert (
    config_b["split"]["allow_create"]
    is False
)


clean_config = {
    key: value

    for key, value
    in config_b.items()

    if not str(
        key
    ).startswith("__")
}


save_config(
    clean_config,
    WORKER_CONFIG,
)


runtime = load_config(
    WORKER_CONFIG
)


print(
    "WORKER_CONFIG =",
    WORKER_CONFIG,
)

print(
    "RUNTIME_CONFIG_GENERATION = PASS"
)


# =============================================================================
# 9. EXACT WORKER CONFIG AUDIT
# =============================================================================

assert (
    runtime["model"]["architecture"]
    == "model_b_graph_level_relational"
)

assert (
    int(
        runtime["project"]["seed"]
    )
    == WORKER_SEED
)


for field in [
    "seed_python",
    "seed_numpy",
    "seed_torch",
    "seed_cuda",
    "seed_dataloader",
]:

    assert (
        int(
            runtime[
                "reproducibility"
            ][field]
        )
        == WORKER_SEED
    )


assert (
    int(
        runtime["split"]["seed"]
    )
    == 42
)

assert (
    runtime["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    runtime["split"]["allow_create"]
    is False
)


assert (
    runtime["a9"][
        "expected_split_fingerprint"
    ]
    == EXPECTED_SPLIT_FINGERPRINT
)


assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["mutants"]
    == EXPECTED_MUTANTS_SHA256
)

assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["wt_companion"]
    == EXPECTED_WT_SHA256
)


assert int(
    runtime["training"]["epochs"]
) == 100

assert int(
    runtime["training"]["batch_size"]
) == 4

assert (
    runtime["training"]["device"]
    == "cuda"
)

assert (
    runtime["loss"]["main"]
    == "nt_xent"
)

assert (
    float(
        runtime["loss"]["temperature"]
    )
    == 0.2
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["mode"]
    == "same_position"
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["strict"]
    is True
)


print(
    "WORKER_CONFIG_AUDIT = PASS"
)


# =============================================================================
# 10. FINAL PRE-LAUNCH FRESH CHECK
#
# Check again immediately before train.py.
# This prevents accidentally double-launching the same seed.
# =============================================================================

seed_runs_immediately_before = (
    find_seed_runs(
        WORKER_SEED
    )
)


assert (
    seed_runs_immediately_before
    == []
), (
    f"STOP: seed {WORKER_SEED} appeared before launch."
)


print(
    "FRESH_LAUNCH_GUARD = PASS"
)


# =============================================================================
# 11. TRAIN
#
# No --resume-from.
# This is explicitly FRESH.
# =============================================================================

command = [
    sys.executable,
    str(
        REPO
        / "scripts"
        / "train.py"
    ),
    "--config",
    str(
        WORKER_CONFIG
    ),
    "--device",
    "cuda",
]


print()
print("=" * 80)
print(
    f"LAUNCHING PRODUCTIVE B{WORKER_SEED} FRESH"
)
print("=" * 80)

print(
    "COMMAND:"
)

print(
    " ".join(
        command
    )
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)


result = subprocess.run(
    command,
    cwd=REPO,
)


TRAIN_RETURN_CODE = (
    result.returncode
)


print()
print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)


# =============================================================================
# 12. DISCOVER EXACT RUN OF THIS WORKER SEED
# =============================================================================

seed_runs_after = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_AFTER =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_after
    ],
)


new_seed_runs = [
    x
    for x in seed_runs_after
    if x["run_dir"]
    not in {
        old["run_dir"]
        for old
        in seed_runs_before
    }
]


assert len(
    new_seed_runs
) == 1, (
    f"Expected exactly one new B{WORKER_SEED} run, "
    f"found {len(new_seed_runs)}."
)


WORKER_RUN = (
    new_seed_runs[0][
        "run_dir"
    ]
)

WORKER_STATUS = (
    new_seed_runs[0][
        "status"
    ]
)


print()
print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "WORKER_STATUS =",
    WORKER_STATUS,
)


# =============================================================================
# 13. TRAIN RETURN / MANIFEST CONSISTENCY
# =============================================================================

if TRAIN_RETURN_CODE == 0:

    assert (
        WORKER_STATUS
        == "completed"
    ), (
        "train.py returned 0 but manifest is not completed."
    )

    TRAINING_RESULT = (
        "COMPLETED"
    )

else:

    TRAINING_RESULT = (
        "FAILED_OR_INTERRUPTED"
    )


# =============================================================================
# 14. FINAL
# =============================================================================

print()
print("=" * 80)
print(
    f"FAST-B3 WORKER B{WORKER_SEED} — FINAL"
)
print("=" * 80)

print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)

print(
    "TRAINING_RESULT =",
    TRAINING_RESULT,
)

print(
    "MANIFEST_STATUS =",
    WORKER_STATUS,
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

if (
    TRAIN_RETURN_CODE == 0
    and
    WORKER_STATUS == "completed"
):

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = YES"
    )

else:

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = NOT_COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = NO"
    )

    print(
        "DO_NOT_RELAUNCH_FRESH_AUTOMATICALLY = TRUE"
    )

print("=" * 80)

MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B23
WORKER_SEED = 23
GPU = Tesla T4
RUN_PRODUCTIVE_QUEUE = False
RECOVERY_PREREQUISITES = PASS
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
WORKING_TREE = ''
SCIENTIFIC_CHECKOUT = PASS
MUTANTS_SHA256 = 92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631
WT_SHA256 = 29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a
HDF5_IDENTITY = PASS
B11_REFERENCE_ACCEPTED = PASS

WORKER_SEED_RUNS_BEFORE = []
B23_ABSENT = PASS
WORKER_CONFIG = /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed23_fresh.yaml
RUNTIME_CONFIG_GENERATION = PASS
WORKER_CONFIG_AUDIT = PASS
FRESH_LAUNCH_GUARD = PASS

LAUNCHING PRODUCTIVE B23 FRESH
COMMAND:
/usr/bin/python3 /content/model_b_workspace/repo/scripts/train.py --config /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed23_fresh.yaml --device cuda
EXECUTION = FRESH
RESUME_FROM = NONE
RUN_PRODUCTIVE_QUEUE = False

TRAIN_RETURN_CODE = 0

WORKER_SEED_RUNS_AFTER 

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B3 — INDEPENDENT PRODUCTIVE WORKER
#
# CHANGE ONLY:
#     WORKER_SEED = 23 / 37 / 41 / 53
#
# FRESH TRAINING
# NO RESUME
# ONE SEED PER RUNTIME
# =============================================================================

from pathlib import Path
import json
import subprocess
import sys
import torch

from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
)

from gnn_siamese.config import (
    save_config,
    load_config,
)


# =============================================================================
# EDIT ONLY THIS VALUE
# =============================================================================

WORKER_SEED = 41


# =============================================================================
# FROZEN WORKER CONTRACT
# =============================================================================

ALLOWED_WORKER_SEEDS = {
    23,
    37,
    41,
    53,
}

RUN_PRODUCTIVE_QUEUE = False

CANONICAL_SPLIT_PATH = (
    "splits/leave_position_out_seed_42.json"
)

EXPECTED_SPLIT_FINGERPRINT = (
    "8dcc611a6575b95242fda70d2b29e6b"
    "f4917a9c781bbeebd851e95e71391d7dd"
)


print("=" * 80)
print(
    f"MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B{WORKER_SEED}"
)
print("=" * 80)


# =============================================================================
# 1. FAIL-CLOSED SAFETY
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False

assert WORKER_SEED in ALLOWED_WORKER_SEEDS, (
    f"Invalid worker seed: {WORKER_SEED}"
)

assert torch.cuda.is_available(), (
    "CUDA GPU required."
)


print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "GPU =",
    torch.cuda.get_device_name(0),
)

print(
    "RUN_PRODUCTIVE_QUEUE =",
    RUN_PRODUCTIVE_QUEUE,
)


# =============================================================================
# 2. REQUIRE RECOVERY CELLS 0.1–0.4
# =============================================================================

required_names = [
    "REPO",
    "SRC",
    "EXPECTED_SHA",
    "LOCAL_ROOT",
    "LOCAL_MUTANTS_HDF5",
    "LOCAL_WT_HDF5",
    "DRIVE_PROJECT_BASE",
    "B11_REFERENCE_RUN",
    "B11_ACCEPTANCE",
    "EXPECTED_MUTANTS_SHA256",
    "EXPECTED_WT_SHA256",
    "sha256_file",
]

for name in required_names:

    assert name in globals(), (
        f"Missing recovery variable {name}. "
        "Run cells 0.1–0.4 first."
    )


assert REPO.is_dir()
assert LOCAL_MUTANTS_HDF5.is_file()
assert LOCAL_WT_HDF5.is_file()
assert B11_ACCEPTANCE.is_file()

print(
    "RECOVERY_PREREQUISITES = PASS"
)


# =============================================================================
# 3. EXACT GIT CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()


print(
    "HEAD =",
    head,
)

print(
    "WORKING_TREE =",
    repr(status),
)


assert head == EXPECTED_SHA
assert status == ""

print(
    "SCIENTIFIC_CHECKOUT = PASS"
)


# =============================================================================
# 4. EXACT DATA IDENTITY
# =============================================================================

mut_sha = sha256_file(
    LOCAL_MUTANTS_HDF5
)

wt_sha = sha256_file(
    LOCAL_WT_HDF5
)


assert (
    mut_sha
    == EXPECTED_MUTANTS_SHA256
)

assert (
    wt_sha
    == EXPECTED_WT_SHA256
)


print(
    "MUTANTS_SHA256 =",
    mut_sha,
)

print(
    "WT_SHA256 =",
    wt_sha,
)

print(
    "HDF5_IDENTITY = PASS"
)


# =============================================================================
# 5. B11 REFERENCE MUST REMAIN ACCEPTED
# =============================================================================

b11_acceptance = json.loads(
    B11_ACCEPTANCE.read_text(
        encoding="utf-8"
    )
)


assert (
    b11_acceptance["status"]
    == "accepted"
)

assert (
    int(
        b11_acceptance["run_seed"]
    )
    == 11
)


print(
    "B11_REFERENCE_ACCEPTED = PASS"
)


# =============================================================================
# 6. PRODUCTIVE ROOTS
# =============================================================================

BASE_B_CONFIG = (
    REPO
    / "configs"
    / "model_b_a9.yaml"
)

MODEL_A_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_a"
    / "runs"
    / "model_a_a9"
)

MODEL_B_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
)

MODEL_B_ARCH_ROOT = (
    MODEL_B_A9_ROOT
    / "model_b_graph_level_relational"
)

WORKER_CONFIG_ROOT = (
    LOCAL_ROOT
    / "runtime_configs"
    / "fast_workers"
)

WORKER_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

WORKER_CONFIG = (
    WORKER_CONFIG_ROOT
    / f"model_b_a9_seed{WORKER_SEED}_fresh.yaml"
)


assert BASE_B_CONFIG.is_file()
assert MODEL_B_ARCH_ROOT.is_dir()


# =============================================================================
# 7. SCAN FOR AN EXISTING RUN OF THIS SEED
#
# Important for concurrent workers:
# inspect manifests by SEED, not by directory count.
# =============================================================================

def find_seed_runs(seed):

    results = []

    for run_dir in sorted(
        MODEL_B_ARCH_ROOT.glob(
            "run_*"
        )
    ):

        manifest_path = (
            run_dir
            / "run_manifest.json"
        )

        if not manifest_path.is_file():
            continue

        try:

            payload = json.loads(
                manifest_path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:
            continue


        manifest_seed = (
            payload
            .get(
                "configuration",
                {}
            )
            .get(
                "seed"
            )
        )


        if manifest_seed == seed:

            results.append(
                {
                    "run_dir":
                        run_dir,

                    "status":
                        payload.get(
                            "status"
                        ),

                    "manifest":
                        payload,
                }
            )

    return results


seed_runs_before = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_BEFORE =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_before
    ],
)


assert seed_runs_before == [], (
    f"STOP: seed {WORKER_SEED} already has a run. "
    "Do not launch fresh automatically."
)


print(
    f"B{WORKER_SEED}_ABSENT = PASS"
)


# =============================================================================
# 8. GENERATE OFFICIAL RUNTIME CONFIG
# =============================================================================

config_a, config_b = (
    resolve_a9_runtime_configs(
        BASE_B_CONFIG,
        architecture=(
            "model_b_graph_level_relational"
        ),
        run_seed=WORKER_SEED,
        mutants_hdf5=(
            LOCAL_MUTANTS_HDF5
        ),
        wt_hdf5=(
            LOCAL_WT_HDF5
        ),
        output_root=(
            MODEL_B_A9_ROOT
        ),
        peer_output_root=(
            MODEL_A_A9_ROOT
        ),
        repo_root=REPO,
    )
)


# DO NOT convert this to an absolute path.
assert (
    config_b["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    int(
        config_b["split"]["seed"]
    )
    == 42
)

assert (
    config_b["split"]["allow_create"]
    is False
)


clean_config = {
    key: value

    for key, value
    in config_b.items()

    if not str(
        key
    ).startswith("__")
}


save_config(
    clean_config,
    WORKER_CONFIG,
)


runtime = load_config(
    WORKER_CONFIG
)


print(
    "WORKER_CONFIG =",
    WORKER_CONFIG,
)

print(
    "RUNTIME_CONFIG_GENERATION = PASS"
)


# =============================================================================
# 9. EXACT WORKER CONFIG AUDIT
# =============================================================================

assert (
    runtime["model"]["architecture"]
    == "model_b_graph_level_relational"
)

assert (
    int(
        runtime["project"]["seed"]
    )
    == WORKER_SEED
)


for field in [
    "seed_python",
    "seed_numpy",
    "seed_torch",
    "seed_cuda",
    "seed_dataloader",
]:

    assert (
        int(
            runtime[
                "reproducibility"
            ][field]
        )
        == WORKER_SEED
    )


assert (
    int(
        runtime["split"]["seed"]
    )
    == 42
)

assert (
    runtime["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    runtime["split"]["allow_create"]
    is False
)


assert (
    runtime["a9"][
        "expected_split_fingerprint"
    ]
    == EXPECTED_SPLIT_FINGERPRINT
)


assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["mutants"]
    == EXPECTED_MUTANTS_SHA256
)

assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["wt_companion"]
    == EXPECTED_WT_SHA256
)


assert int(
    runtime["training"]["epochs"]
) == 100

assert int(
    runtime["training"]["batch_size"]
) == 4

assert (
    runtime["training"]["device"]
    == "cuda"
)

assert (
    runtime["loss"]["main"]
    == "nt_xent"
)

assert (
    float(
        runtime["loss"]["temperature"]
    )
    == 0.2
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["mode"]
    == "same_position"
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["strict"]
    is True
)


print(
    "WORKER_CONFIG_AUDIT = PASS"
)


# =============================================================================
# 10. FINAL PRE-LAUNCH FRESH CHECK
#
# Check again immediately before train.py.
# This prevents accidentally double-launching the same seed.
# =============================================================================

seed_runs_immediately_before = (
    find_seed_runs(
        WORKER_SEED
    )
)


assert (
    seed_runs_immediately_before
    == []
), (
    f"STOP: seed {WORKER_SEED} appeared before launch."
)


print(
    "FRESH_LAUNCH_GUARD = PASS"
)


# =============================================================================
# 11. TRAIN
#
# No --resume-from.
# This is explicitly FRESH.
# =============================================================================

command = [
    sys.executable,
    str(
        REPO
        / "scripts"
        / "train.py"
    ),
    "--config",
    str(
        WORKER_CONFIG
    ),
    "--device",
    "cuda",
]


print()
print("=" * 80)
print(
    f"LAUNCHING PRODUCTIVE B{WORKER_SEED} FRESH"
)
print("=" * 80)

print(
    "COMMAND:"
)

print(
    " ".join(
        command
    )
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)


result = subprocess.run(
    command,
    cwd=REPO,
)


TRAIN_RETURN_CODE = (
    result.returncode
)


print()
print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)


# =============================================================================
# 12. DISCOVER EXACT RUN OF THIS WORKER SEED
# =============================================================================

seed_runs_after = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_AFTER =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_after
    ],
)


new_seed_runs = [
    x
    for x in seed_runs_after
    if x["run_dir"]
    not in {
        old["run_dir"]
        for old
        in seed_runs_before
    }
]


assert len(
    new_seed_runs
) == 1, (
    f"Expected exactly one new B{WORKER_SEED} run, "
    f"found {len(new_seed_runs)}."
)


WORKER_RUN = (
    new_seed_runs[0][
        "run_dir"
    ]
)

WORKER_STATUS = (
    new_seed_runs[0][
        "status"
    ]
)


print()
print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "WORKER_STATUS =",
    WORKER_STATUS,
)


# =============================================================================
# 13. TRAIN RETURN / MANIFEST CONSISTENCY
# =============================================================================

if TRAIN_RETURN_CODE == 0:

    assert (
        WORKER_STATUS
        == "completed"
    ), (
        "train.py returned 0 but manifest is not completed."
    )

    TRAINING_RESULT = (
        "COMPLETED"
    )

else:

    TRAINING_RESULT = (
        "FAILED_OR_INTERRUPTED"
    )


# =============================================================================
# 14. FINAL
# =============================================================================

print()
print("=" * 80)
print(
    f"FAST-B3 WORKER B{WORKER_SEED} — FINAL"
)
print("=" * 80)

print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)

print(
    "TRAINING_RESULT =",
    TRAINING_RESULT,
)

print(
    "MANIFEST_STATUS =",
    WORKER_STATUS,
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

if (
    TRAIN_RETURN_CODE == 0
    and
    WORKER_STATUS == "completed"
):

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = YES"
    )

else:

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = NOT_COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = NO"
    )

    print(
        "DO_NOT_RELAUNCH_FRESH_AUTOMATICALLY = TRUE"
    )

print("=" * 80)

MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B41
WORKER_SEED = 41
GPU = Tesla T4
RUN_PRODUCTIVE_QUEUE = False
RECOVERY_PREREQUISITES = PASS
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
WORKING_TREE = ''
SCIENTIFIC_CHECKOUT = PASS
MUTANTS_SHA256 = 92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631
WT_SHA256 = 29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a
HDF5_IDENTITY = PASS
B11_REFERENCE_ACCEPTED = PASS

WORKER_SEED_RUNS_BEFORE = []
B41_ABSENT = PASS
WORKER_CONFIG = /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed41_fresh.yaml
RUNTIME_CONFIG_GENERATION = PASS
WORKER_CONFIG_AUDIT = PASS
FRESH_LAUNCH_GUARD = PASS

LAUNCHING PRODUCTIVE B41 FRESH
COMMAND:
/usr/bin/python3 /content/model_b_workspace/repo/scripts/train.py --config /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed41_fresh.yaml --device cuda
EXECUTION = FRESH
RESUME_FROM = NONE
RUN_PRODUCTIVE_QUEUE = False

TRAIN_RETURN_CODE = 0

WORKER_SEED_RUNS_AFTER 

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B3 — INDEPENDENT PRODUCTIVE WORKER
#
# CHANGE ONLY:
#     WORKER_SEED = 23 / 37 / 41 / 53
#
# FRESH TRAINING
# NO RESUME
# ONE SEED PER RUNTIME
# =============================================================================

from pathlib import Path
import json
import subprocess
import sys
import torch

from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
)

from gnn_siamese.config import (
    save_config,
    load_config,
)


# =============================================================================
# EDIT ONLY THIS VALUE
# =============================================================================

WORKER_SEED = 53


# =============================================================================
# FROZEN WORKER CONTRACT
# =============================================================================

ALLOWED_WORKER_SEEDS = {
    23,
    37,
    41,
    53,
}

RUN_PRODUCTIVE_QUEUE = False

CANONICAL_SPLIT_PATH = (
    "splits/leave_position_out_seed_42.json"
)

EXPECTED_SPLIT_FINGERPRINT = (
    "8dcc611a6575b95242fda70d2b29e6b"
    "f4917a9c781bbeebd851e95e71391d7dd"
)


print("=" * 80)
print(
    f"MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B{WORKER_SEED}"
)
print("=" * 80)


# =============================================================================
# 1. FAIL-CLOSED SAFETY
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False

assert WORKER_SEED in ALLOWED_WORKER_SEEDS, (
    f"Invalid worker seed: {WORKER_SEED}"
)

assert torch.cuda.is_available(), (
    "CUDA GPU required."
)


print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "GPU =",
    torch.cuda.get_device_name(0),
)

print(
    "RUN_PRODUCTIVE_QUEUE =",
    RUN_PRODUCTIVE_QUEUE,
)


# =============================================================================
# 2. REQUIRE RECOVERY CELLS 0.1–0.4
# =============================================================================

required_names = [
    "REPO",
    "SRC",
    "EXPECTED_SHA",
    "LOCAL_ROOT",
    "LOCAL_MUTANTS_HDF5",
    "LOCAL_WT_HDF5",
    "DRIVE_PROJECT_BASE",
    "B11_REFERENCE_RUN",
    "B11_ACCEPTANCE",
    "EXPECTED_MUTANTS_SHA256",
    "EXPECTED_WT_SHA256",
    "sha256_file",
]

for name in required_names:

    assert name in globals(), (
        f"Missing recovery variable {name}. "
        "Run cells 0.1–0.4 first."
    )


assert REPO.is_dir()
assert LOCAL_MUTANTS_HDF5.is_file()
assert LOCAL_WT_HDF5.is_file()
assert B11_ACCEPTANCE.is_file()

print(
    "RECOVERY_PREREQUISITES = PASS"
)


# =============================================================================
# 3. EXACT GIT CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()


print(
    "HEAD =",
    head,
)

print(
    "WORKING_TREE =",
    repr(status),
)


assert head == EXPECTED_SHA
assert status == ""

print(
    "SCIENTIFIC_CHECKOUT = PASS"
)


# =============================================================================
# 4. EXACT DATA IDENTITY
# =============================================================================

mut_sha = sha256_file(
    LOCAL_MUTANTS_HDF5
)

wt_sha = sha256_file(
    LOCAL_WT_HDF5
)


assert (
    mut_sha
    == EXPECTED_MUTANTS_SHA256
)

assert (
    wt_sha
    == EXPECTED_WT_SHA256
)


print(
    "MUTANTS_SHA256 =",
    mut_sha,
)

print(
    "WT_SHA256 =",
    wt_sha,
)

print(
    "HDF5_IDENTITY = PASS"
)


# =============================================================================
# 5. B11 REFERENCE MUST REMAIN ACCEPTED
# =============================================================================

b11_acceptance = json.loads(
    B11_ACCEPTANCE.read_text(
        encoding="utf-8"
    )
)


assert (
    b11_acceptance["status"]
    == "accepted"
)

assert (
    int(
        b11_acceptance["run_seed"]
    )
    == 11
)


print(
    "B11_REFERENCE_ACCEPTED = PASS"
)


# =============================================================================
# 6. PRODUCTIVE ROOTS
# =============================================================================

BASE_B_CONFIG = (
    REPO
    / "configs"
    / "model_b_a9.yaml"
)

MODEL_A_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_a"
    / "runs"
    / "model_a_a9"
)

MODEL_B_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
)

MODEL_B_ARCH_ROOT = (
    MODEL_B_A9_ROOT
    / "model_b_graph_level_relational"
)

WORKER_CONFIG_ROOT = (
    LOCAL_ROOT
    / "runtime_configs"
    / "fast_workers"
)

WORKER_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

WORKER_CONFIG = (
    WORKER_CONFIG_ROOT
    / f"model_b_a9_seed{WORKER_SEED}_fresh.yaml"
)


assert BASE_B_CONFIG.is_file()
assert MODEL_B_ARCH_ROOT.is_dir()


# =============================================================================
# 7. SCAN FOR AN EXISTING RUN OF THIS SEED
#
# Important for concurrent workers:
# inspect manifests by SEED, not by directory count.
# =============================================================================

def find_seed_runs(seed):

    results = []

    for run_dir in sorted(
        MODEL_B_ARCH_ROOT.glob(
            "run_*"
        )
    ):

        manifest_path = (
            run_dir
            / "run_manifest.json"
        )

        if not manifest_path.is_file():
            continue

        try:

            payload = json.loads(
                manifest_path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:
            continue


        manifest_seed = (
            payload
            .get(
                "configuration",
                {}
            )
            .get(
                "seed"
            )
        )


        if manifest_seed == seed:

            results.append(
                {
                    "run_dir":
                        run_dir,

                    "status":
                        payload.get(
                            "status"
                        ),

                    "manifest":
                        payload,
                }
            )

    return results


seed_runs_before = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_BEFORE =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_before
    ],
)


assert seed_runs_before == [], (
    f"STOP: seed {WORKER_SEED} already has a run. "
    "Do not launch fresh automatically."
)


print(
    f"B{WORKER_SEED}_ABSENT = PASS"
)


# =============================================================================
# 8. GENERATE OFFICIAL RUNTIME CONFIG
# =============================================================================

config_a, config_b = (
    resolve_a9_runtime_configs(
        BASE_B_CONFIG,
        architecture=(
            "model_b_graph_level_relational"
        ),
        run_seed=WORKER_SEED,
        mutants_hdf5=(
            LOCAL_MUTANTS_HDF5
        ),
        wt_hdf5=(
            LOCAL_WT_HDF5
        ),
        output_root=(
            MODEL_B_A9_ROOT
        ),
        peer_output_root=(
            MODEL_A_A9_ROOT
        ),
        repo_root=REPO,
    )
)


# DO NOT convert this to an absolute path.
assert (
    config_b["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    int(
        config_b["split"]["seed"]
    )
    == 42
)

assert (
    config_b["split"]["allow_create"]
    is False
)


clean_config = {
    key: value

    for key, value
    in config_b.items()

    if not str(
        key
    ).startswith("__")
}


save_config(
    clean_config,
    WORKER_CONFIG,
)


runtime = load_config(
    WORKER_CONFIG
)


print(
    "WORKER_CONFIG =",
    WORKER_CONFIG,
)

print(
    "RUNTIME_CONFIG_GENERATION = PASS"
)


# =============================================================================
# 9. EXACT WORKER CONFIG AUDIT
# =============================================================================

assert (
    runtime["model"]["architecture"]
    == "model_b_graph_level_relational"
)

assert (
    int(
        runtime["project"]["seed"]
    )
    == WORKER_SEED
)


for field in [
    "seed_python",
    "seed_numpy",
    "seed_torch",
    "seed_cuda",
    "seed_dataloader",
]:

    assert (
        int(
            runtime[
                "reproducibility"
            ][field]
        )
        == WORKER_SEED
    )


assert (
    int(
        runtime["split"]["seed"]
    )
    == 42
)

assert (
    runtime["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    runtime["split"]["allow_create"]
    is False
)


assert (
    runtime["a9"][
        "expected_split_fingerprint"
    ]
    == EXPECTED_SPLIT_FINGERPRINT
)


assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["mutants"]
    == EXPECTED_MUTANTS_SHA256
)

assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["wt_companion"]
    == EXPECTED_WT_SHA256
)


assert int(
    runtime["training"]["epochs"]
) == 100

assert int(
    runtime["training"]["batch_size"]
) == 4

assert (
    runtime["training"]["device"]
    == "cuda"
)

assert (
    runtime["loss"]["main"]
    == "nt_xent"
)

assert (
    float(
        runtime["loss"]["temperature"]
    )
    == 0.2
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["mode"]
    == "same_position"
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["strict"]
    is True
)


print(
    "WORKER_CONFIG_AUDIT = PASS"
)


# =============================================================================
# 10. FINAL PRE-LAUNCH FRESH CHECK
#
# Check again immediately before train.py.
# This prevents accidentally double-launching the same seed.
# =============================================================================

seed_runs_immediately_before = (
    find_seed_runs(
        WORKER_SEED
    )
)


assert (
    seed_runs_immediately_before
    == []
), (
    f"STOP: seed {WORKER_SEED} appeared before launch."
)


print(
    "FRESH_LAUNCH_GUARD = PASS"
)


# =============================================================================
# 11. TRAIN
#
# No --resume-from.
# This is explicitly FRESH.
# =============================================================================

command = [
    sys.executable,
    str(
        REPO
        / "scripts"
        / "train.py"
    ),
    "--config",
    str(
        WORKER_CONFIG
    ),
    "--device",
    "cuda",
]


print()
print("=" * 80)
print(
    f"LAUNCHING PRODUCTIVE B{WORKER_SEED} FRESH"
)
print("=" * 80)

print(
    "COMMAND:"
)

print(
    " ".join(
        command
    )
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)


result = subprocess.run(
    command,
    cwd=REPO,
)


TRAIN_RETURN_CODE = (
    result.returncode
)


print()
print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)


# =============================================================================
# 12. DISCOVER EXACT RUN OF THIS WORKER SEED
# =============================================================================

seed_runs_after = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_AFTER =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_after
    ],
)


new_seed_runs = [
    x
    for x in seed_runs_after
    if x["run_dir"]
    not in {
        old["run_dir"]
        for old
        in seed_runs_before
    }
]


assert len(
    new_seed_runs
) == 1, (
    f"Expected exactly one new B{WORKER_SEED} run, "
    f"found {len(new_seed_runs)}."
)


WORKER_RUN = (
    new_seed_runs[0][
        "run_dir"
    ]
)

WORKER_STATUS = (
    new_seed_runs[0][
        "status"
    ]
)


print()
print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "WORKER_STATUS =",
    WORKER_STATUS,
)


# =============================================================================
# 13. TRAIN RETURN / MANIFEST CONSISTENCY
# =============================================================================

if TRAIN_RETURN_CODE == 0:

    assert (
        WORKER_STATUS
        == "completed"
    ), (
        "train.py returned 0 but manifest is not completed."
    )

    TRAINING_RESULT = (
        "COMPLETED"
    )

else:

    TRAINING_RESULT = (
        "FAILED_OR_INTERRUPTED"
    )


# =============================================================================
# 14. FINAL
# =============================================================================

print()
print("=" * 80)
print(
    f"FAST-B3 WORKER B{WORKER_SEED} — FINAL"
)
print("=" * 80)

print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)

print(
    "TRAINING_RESULT =",
    TRAINING_RESULT,
)

print(
    "MANIFEST_STATUS =",
    WORKER_STATUS,
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

if (
    TRAIN_RETURN_CODE == 0
    and
    WORKER_STATUS == "completed"
):

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = YES"
    )

else:

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = NOT_COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = NO"
    )

    print(
        "DO_NOT_RELAUNCH_FRESH_AUTOMATICALLY = TRUE"
    )

print("=" * 80)

MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B53
WORKER_SEED = 53
GPU = Tesla T4
RUN_PRODUCTIVE_QUEUE = False
RECOVERY_PREREQUISITES = PASS
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
WORKING_TREE = ''
SCIENTIFIC_CHECKOUT = PASS
MUTANTS_SHA256 = 92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631
WT_SHA256 = 29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a
HDF5_IDENTITY = PASS
B11_REFERENCE_ACCEPTED = PASS

WORKER_SEED_RUNS_BEFORE = []
B53_ABSENT = PASS
WORKER_CONFIG = /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed53_fresh.yaml
RUNTIME_CONFIG_GENERATION = PASS
WORKER_CONFIG_AUDIT = PASS
FRESH_LAUNCH_GUARD = PASS

LAUNCHING PRODUCTIVE B53 FRESH
COMMAND:
/usr/bin/python3 /content/model_b_workspace/repo/scripts/train.py --config /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed53_fresh.yaml --device cuda
EXECUTION = FRESH
RESUME_FROM = NONE
RUN_PRODUCTIVE_QUEUE = False

TRAIN_RETURN_CODE = 0

WORKER_SEED_RUNS_AFTER 

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B3 — INDEPENDENT PRODUCTIVE WORKER
#
# CHANGE ONLY:
#     WORKER_SEED = 23 / 37 / 41 / 53
#
# FRESH TRAINING
# NO RESUME
# ONE SEED PER RUNTIME
# =============================================================================

from pathlib import Path
import json
import subprocess
import sys
import torch

from scripts.a9_preflight import (
    resolve_a9_runtime_configs,
)

from gnn_siamese.config import (
    save_config,
    load_config,
)


# =============================================================================
# EDIT ONLY THIS VALUE
# =============================================================================

WORKER_SEED = 37


# =============================================================================
# FROZEN WORKER CONTRACT
# =============================================================================

ALLOWED_WORKER_SEEDS = {
    23,
    37,
    41,
    53,
}

RUN_PRODUCTIVE_QUEUE = False

CANONICAL_SPLIT_PATH = (
    "splits/leave_position_out_seed_42.json"
)

EXPECTED_SPLIT_FINGERPRINT = (
    "8dcc611a6575b95242fda70d2b29e6b"
    "f4917a9c781bbeebd851e95e71391d7dd"
)


print("=" * 80)
print(
    f"MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B{WORKER_SEED}"
)
print("=" * 80)


# =============================================================================
# 1. FAIL-CLOSED SAFETY
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False

assert WORKER_SEED in ALLOWED_WORKER_SEEDS, (
    f"Invalid worker seed: {WORKER_SEED}"
)

assert torch.cuda.is_available(), (
    "CUDA GPU required."
)


print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "GPU =",
    torch.cuda.get_device_name(0),
)

print(
    "RUN_PRODUCTIVE_QUEUE =",
    RUN_PRODUCTIVE_QUEUE,
)


# =============================================================================
# 2. REQUIRE RECOVERY CELLS 0.1–0.4
# =============================================================================

required_names = [
    "REPO",
    "SRC",
    "EXPECTED_SHA",
    "LOCAL_ROOT",
    "LOCAL_MUTANTS_HDF5",
    "LOCAL_WT_HDF5",
    "DRIVE_PROJECT_BASE",
    "B11_REFERENCE_RUN",
    "B11_ACCEPTANCE",
    "EXPECTED_MUTANTS_SHA256",
    "EXPECTED_WT_SHA256",
    "sha256_file",
]

for name in required_names:

    assert name in globals(), (
        f"Missing recovery variable {name}. "
        "Run cells 0.1–0.4 first."
    )


assert REPO.is_dir()
assert LOCAL_MUTANTS_HDF5.is_file()
assert LOCAL_WT_HDF5.is_file()
assert B11_ACCEPTANCE.is_file()

print(
    "RECOVERY_PREREQUISITES = PASS"
)


# =============================================================================
# 3. EXACT GIT CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()


print(
    "HEAD =",
    head,
)

print(
    "WORKING_TREE =",
    repr(status),
)


assert head == EXPECTED_SHA
assert status == ""

print(
    "SCIENTIFIC_CHECKOUT = PASS"
)


# =============================================================================
# 4. EXACT DATA IDENTITY
# =============================================================================

mut_sha = sha256_file(
    LOCAL_MUTANTS_HDF5
)

wt_sha = sha256_file(
    LOCAL_WT_HDF5
)


assert (
    mut_sha
    == EXPECTED_MUTANTS_SHA256
)

assert (
    wt_sha
    == EXPECTED_WT_SHA256
)


print(
    "MUTANTS_SHA256 =",
    mut_sha,
)

print(
    "WT_SHA256 =",
    wt_sha,
)

print(
    "HDF5_IDENTITY = PASS"
)


# =============================================================================
# 5. B11 REFERENCE MUST REMAIN ACCEPTED
# =============================================================================

b11_acceptance = json.loads(
    B11_ACCEPTANCE.read_text(
        encoding="utf-8"
    )
)


assert (
    b11_acceptance["status"]
    == "accepted"
)

assert (
    int(
        b11_acceptance["run_seed"]
    )
    == 11
)


print(
    "B11_REFERENCE_ACCEPTED = PASS"
)


# =============================================================================
# 6. PRODUCTIVE ROOTS
# =============================================================================

BASE_B_CONFIG = (
    REPO
    / "configs"
    / "model_b_a9.yaml"
)

MODEL_A_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_a"
    / "runs"
    / "model_a_a9"
)

MODEL_B_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
)

MODEL_B_ARCH_ROOT = (
    MODEL_B_A9_ROOT
    / "model_b_graph_level_relational"
)

WORKER_CONFIG_ROOT = (
    LOCAL_ROOT
    / "runtime_configs"
    / "fast_workers"
)

WORKER_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

WORKER_CONFIG = (
    WORKER_CONFIG_ROOT
    / f"model_b_a9_seed{WORKER_SEED}_fresh.yaml"
)


assert BASE_B_CONFIG.is_file()
assert MODEL_B_ARCH_ROOT.is_dir()


# =============================================================================
# 7. SCAN FOR AN EXISTING RUN OF THIS SEED
#
# Important for concurrent workers:
# inspect manifests by SEED, not by directory count.
# =============================================================================

def find_seed_runs(seed):

    results = []

    for run_dir in sorted(
        MODEL_B_ARCH_ROOT.glob(
            "run_*"
        )
    ):

        manifest_path = (
            run_dir
            / "run_manifest.json"
        )

        if not manifest_path.is_file():
            continue

        try:

            payload = json.loads(
                manifest_path.read_text(
                    encoding="utf-8"
                )
            )

        except Exception:
            continue


        manifest_seed = (
            payload
            .get(
                "configuration",
                {}
            )
            .get(
                "seed"
            )
        )


        if manifest_seed == seed:

            results.append(
                {
                    "run_dir":
                        run_dir,

                    "status":
                        payload.get(
                            "status"
                        ),

                    "manifest":
                        payload,
                }
            )

    return results


seed_runs_before = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_BEFORE =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_before
    ],
)


assert seed_runs_before == [], (
    f"STOP: seed {WORKER_SEED} already has a run. "
    "Do not launch fresh automatically."
)


print(
    f"B{WORKER_SEED}_ABSENT = PASS"
)


# =============================================================================
# 8. GENERATE OFFICIAL RUNTIME CONFIG
# =============================================================================

config_a, config_b = (
    resolve_a9_runtime_configs(
        BASE_B_CONFIG,
        architecture=(
            "model_b_graph_level_relational"
        ),
        run_seed=WORKER_SEED,
        mutants_hdf5=(
            LOCAL_MUTANTS_HDF5
        ),
        wt_hdf5=(
            LOCAL_WT_HDF5
        ),
        output_root=(
            MODEL_B_A9_ROOT
        ),
        peer_output_root=(
            MODEL_A_A9_ROOT
        ),
        repo_root=REPO,
    )
)


# DO NOT convert this to an absolute path.
assert (
    config_b["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    int(
        config_b["split"]["seed"]
    )
    == 42
)

assert (
    config_b["split"]["allow_create"]
    is False
)


clean_config = {
    key: value

    for key, value
    in config_b.items()

    if not str(
        key
    ).startswith("__")
}


save_config(
    clean_config,
    WORKER_CONFIG,
)


runtime = load_config(
    WORKER_CONFIG
)


print(
    "WORKER_CONFIG =",
    WORKER_CONFIG,
)

print(
    "RUNTIME_CONFIG_GENERATION = PASS"
)


# =============================================================================
# 9. EXACT WORKER CONFIG AUDIT
# =============================================================================

assert (
    runtime["model"]["architecture"]
    == "model_b_graph_level_relational"
)

assert (
    int(
        runtime["project"]["seed"]
    )
    == WORKER_SEED
)


for field in [
    "seed_python",
    "seed_numpy",
    "seed_torch",
    "seed_cuda",
    "seed_dataloader",
]:

    assert (
        int(
            runtime[
                "reproducibility"
            ][field]
        )
        == WORKER_SEED
    )


assert (
    int(
        runtime["split"]["seed"]
    )
    == 42
)

assert (
    runtime["split"]["persist_path"]
    == CANONICAL_SPLIT_PATH
)

assert (
    runtime["split"]["allow_create"]
    is False
)


assert (
    runtime["a9"][
        "expected_split_fingerprint"
    ]
    == EXPECTED_SPLIT_FINGERPRINT
)


assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["mutants"]
    == EXPECTED_MUTANTS_SHA256
)

assert (
    runtime["a9"][
        "expected_hdf5_fingerprints"
    ]["wt_companion"]
    == EXPECTED_WT_SHA256
)


assert int(
    runtime["training"]["epochs"]
) == 100

assert int(
    runtime["training"]["batch_size"]
) == 4

assert (
    runtime["training"]["device"]
    == "cuda"
)

assert (
    runtime["loss"]["main"]
    == "nt_xent"
)

assert (
    float(
        runtime["loss"]["temperature"]
    )
    == 0.2
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["mode"]
    == "same_position"
)

assert (
    runtime["loss"][
        "false_negative_mask"
    ]["strict"]
    is True
)


print(
    "WORKER_CONFIG_AUDIT = PASS"
)


# =============================================================================
# 10. FINAL PRE-LAUNCH FRESH CHECK
#
# Check again immediately before train.py.
# This prevents accidentally double-launching the same seed.
# =============================================================================

seed_runs_immediately_before = (
    find_seed_runs(
        WORKER_SEED
    )
)


assert (
    seed_runs_immediately_before
    == []
), (
    f"STOP: seed {WORKER_SEED} appeared before launch."
)


print(
    "FRESH_LAUNCH_GUARD = PASS"
)


# =============================================================================
# 11. TRAIN
#
# No --resume-from.
# This is explicitly FRESH.
# =============================================================================

command = [
    sys.executable,
    str(
        REPO
        / "scripts"
        / "train.py"
    ),
    "--config",
    str(
        WORKER_CONFIG
    ),
    "--device",
    "cuda",
]


print()
print("=" * 80)
print(
    f"LAUNCHING PRODUCTIVE B{WORKER_SEED} FRESH"
)
print("=" * 80)

print(
    "COMMAND:"
)

print(
    " ".join(
        command
    )
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print("=" * 80)


result = subprocess.run(
    command,
    cwd=REPO,
)


TRAIN_RETURN_CODE = (
    result.returncode
)


print()
print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)


# =============================================================================
# 12. DISCOVER EXACT RUN OF THIS WORKER SEED
# =============================================================================

seed_runs_after = find_seed_runs(
    WORKER_SEED
)


print()
print(
    "WORKER_SEED_RUNS_AFTER =",
    [
        {
            "run_dir":
                str(x["run_dir"]),

            "status":
                x["status"],
        }

        for x in seed_runs_after
    ],
)


new_seed_runs = [
    x
    for x in seed_runs_after
    if x["run_dir"]
    not in {
        old["run_dir"]
        for old
        in seed_runs_before
    }
]


assert len(
    new_seed_runs
) == 1, (
    f"Expected exactly one new B{WORKER_SEED} run, "
    f"found {len(new_seed_runs)}."
)


WORKER_RUN = (
    new_seed_runs[0][
        "run_dir"
    ]
)

WORKER_STATUS = (
    new_seed_runs[0][
        "status"
    ]
)


print()
print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "WORKER_STATUS =",
    WORKER_STATUS,
)


# =============================================================================
# 13. TRAIN RETURN / MANIFEST CONSISTENCY
# =============================================================================

if TRAIN_RETURN_CODE == 0:

    assert (
        WORKER_STATUS
        == "completed"
    ), (
        "train.py returned 0 but manifest is not completed."
    )

    TRAINING_RESULT = (
        "COMPLETED"
    )

else:

    TRAINING_RESULT = (
        "FAILED_OR_INTERRUPTED"
    )


# =============================================================================
# 14. FINAL
# =============================================================================

print()
print("=" * 80)
print(
    f"FAST-B3 WORKER B{WORKER_SEED} — FINAL"
)
print("=" * 80)

print(
    "WORKER_SEED =",
    WORKER_SEED,
)

print(
    "WORKER_RUN =",
    WORKER_RUN,
)

print(
    "TRAIN_RETURN_CODE =",
    TRAIN_RETURN_CODE,
)

print(
    "TRAINING_RESULT =",
    TRAINING_RESULT,
)

print(
    "MANIFEST_STATUS =",
    WORKER_STATUS,
)

print(
    "EXECUTION = FRESH"
)

print(
    "RESUME_FROM = NONE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

if (
    TRAIN_RETURN_CODE == 0
    and
    WORKER_STATUS == "completed"
):

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = YES"
    )

else:

    print(
        f"B{WORKER_SEED}_PRODUCTIVE_TRAINING = NOT_COMPLETED"
    )

    print(
        "READY_FOR_BATCH_POSTRUN_AUDIT = NO"
    )

    print(
        "DO_NOT_RELAUNCH_FRESH_AUTOMATICALLY = TRUE"
    )

print("=" * 80)

MODEL B A9 — FAST-B3 PRODUCTIVE WORKER B37
WORKER_SEED = 37
GPU = Tesla T4
RUN_PRODUCTIVE_QUEUE = False
RECOVERY_PREREQUISITES = PASS
HEAD = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
WORKING_TREE = ''
SCIENTIFIC_CHECKOUT = PASS
MUTANTS_SHA256 = 92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631
WT_SHA256 = 29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a
HDF5_IDENTITY = PASS
B11_REFERENCE_ACCEPTED = PASS

WORKER_SEED_RUNS_BEFORE = []
B37_ABSENT = PASS
WORKER_CONFIG = /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed37_fresh.yaml
RUNTIME_CONFIG_GENERATION = PASS
WORKER_CONFIG_AUDIT = PASS
FRESH_LAUNCH_GUARD = PASS

LAUNCHING PRODUCTIVE B37 FRESH
COMMAND:
/usr/bin/python3 /content/model_b_workspace/repo/scripts/train.py --config /content/model_b_workspace/runtime_configs/fast_workers/model_b_a9_seed37_fresh.yaml --device cuda
EXECUTION = FRESH
RESUME_FROM = NONE
RUN_PRODUCTIVE_QUEUE = False

TRAIN_RETURN_CODE = 0

WORKER_SEED_RUNS_AFTER 

# Auditoria de seeds

In [ ]:
# =============================================================================
# MODEL B A9 — FAST-B4 CONSTANTS
# Frozen scientific fingerprints
# =============================================================================

EXPECTED_MUTANTS_SHA256 = (
    "92eb242f5565db6194a5e29e3469775f"
    "bf7d8c07280ed2d8cdfd5ab98b5b5631"
)

EXPECTED_WT_SHA256 = (
    "29f68e98ae300207511594e0baf7621b"
    "a9e67c85d8df12ceee812a1dea3aa91a"
)

EXPECTED_COMBINED_SHA256 = (
    "9a320862a54566d232e1ef2a4eafa468"
    "cf900a19186dc0fd7c92aaf0abe06c68"
)

EXPECTED_SPLIT_FINGERPRINT = (
    "8dcc611a6575b95242fda70d2b29e6b"
    "f4917a9c781bbeebd851e95e71391d7dd"
)

print("EXPECTED_MUTANTS_SHA256 =", EXPECTED_MUTANTS_SHA256)
print("EXPECTED_WT_SHA256 =", EXPECTED_WT_SHA256)
print("EXPECTED_COMBINED_SHA256 =", EXPECTED_COMBINED_SHA256)
print("EXPECTED_SPLIT_FINGERPRINT =", EXPECTED_SPLIT_FINGERPRINT)

assert len(EXPECTED_MUTANTS_SHA256) == 64
assert len(EXPECTED_WT_SHA256) == 64
assert len(EXPECTED_COMBINED_SHA256) == 64
assert len(EXPECTED_SPLIT_FINGERPRINT) == 64

print("FAST_B4_FROZEN_CONSTANTS = PASS")

EXPECTED_MUTANTS_SHA256 = 92eb242f5565db6194a5e29e3469775fbf7d8c07280ed2d8cdfd5ab98b5b5631
EXPECTED_WT_SHA256 = 29f68e98ae300207511594e0baf7621ba9e67c85d8df12ceee812a1dea3aa91a
EXPECTED_COMBINED_SHA256 = 9a320862a54566d232e1ef2a4eafa468cf900a19186dc0fd7c92aaf0abe06c68
EXPECTED_SPLIT_FINGERPRINT = 8dcc611a6575b95242fda70d2b29e6bf4917a9c781bbeebd851e95e71391d7dd
FAST_B4_FROZEN_CONSTANTS = PASS


In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B4 — JOINT POST-RUN AUDIT
#
# SEEDS:
# 11 / 23 / 37 / 41 / 53
#
# READ ONLY
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# NO ACCEPTANCE MATERIALIZATION
# =============================================================================

from pathlib import Path
import json
import math
import subprocess

from gnn_siamese.config import load_config
from gnn_siamese.training import load_checkpoint
from gnn_siamese.training.a9_contract import (
    A9ContractError,
    validate_a9_run_acceptance,
)


print("=" * 80)
print("MODEL B A9 — FAST-B4 JOINT POST-RUN AUDIT")
print("=" * 80)

RUN_PRODUCTIVE_QUEUE = False
assert RUN_PRODUCTIVE_QUEUE is False


# =============================================================================
# 1. EXACT PRODUCTIVE RUN MAP
# =============================================================================

MODEL_B_ARCH_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
    / "model_b_graph_level_relational"
)

RUNS = {
    11: (
        MODEL_B_ARCH_ROOT
        / "run_20260914T143850.128514Z-e382b31e"
    ),

    23: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T095945.494670Z-9a96c769"
    ),

    37: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T114757.375432Z-cf5a1245"
    ),

    41: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T103715.331004Z-a30da1be"
    ),

    53: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T111311.350252Z-628b5469"
    ),
}

PRODUCTIVE_SEEDS = (
    11,
    23,
    37,
    41,
    53,
)


# =============================================================================
# 2. SCIENTIFIC CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()

assert head == EXPECTED_SHA
assert status == ""

print("SCIENTIFIC_COMMIT =", head)
print("SCIENTIFIC_CHECKOUT = PASS")


# =============================================================================
# 3. HELPERS
# =============================================================================

def load_json(path):

    payload = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    assert isinstance(payload, dict)

    return payload


def numeric_values_finite(value):

    if isinstance(value, bool):
        return True

    if value is None:
        return True

    if isinstance(value, str):
        return True

    if isinstance(value, (int, float)):
        return math.isfinite(
            float(value)
        )

    if isinstance(value, dict):

        return all(
            numeric_values_finite(v)
            for v in value.values()
        )

    if isinstance(value, list):

        return all(
            numeric_values_finite(v)
            for v in value
        )

    return True


# =============================================================================
# 4. AUDIT EACH PRODUCTIVE RUN
# =============================================================================

audit_results = {}


for seed in PRODUCTIVE_SEEDS:

    print()
    print("=" * 80)
    print(f"POST-RUN AUDIT — B{seed}")
    print("=" * 80)

    run_dir = RUNS[seed]

    assert run_dir.is_dir(), (
        f"Missing run directory for B{seed}: {run_dir}"
    )

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    config_path = (
        run_dir
        / "config_resolved.yaml"
    )

    split_path = (
        run_dir
        / "split.json"
    )

    gradient_path = (
        run_dir
        / "gradient_audit.json"
    )

    best_path = (
        run_dir
        / "checkpoints"
        / "best.pt"
    )

    last_path = (
        run_dir
        / "checkpoints"
        / "last.pt"
    )

    metrics_path = (
        run_dir
        / "metrics.jsonl"
    )


    # -------------------------------------------------------------------------
    # Required artifacts
    # -------------------------------------------------------------------------

    required_paths = [
        manifest_path,
        config_path,
        split_path,
        gradient_path,
        best_path,
        last_path,
        metrics_path,
    ]

    for path in required_paths:

        assert path.is_file(), (
            f"B{seed}: missing {path}"
        )

        assert path.stat().st_size > 0, (
            f"B{seed}: empty {path}"
        )

    print("REQUIRED_ARTIFACTS = PASS")


    # -------------------------------------------------------------------------
    # Manifest + config
    # -------------------------------------------------------------------------

    manifest = load_json(
        manifest_path
    )

    config = load_config(
        config_path
    )

    assert (
        manifest["status"]
        == "completed"
    )

    assert (
        manifest["architecture"]
        == "model_b_graph_level_relational"
    )

    assert (
        int(
            manifest[
                "configuration"
            ]["seed"]
        )
        == seed
    )

    assert (
        int(
            manifest[
                "configuration"
            ][
                "seed_bundle"
            ]["split"]
        )
        == 42
    )

    assert (
        int(
            config["project"]["seed"]
        )
        == seed
    )

    print("MANIFEST_STATUS_COMPLETED = PASS")
    print("RUN_SEED_IDENTITY = PASS")
    print("SPLIT_SEED_42 = PASS")


    # -------------------------------------------------------------------------
    # Fresh execution
    # -------------------------------------------------------------------------

    resume_from = (
        manifest
        .get(
            "training",
            {}
        )
        .get(
            "resume_from"
        )
    )

    assert resume_from is None, (
        f"B{seed}: expected fresh run, "
        f"resume_from={resume_from!r}"
    )

    print("FRESH_NO_RESUME = PASS")


    # -------------------------------------------------------------------------
    # Official A9 validator:
    #
    # It MUST pass every post-run check and stop ONLY because
    # representations have not yet been supplied.
    # -------------------------------------------------------------------------

    try:

        validate_a9_run_acceptance(
            run_dir,
            config,
            representations=None,
        )

    except A9ContractError as exc:

        expected_message = (
            "Required A9 pair representations "
            "were not supplied for acceptance."
        )

        assert (
            str(exc)
            == expected_message
        ), (
            f"B{seed} failed before representation "
            f"stage: {exc}"
        )

    else:

        raise AssertionError(
            f"B{seed}: validator unexpectedly "
            "accepted without required representations."
        )


    print(
        "OFFICIAL_POSTRUN_CHECKS_BEFORE_REPRESENTATIONS = PASS"
    )


    # -------------------------------------------------------------------------
    # Checkpoints
    # -------------------------------------------------------------------------

    best = load_checkpoint(
        best_path
    )

    last = load_checkpoint(
        last_path
    )

    best_epoch = int(
        best["epoch_completed"]
    )

    last_epoch = int(
        last["epoch_completed"]
    )

    manifest_epochs = int(
        manifest[
            "training"
        ][
            "epochs_completed"
        ]
    )


    assert best_epoch >= 1

    assert last_epoch >= best_epoch

    assert (
        last_epoch
        == manifest_epochs
    )


    assert (
        best[
            "split_fingerprint"
        ]
        == EXPECTED_SPLIT_FINGERPRINT
    )

    assert (
        last[
            "split_fingerprint"
        ]
        == EXPECTED_SPLIT_FINGERPRINT
    )


    print(
        "BEST_EPOCH =",
        best_epoch,
    )

    print(
        "LAST_EPOCH =",
        last_epoch,
    )

    print(
        "CHECKPOINT_EPOCH_CONSISTENCY = PASS"
    )

    print(
        "CHECKPOINT_SPLIT_IDENTITY = PASS"
    )


    # -------------------------------------------------------------------------
    # Metrics
    # -------------------------------------------------------------------------

    metric_lines = [
        line
        for line in (
            metrics_path
            .read_text(
                encoding="utf-8"
            )
            .splitlines()
        )
        if line.strip()
    ]

    metric_rows = [
        json.loads(line)
        for line in metric_lines
    ]


    assert (
        len(metric_rows)
        == manifest_epochs
    ), (
        f"B{seed}: metrics rows "
        f"{len(metric_rows)} != "
        f"epochs {manifest_epochs}"
    )

    assert numeric_values_finite(
        metric_rows
    )


    print(
        "METRIC_ROWS =",
        len(metric_rows),
    )

    print(
        "METRICS_EPOCH_COUNT = PASS"
    )

    print(
        "METRICS_FINITE = PASS"
    )


    # -------------------------------------------------------------------------
    # HDF5 identity directly from manifest
    # -------------------------------------------------------------------------

    fp = (
        manifest[
            "data"
        ][
            "hdf5_content_fingerprint"
        ]
    )

    file_records = {
        item["role"]:
            item["digest"]

        for item in fp["files"]
    }


    assert (
        file_records["mutants"]
        == EXPECTED_MUTANTS_SHA256
    )

    assert (
        file_records["wt_companion"]
        == EXPECTED_WT_SHA256
    )

    assert (
        fp["combined"]["digest"]
        == EXPECTED_COMBINED_SHA256
    )


    print(
        "MUTANTS_FINGERPRINT = PASS"
    )

    print(
        "WT_FINGERPRINT = PASS"
    )

    print(
        "COMBINED_FINGERPRINT = PASS"
    )


    # -------------------------------------------------------------------------
    # Dataset inventory
    # -------------------------------------------------------------------------

    inventory = (
        manifest[
            "data"
        ][
            "inventory"
        ]
    )

    assert (
        int(
            inventory[
                "biological_variants"
            ]
        )
        == 483
    )

    assert (
        int(
            inventory[
                "native_wt_controls"
            ]
        )
        == 1
    )


    print(
        "BIOLOGICAL_VARIANTS_483 = PASS"
    )

    print(
        "NATIVE_WT_CONTROLS_1 = PASS"
    )


    # -------------------------------------------------------------------------
    # Result
    # -------------------------------------------------------------------------

    audit_results[seed] = {
        "run_dir":
            str(run_dir),

        "epochs_completed":
            manifest_epochs,

        "best_epoch":
            best_epoch,

        "last_epoch":
            last_epoch,

        "stopped_early":
            manifest[
                "training"
            ].get(
                "stopped_early"
            ),

        "status":
            "PASS",
    }


    print()
    print(
        f"B{seed}_POSTRUN_AUDIT = PASS"
    )


# =============================================================================
# 5. FIVE-SEED CONSISTENCY
# =============================================================================

assert (
    set(audit_results)
    == set(PRODUCTIVE_SEEDS)
)

assert all(
    item["status"] == "PASS"
    for item in audit_results.values()
)


run_paths = [
    item["run_dir"]
    for item in audit_results.values()
]

assert (
    len(set(run_paths))
    == 5
)


print()
print("=" * 80)
print("FIVE-SEED SUMMARY")
print("=" * 80)

for seed in PRODUCTIVE_SEEDS:

    item = audit_results[
        seed
    ]

    print(
        f"B{seed}: "
        f"best_epoch={item['best_epoch']} "
        f"last_epoch={item['last_epoch']} "
        f"stopped_early={item['stopped_early']} "
        f"status={item['status']}"
    )


# =============================================================================
# FINAL
# =============================================================================

FAST_B4_ALL_PASS = True

print()
print("=" * 80)
print("FAST-B4 — FINAL")
print("=" * 80)

for seed in PRODUCTIVE_SEEDS:

    print(
        f"B{seed}_POSTRUN_AUDIT = PASS"
    )

print()
print(
    "FIVE_PRODUCTIVE_RUNS_PRESENT = PASS"
)

print(
    "FIVE_PRODUCTIVE_RUNS_DISTINCT = PASS"
)

print(
    "FIVE_SEED_POSTRUN_AUDIT = PASS"
)

print(
    "OFFICIAL_A9_PREREP_CHECKS = PASS"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_ACCEPTANCE_WRITTEN = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print(
    "READY_FOR_FAST_B5_REPRESENTATIONS = YES"
)

print("=" * 80)

MODEL B A9 — FAST-B4 JOINT POST-RUN AUDIT
SCIENTIFIC_COMMIT = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
SCIENTIFIC_CHECKOUT = PASS

POST-RUN AUDIT — B11
REQUIRED_ARTIFACTS = PASS
MANIFEST_STATUS_COMPLETED = PASS
RUN_SEED_IDENTITY = PASS
SPLIT_SEED_42 = PASS
FRESH_NO_RESUME = PASS
OFFICIAL_POSTRUN_CHECKS_BEFORE_REPRESENTATIONS = PASS
BEST_EPOCH = 24
LAST_EPOCH = 39
CHECKPOINT_EPOCH_CONSISTENCY = PASS
CHECKPOINT_SPLIT_IDENTITY = PASS
METRIC_ROWS = 39
METRICS_EPOCH_COUNT = PASS
METRICS_FINITE = PASS
MUTANTS_FINGERPRINT = PASS
WT_FINGERPRINT = PASS
COMBINED_FINGERPRINT = PASS
BIOLOGICAL_VARIANTS_483 = PASS
NATIVE_WT_CONTROLS_1 = PASS

B11_POSTRUN_AUDIT = PASS

POST-RUN AUDIT — B23
REQUIRED_ARTIFACTS = PASS
MANIFEST_STATUS_COMPLETED = PASS
RUN_SEED_IDENTITY = PASS
SPLIT_SEED_42 = PASS
FRESH_NO_RESUME = PASS
OFFICIAL_POSTRUN_CHECKS_BEFORE_REPRESENTATIONS = PASS
BEST_EPOCH = 38
LAST_EPOCH = 53
CHECKPOINT_EPOCH_CONSISTENCY = PASS
CHECKPOINT_SPLIT_IDENTITY = PASS
METRIC_ROWS = 53
METRICS_EPOCH_C

In [ ]:
# =============================================================================
# MODEL B A9 — FAST TRACK
# FAST-B5 — REPRESENTATIONS + OFFICIAL A9 ACCEPTANCE
#
# B11:
#   preserve existing official acceptance
#
# B23 / B37 / B41 / B53:
#   - rebuild model from run config
#   - load BEST checkpoint
#   - extract 483 representations without augmentation
#   - validate representations
#   - call official scripts/a9_acceptance.py
#   - persist a9_acceptance.json in each productive run
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# =============================================================================

from pathlib import Path
import hashlib
import json
import subprocess
import sys
import gc

import numpy as np
import torch

from torch.utils.data import DataLoader

from gnn_siamese.config import load_config
from gnn_siamese.builders import (
    build_dataset_bundle,
    build_model,
)
from gnn_siamese.data import collate_mut_wt_pairs
from gnn_siamese.training import load_checkpoint
from gnn_siamese.training.a9_contract import (
    validate_a9_run_acceptance,
)


print("=" * 80)
print("MODEL B A9 — FAST-B5 REPRESENTATIONS + OFFICIAL ACCEPTANCE")
print("=" * 80)

RUN_PRODUCTIVE_QUEUE = False
assert RUN_PRODUCTIVE_QUEUE is False

assert FAST_B4_ALL_PASS is True


# =============================================================================
# 1. FROZEN RUN MAP
# =============================================================================

MODEL_B_ARCH_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
    / "model_b_graph_level_relational"
)

RUNS = {
    11: (
        MODEL_B_ARCH_ROOT
        / "run_20260914T143850.128514Z-e382b31e"
    ),

    23: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T095945.494670Z-9a96c769"
    ),

    37: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T114757.375432Z-cf5a1245"
    ),

    41: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T103715.331004Z-a30da1be"
    ),

    53: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T111311.350252Z-628b5469"
    ),
}

PRODUCTIVE_SEEDS = (
    11,
    23,
    37,
    41,
    53,
)

SEEDS_TO_ACCEPT = (
    23,
    37,
    41,
    53,
)


# =============================================================================
# 2. HELPERS
# =============================================================================

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):

    payload = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    assert isinstance(payload, dict)

    return payload


# =============================================================================
# 3. SCIENTIFIC CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()

assert head == EXPECTED_SHA
assert status == ""

print("SCIENTIFIC_COMMIT =", head)
print("SCIENTIFIC_CHECKOUT = PASS")


# =============================================================================
# 4. DEVICE
# =============================================================================

assert torch.cuda.is_available()

DEVICE = torch.device("cuda")

print(
    "DEVICE =",
    DEVICE,
)

print(
    "GPU =",
    torch.cuda.get_device_name(0),
)


# =============================================================================
# 5. PRESERVE + VERIFY B11 ACCEPTANCE
# =============================================================================

B11_ACCEPTANCE = (
    RUNS[11]
    / "a9_acceptance.json"
)

assert B11_ACCEPTANCE.is_file()

b11_acceptance_before = (
    B11_ACCEPTANCE.read_bytes()
)

b11_acceptance_sha_before = (
    sha256_file(
        B11_ACCEPTANCE
    )
)

b11_acceptance = load_json(
    B11_ACCEPTANCE
)

assert (
    b11_acceptance["status"]
    == "accepted"
)

assert (
    int(
        b11_acceptance["run_seed"]
    )
    == 11
)

assert (
    b11_acceptance[
        "automatic_rejection_checks"
    ]
    == "PASS"
)

assert (
    set(
        b11_acceptance[
            "representation_metrics"
        ]
    )
    == {
        "r_delta",
        "z_delta",
        "z_instance_pair",
    }
)


print()
print(
    "B11_ACCEPTANCE_SHA256 =",
    b11_acceptance_sha_before,
)

print(
    "B11_EXISTING_ACCEPTANCE = PASS"
)

print(
    "B11_WILL_NOT_BE_REWRITTEN = TRUE"
)


# =============================================================================
# 6. MAKE SURE THE FOUR NEW ACCEPTANCES DO NOT ALREADY EXIST
# =============================================================================

for seed in SEEDS_TO_ACCEPT:

    acceptance_path = (
        RUNS[seed]
        / "a9_acceptance.json"
    )

    assert not acceptance_path.exists(), (
        f"STOP: B{seed} already has "
        f"{acceptance_path}. "
        "Do not overwrite automatically."
    )


print(
    "NEW_ACCEPTANCE_TARGETS_ABSENT = PASS"
)


# =============================================================================
# 7. TEMPORARY REPRESENTATION DIRECTORY
#
# /content only — temporary.
# Nothing scientific is persisted here permanently.
# =============================================================================

TEMP_REP_ROOT = (
    LOCAL_ROOT
    / "a9_acceptance_inputs"
    / "fast_b5"
)

TEMP_REP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 8. EXTRACTION / ACCEPTANCE LOOP
# =============================================================================

acceptance_results = {}
representation_summaries = {}

reference_variant_ids = None


for seed in SEEDS_TO_ACCEPT:

    print()
    print("=" * 80)
    print(
        f"FAST-B5 — B{seed}"
    )
    print("=" * 80)

    run_dir = RUNS[seed]

    config_path = (
        run_dir
        / "config_resolved.yaml"
    )

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    best_path = (
        run_dir
        / "checkpoints"
        / "best.pt"
    )

    acceptance_path = (
        run_dir
        / "a9_acceptance.json"
    )


    # -------------------------------------------------------------------------
    # Run identity
    # -------------------------------------------------------------------------

    assert run_dir.is_dir()
    assert config_path.is_file()
    assert manifest_path.is_file()
    assert best_path.is_file()
    assert not acceptance_path.exists()

    manifest = load_json(
        manifest_path
    )

    config = load_config(
        config_path
    )

    assert (
        manifest["status"]
        == "completed"
    )

    assert (
        manifest["architecture"]
        == "model_b_graph_level_relational"
    )

    assert (
        int(
            manifest[
                "configuration"
            ]["seed"]
        )
        == seed
    )

    assert (
        int(
            config[
                "project"
            ]["seed"]
        )
        == seed
    )


    print(
        "RUN_IDENTITY = PASS"
    )


    # -------------------------------------------------------------------------
    # Rebuild exact biological dataset
    # -------------------------------------------------------------------------

    dataset_bundle = (
        build_dataset_bundle(
            config
        )
    )

    dataset = (
        dataset_bundle.dataset
    )

    assert (
        len(dataset.pairs)
        == 483
    )

    assert (
        len(
            dataset.native_wt_controls
        )
        == 1
    )


    print(
        "BIOLOGICAL_DATASET_SIZE =",
        len(dataset.pairs),
    )

    print(
        "DATASET_REBUILD = PASS"
    )


    # -------------------------------------------------------------------------
    # Build Model B from exact resolved config
    # -------------------------------------------------------------------------

    model = build_model(
        config,
        dataset,
    )

    assert (
        model.__class__.__name__
        == "ModelBGraphLevelRelationalContrastive"
    )

    model = model.to(
        DEVICE
    )


    print(
        "MODEL_CLASS =",
        model.__class__.__name__,
    )

    print(
        "MODEL_REBUILD = PASS"
    )


    # -------------------------------------------------------------------------
    # Load BEST checkpoint
    # -------------------------------------------------------------------------

    checkpoint = load_checkpoint(
        best_path,
        map_location="cpu",
    )

    assert (
        int(
            checkpoint["seed"]
        )
        == seed
    )

    assert (
        checkpoint[
            "split_fingerprint"
        ]
        == EXPECTED_SPLIT_FINGERPRINT
    )

    assert (
        checkpoint[
            "architecture"
        ]
        == "model_b_graph_level_relational"
    )


    load_result = (
        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ],
            strict=True,
        )
    )


    assert (
        list(
            load_result.missing_keys
        )
        == []
    )

    assert (
        list(
            load_result.unexpected_keys
        )
        == []
    )


    model.eval()


    print(
        "REPRESENTATION_CHECKPOINT = best.pt"
    )

    print(
        "REPRESENTATION_EPOCH =",
        checkpoint[
            "epoch_completed"
        ],
    )

    print(
        "MISSING_STATE_KEYS =",
        list(
            load_result.missing_keys
        ),
    )

    print(
        "UNEXPECTED_STATE_KEYS =",
        list(
            load_result.unexpected_keys
        ),
    )

    print(
        "STRICT_STATE_DICT_LOAD = PASS"
    )


    # -------------------------------------------------------------------------
    # Deterministic full-dataset loader
    #
    # No augmentation.
    # No shuffle.
    # All 483 biological variants.
    # -------------------------------------------------------------------------

    representation_loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_mut_wt_pairs,
    )


    assert (
        len(
            representation_loader
        )
        == 121
    )


    print(
        "NUM_VARIANTS =",
        len(dataset),
    )

    print(
        "NUM_BATCHES =",
        len(
            representation_loader
        ),
    )

    print(
        "SHUFFLE = False"
    )

    print(
        "AUGMENTATION = False"
    )

    print(
        "REPRESENTATION_LOADER = PASS"
    )


    # -------------------------------------------------------------------------
    # Extract exact Model B relational representations
    # -------------------------------------------------------------------------

    representation_chunks = {
        "r_delta": [],
        "z_delta": [],
        "z_instance_pair": [],
    }

    variant_ids = []


    with torch.inference_mode():

        for batch in representation_loader:

            variant_ids.extend(
                list(
                    batch.variant_ids
                )
            )

            batch = batch.to(
                DEVICE
            )

            output = (
                model.siamese_model(
                    graph_mut=(
                        batch.graph_mut
                    ),
                    graph_wt=(
                        batch.graph_wt
                    ),
                    allow_trainable_z_delta=True,
                )
            )


            assert output.r_delta is not None
            assert output.z_delta is not None
            assert output.z_instance_pair is not None


            representation_chunks[
                "r_delta"
            ].append(
                output.r_delta
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            representation_chunks[
                "z_delta"
            ].append(
                output.z_delta
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            representation_chunks[
                "z_instance_pair"
            ].append(
                output.z_instance_pair
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )


    representations = {
        name:
            np.concatenate(
                chunks,
                axis=0,
            )

        for name, chunks
        in representation_chunks.items()
    }


    # -------------------------------------------------------------------------
    # Exact extraction checks
    # -------------------------------------------------------------------------

    assert len(
        variant_ids
    ) == 483

    assert len(
        set(
            variant_ids
        )
    ) == 483


    if reference_variant_ids is None:

        reference_variant_ids = (
            tuple(
                variant_ids
            )
        )

    else:

        assert (
            tuple(
                variant_ids
            )
            == reference_variant_ids
        )


    assert (
        representations[
            "r_delta"
        ].shape
        == (
            483,
            640,
        )
    )

    assert (
        representations[
            "z_delta"
        ].shape
        == (
            483,
            128,
        )
    )

    assert (
        representations[
            "z_instance_pair"
        ].shape
        == (
            483,
            64,
        )
    )


    for name, matrix in representations.items():

        assert (
            matrix.dtype
            == np.float32
        )

        assert (
            np.isfinite(
                matrix
            ).all()
        )

        assert (
            np.unique(
                matrix,
                axis=0,
            ).shape[0]
            == 483
        )


        print()
        print(
            name
        )

        print(
            " shape =",
            matrix.shape,
        )

        print(
            " dtype =",
            matrix.dtype,
        )

        print(
            " finite =",
            bool(
                np.isfinite(
                    matrix
                ).all()
            ),
        )

        print(
            " unique_rows =",
            np.unique(
                matrix,
                axis=0,
            ).shape[0],
        )

        print(
            " mean =",
            float(
                matrix.mean()
            ),
        )

        print(
            " std =",
            float(
                matrix.std()
            ),
        )


    print()
    print(
        "VARIANT_IDS =",
        len(
            variant_ids
        ),
    )

    print(
        "UNIQUE_VARIANT_IDS =",
        len(
            set(
                variant_ids
            )
        ),
    )

    print(
        "REPRESENTATION_EXTRACTION = PASS"
    )

    print(
        "REPRESENTATION_BASIC_QC = PASS"
    )


    # -------------------------------------------------------------------------
    # In-memory official acceptance FIRST
    #
    # This is read-only and proves the representation package is acceptable
    # before we write a9_acceptance.json.
    # -------------------------------------------------------------------------

    in_memory_acceptance = (
        validate_a9_run_acceptance(
            run_dir,
            config,
            representations=(
                representations
            ),
        )
    )


    assert (
        in_memory_acceptance[
            "status"
        ]
        == "accepted"
    )

    assert (
        int(
            in_memory_acceptance[
                "run_seed"
            ]
        )
        == seed
    )

    assert (
        in_memory_acceptance[
            "architecture"
        ]
        == "model_b_graph_level_relational"
    )

    assert (
        in_memory_acceptance[
            "automatic_rejection_checks"
        ]
        == "PASS"
    )

    assert (
        set(
            in_memory_acceptance[
                "representation_metrics"
            ]
        )
        == {
            "r_delta",
            "z_delta",
            "z_instance_pair",
        }
    )


    print()
    print(
        "IN_MEMORY_A9_ACCEPTANCE = PASS"
    )


    # -------------------------------------------------------------------------
    # Temporary NPZ for the official CLI
    # -------------------------------------------------------------------------

    temp_npz = (
        TEMP_REP_ROOT
        / f"B{seed}_best_representations.npz"
    )


    np.savez(
        temp_npz,
        r_delta=(
            representations[
                "r_delta"
            ]
        ),
        z_delta=(
            representations[
                "z_delta"
            ]
        ),
        z_instance_pair=(
            representations[
                "z_instance_pair"
            ]
        ),
    )


    assert temp_npz.is_file()
    assert temp_npz.stat().st_size > 0


    temp_npz_sha = (
        sha256_file(
            temp_npz
        )
    )


    print(
        "TEMP_REPRESENTATIONS_NPZ =",
        temp_npz,
    )

    print(
        "TEMP_REPRESENTATIONS_SHA256 =",
        temp_npz_sha,
    )


    # -------------------------------------------------------------------------
    # Official repository CLI
    #
    # This is the operation that writes:
    #
    # run_dir/a9_acceptance.json
    # -------------------------------------------------------------------------

    command = [
        sys.executable,
        str(
            REPO
            / "scripts"
            / "a9_acceptance.py"
        ),
        "--run-dir",
        str(
            run_dir
        ),
        "--config",
        str(
            config_path
        ),
        "--pair-representations",
        str(
            temp_npz
        ),
    ]


    official = subprocess.run(
        command,
        cwd=REPO,
        text=True,
        capture_output=True,
    )


    print()
    print(
        "OFFICIAL_ACCEPTANCE_RETURN_CODE =",
        official.returncode,
    )


    if official.stdout:

        print()
        print(
            "OFFICIAL ACCEPTANCE OUTPUT:"
        )

        print(
            official.stdout
        )


    if official.stderr:

        print()
        print(
            "OFFICIAL ACCEPTANCE STDERR:"
        )

        print(
            official.stderr
        )


    assert (
        official.returncode
        == 0
    ), (
        f"B{seed}: official acceptance failed."
    )


    # -------------------------------------------------------------------------
    # Verify persistent official acceptance
    # -------------------------------------------------------------------------

    assert acceptance_path.is_file()
    assert acceptance_path.stat().st_size > 0


    persisted = load_json(
        acceptance_path
    )


    assert persisted == (
        in_memory_acceptance
    )

    assert (
        persisted["status"]
        == "accepted"
    )

    assert (
        int(
            persisted[
                "run_seed"
            ]
        )
        == seed
    )

    assert (
        persisted[
            "automatic_rejection_checks"
        ]
        == "PASS"
    )


    acceptance_sha = (
        sha256_file(
            acceptance_path
        )
    )


    print(
        "A9_ACCEPTANCE_PATH =",
        acceptance_path,
    )

    print(
        "A9_ACCEPTANCE_SHA256 =",
        acceptance_sha,
    )

    print(
        "PERSISTED_ACCEPTANCE_CONTENT = PASS"
    )


    # -------------------------------------------------------------------------
    # Remove temporary NPZ
    # -------------------------------------------------------------------------

    temp_npz.unlink()

    assert not temp_npz.exists()

    print(
        "TEMP_NPZ_REMOVED = PASS"
    )


    # -------------------------------------------------------------------------
    # Store summary
    # -------------------------------------------------------------------------

    representation_summaries[
        seed
    ] = {
        name: {
            "shape":
                list(
                    matrix.shape
                ),

            "mean":
                float(
                    matrix.mean()
                ),

            "std":
                float(
                    matrix.std()
                ),

            "unique_rows":
                int(
                    np.unique(
                        matrix,
                        axis=0,
                    ).shape[0]
                ),
        }

        for name, matrix
        in representations.items()
    }


    acceptance_results[
        seed
    ] = {
        "status":
            persisted[
                "status"
            ],

        "acceptance_path":
            str(
                acceptance_path
            ),

        "sha256":
            acceptance_sha,

        "best_epoch":
            int(
                checkpoint[
                    "epoch_completed"
                ]
            ),
    }


    print()
    print(
        f"B{seed}_OFFICIAL_A9_ACCEPTANCE = PASS"
    )

    print(
        f"B{seed}_A9_STATUS = CLOSED_ACCEPTED"
    )


    # -------------------------------------------------------------------------
    # Free GPU / RAM before next seed
    # -------------------------------------------------------------------------

    del output
    del representations
    del representation_chunks
    del representation_loader
    del model
    del checkpoint
    del dataset
    del dataset_bundle

    gc.collect()

    torch.cuda.empty_cache()


# =============================================================================
# 9. VERIFY B11 WAS NOT MODIFIED
# =============================================================================

assert (
    B11_ACCEPTANCE.read_bytes()
    == b11_acceptance_before
)

assert (
    sha256_file(
        B11_ACCEPTANCE
    )
    == b11_acceptance_sha_before
)


print()
print(
    "B11_ACCEPTANCE_UNCHANGED = PASS"
)


# =============================================================================
# 10. VERIFY ALL FIVE OFFICIAL ACCEPTANCES
# =============================================================================

all_acceptances = {}


for seed in PRODUCTIVE_SEEDS:

    acceptance_path = (
        RUNS[seed]
        / "a9_acceptance.json"
    )

    assert acceptance_path.is_file()

    payload = load_json(
        acceptance_path
    )

    assert (
        payload["status"]
        == "accepted"
    )

    assert (
        int(
            payload[
                "run_seed"
            ]
        )
        == seed
    )

    assert (
        payload[
            "automatic_rejection_checks"
        ]
        == "PASS"
    )

    assert (
        set(
            payload[
                "representation_metrics"
            ]
        )
        == {
            "r_delta",
            "z_delta",
            "z_instance_pair",
        }
    )


    all_acceptances[
        seed
    ] = {
        "path":
            str(
                acceptance_path
            ),

        "sha256":
            sha256_file(
                acceptance_path
            ),
    }


    print(
        f"B{seed}_FINAL_ACCEPTANCE = PASS"
    )


# =============================================================================
# 11. TEMPORARY DIRECTORY MUST CONTAIN NO NPZ FILES
# =============================================================================

leftover_npz = list(
    TEMP_REP_ROOT.glob(
        "*.npz"
    )
)

print()
print(
    "LEFTOVER_TEMP_NPZ =",
    [
        str(x)
        for x in leftover_npz
    ],
)

assert leftover_npz == []

print(
    "TEMP_REPRESENTATION_CLEANUP = PASS"
)


# =============================================================================
# FINAL
# =============================================================================

FAST_B5_ALL_ACCEPTED = True


print()
print("=" * 80)
print("FAST-B5 — FINAL")
print("=" * 80)


for seed in PRODUCTIVE_SEEDS:

    print()
    print(
        f"B{seed} = CLOSED_ACCEPTED"
    )

    print(
        "  acceptance =",
        all_acceptances[
            seed
        ]["path"],
    )

    print(
        "  sha256 =",
        all_acceptances[
            seed
        ]["sha256"],
    )


print()
print(
    "FIVE_SEED_REPRESENTATION_EXTRACTION = PASS"
)

print(
    "FIVE_OFFICIAL_A9_ACCEPTANCES = PASS"
)

print(
    "B11_ACCEPTANCE_UNCHANGED = PASS"
)

print(
    "TEMP_REPRESENTATION_CLEANUP = PASS"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_CHECKPOINT_MODIFIED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print(
    "MODEL_B_A9_ACCEPTANCE = CLOSED"
)

print(
    "READY_FOR_FAST_B6_AGGREGATION = YES"
)

print("=" * 80)

MODEL B A9 — FAST-B5 REPRESENTATIONS + OFFICIAL ACCEPTANCE
SCIENTIFIC_COMMIT = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
SCIENTIFIC_CHECKOUT = PASS
DEVICE = cuda
GPU = Tesla T4

B11_ACCEPTANCE_SHA256 = c9285e995fa069ec7fe9622909918f4fac024dda8f3eea7ac6b137c08f2641e2
B11_EXISTING_ACCEPTANCE = PASS
B11_WILL_NOT_BE_REWRITTEN = TRUE
NEW_ACCEPTANCE_TARGETS_ABSENT = PASS

FAST-B5 — B23
RUN_IDENTITY = PASS
BIOLOGICAL_DATASET_SIZE = 483
DATASET_REBUILD = PASS
MODEL_CLASS = ModelBGraphLevelRelationalContrastive
MODEL_REBUILD = PASS
REPRESENTATION_CHECKPOINT = best.pt
REPRESENTATION_EPOCH = 38
MISSING_STATE_KEYS = []
UNEXPECTED_STATE_KEYS = []
STRICT_STATE_DICT_LOAD = PASS
NUM_VARIANTS = 483
NUM_BATCHES = 121
SHUFFLE = False
AUGMENTATION = False
REPRESENTATION_LOADER = PASS

r_delta
 shape = (483, 640)
 dtype = float32
 finite = True
 unique_rows = 483
 mean = 0.2330833524465561
 std = 1.2129902839660645

z_delta
 shape = (483, 128)
 dtype = float32
 finite = True
 unique_rows = 483
 mean = -0.05

In [ ]:
# =============================================================================
# MODEL B A9
# 0.3-CPU — RESTORE POST-TRAINING ANALYSIS ENVIRONMENT
#
# PURPOSE
# -------
# Restore a CPU-compatible software environment for READ-ONLY post-training
# analyses when the exact B11 CUDA runtime is temporarily unavailable.
#
# THIS IS NOT THE EXACT PRODUCTIVE B11 ENVIRONMENT.
#
# B11 reference:
#   Python  = exact match required
#   PyTorch = same upstream/base version required (local build suffix may differ)
#   PyG     = exact match required
#
# Allowed example:
#   B11     torch 2.11.0+cu128
#   current torch 2.11.0+cpu
#
# NOT ALLOWED:
#   B11     torch 2.11.0+cu128
#   current torch 2.10.x / 2.12.x / etc.
#
# SAFE SCOPE
# ----------
# YES:
#   - FAST-B6
#   - checkpoint loading on CPU
#   - manifest / acceptance inspection
#   - multiseed aggregation
#   - deterministic post-training analysis
#
# NO:
#   - productive training
#   - resume training
#   - modifying checkpoints
#   - modifying acceptance files
#   - pretending this is an exact B11 environment
#
# IMPORTANT:
#   - NEVER remove torch_geometric from sys.modules.
#   - NEVER reload torch_geometric.
#   - NEVER manipulate PyTorch DataPipe registries.
# =============================================================================

from pathlib import Path

import os
import json
import platform
import subprocess
import sys
import importlib
import importlib.metadata

import torch


print("=" * 80)
print("MODEL B A9 — 0.3-CPU POST-TRAINING ANALYSIS ENVIRONMENT")
print("=" * 80)


# =============================================================================
# 0. SAFETY
# =============================================================================

assert RUN_PRODUCTIVE_QUEUE is False, (
    "STOP: RUN_PRODUCTIVE_QUEUE must remain False."
)

print("RUN_PRODUCTIVE_QUEUE = False")


# =============================================================================
# 1. LOAD FROZEN B11 REFERENCE ENVIRONMENT
# =============================================================================

assert B11_MANIFEST.is_file(), (
    f"B11 manifest missing: {B11_MANIFEST}"
)

manifest = json.loads(
    B11_MANIFEST.read_text(
        encoding="utf-8"
    )
)


dependency_blocks = []


def find_dependency_block(obj, path="root"):

    if isinstance(obj, dict):

        if {
            "python",
            "pytorch",
            "torch_geometric",
        }.issubset(obj.keys()):

            dependency_blocks.append(
                (
                    path,
                    obj,
                )
            )

        for key, value in obj.items():

            find_dependency_block(
                value,
                f"{path}.{key}",
            )

    elif isinstance(obj, list):

        for index, value in enumerate(obj):

            find_dependency_block(
                value,
                f"{path}[{index}]",
            )


find_dependency_block(
    manifest
)

assert dependency_blocks, (
    "B11 manifest does not contain "
    "python/pytorch/torch_geometric metadata."
)

dependency_path, recorded = (
    dependency_blocks[0]
)


RECORDED_PYTHON = str(
    recorded["python"]
)

RECORDED_TORCH = str(
    recorded["pytorch"]
)

RECORDED_PYG = str(
    recorded["torch_geometric"]
)


print()
print(
    "REFERENCE_DEPENDENCY_BLOCK =",
    dependency_path,
)

print(
    "B11 Python =",
    RECORDED_PYTHON,
)

print(
    "B11 PyTorch =",
    RECORDED_TORCH,
)

print(
    "B11 PyG =",
    RECORDED_PYG,
)


# =============================================================================
# 2. CURRENT RUNTIME
# =============================================================================

CURRENT_PYTHON = (
    platform.python_version()
)

CURRENT_TORCH = (
    str(torch.__version__)
)


print()
print(
    "Current Python =",
    CURRENT_PYTHON,
)

print(
    "Current PyTorch =",
    CURRENT_TORCH,
)

print(
    "Current torch CUDA =",
    torch.version.cuda,
)

print(
    "CUDA available =",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():

    print(
        "GPU =",
        torch.cuda.get_device_name(0),
    )

else:

    print(
        "ANALYSIS_DEVICE = CPU"
    )


# =============================================================================
# 3. PYTHON MUST STILL MATCH B11 EXACTLY
# =============================================================================

assert (
    CURRENT_PYTHON
    == RECORDED_PYTHON
), (
    "STOP: Python differs from the B11 environment. "
    f"B11={RECORDED_PYTHON}; current={CURRENT_PYTHON}"
)


print()
print(
    "PYTHON_EXACT_MATCH = PASS"
)


# =============================================================================
# 4. PYTORCH BASE VERSION MUST MATCH B11
#
# Example:
#   2.11.0+cu128 -> base = 2.11.0
#   2.11.0+cpu   -> base = 2.11.0
#
# This deliberately DOES NOT claim exact environment equivalence.
# =============================================================================

def torch_base_version(version):

    return (
        str(version)
        .split("+", 1)[0]
        .strip()
    )


RECORDED_TORCH_BASE = (
    torch_base_version(
        RECORDED_TORCH
    )
)

CURRENT_TORCH_BASE = (
    torch_base_version(
        CURRENT_TORCH
    )
)


print(
    "B11 PyTorch base =",
    RECORDED_TORCH_BASE,
)

print(
    "Current PyTorch base =",
    CURRENT_TORCH_BASE,
)


assert (
    CURRENT_TORCH_BASE
    == RECORDED_TORCH_BASE
), (
    "STOP: PyTorch upstream/base version differs from B11. "
    f"B11={RECORDED_TORCH}; current={CURRENT_TORCH}. "
    "Do not continue with post-training analysis."
)


print(
    "PYTORCH_BASE_VERSION_MATCH = PASS"
)


# Explicitly document whether build is exact.

TORCH_EXACT_BUILD_MATCH = (
    CURRENT_TORCH
    == RECORDED_TORCH
)


print(
    "PYTORCH_EXACT_BUILD_MATCH =",
    TORCH_EXACT_BUILD_MATCH,
)


if not TORCH_EXACT_BUILD_MATCH:

    print(
        "PYTORCH_BUILD_DIFFERENCE = DOCUMENTED"
    )

    print(
        "REFERENCE_BUILD =",
        RECORDED_TORCH,
    )

    print(
        "ANALYSIS_BUILD =",
        CURRENT_TORCH,
    )


# =============================================================================
# 5. THIS CELL IS POST-TRAINING ONLY
# =============================================================================

POST_TRAINING_ANALYSIS_ONLY = True

assert POST_TRAINING_ANALYSIS_ONLY is True
assert RUN_PRODUCTIVE_QUEUE is False


print()
print(
    "POST_TRAINING_ANALYSIS_ONLY = TRUE"
)


# =============================================================================
# 6. DETECT CURRENT PYG PROCESS STATE
# =============================================================================

PYG_ALREADY_LOADED = (
    "torch_geometric"
    in sys.modules
)


print()
print(
    "PYG_ALREADY_LOADED =",
    PYG_ALREADY_LOADED,
)


# =============================================================================
# 7A. IF PYG ALREADY LOADED, DO NOT RELOAD IT
# =============================================================================

if PYG_ALREADY_LOADED:

    torch_geometric = (
        sys.modules[
            "torch_geometric"
        ]
    )

    CURRENT_PYG = str(
        torch_geometric.__version__
    )


    print(
        "Current PyG =",
        CURRENT_PYG,
    )


    assert (
        CURRENT_PYG
        == RECORDED_PYG
    ), (
        "STOP: another PyG version is already loaded. "
        f"B11={RECORDED_PYG}; current={CURRENT_PYG}. "
        "Restart the Colab runtime before continuing."
    )


    print(
        "PYG_REUSED_FROM_CURRENT_PROCESS = YES"
    )


# =============================================================================
# 7B. PYG NOT LOADED — VERIFY CLEAN PROCESS STATE
# =============================================================================

else:

    from torch.utils.data.datapipes.datapipe import (
        IterDataPipe,
        MapDataPipe,
    )


    iter_functions = getattr(
        IterDataPipe,
        "functions",
        {},
    )

    map_functions = getattr(
        MapDataPipe,
        "functions",
        {},
    )


    BATCH_GRAPHS_IN_ITER = (
        "batch_graphs"
        in iter_functions
    )

    BATCH_GRAPHS_IN_MAP = (
        "batch_graphs"
        in map_functions
    )


    STALE_PYG_DATAPIPE_REGISTRY = (
        BATCH_GRAPHS_IN_ITER
        or
        BATCH_GRAPHS_IN_MAP
    )


    print()
    print(
        "BATCH_GRAPHS_IN_ITER_REGISTRY =",
        BATCH_GRAPHS_IN_ITER,
    )

    print(
        "BATCH_GRAPHS_IN_MAP_REGISTRY =",
        BATCH_GRAPHS_IN_MAP,
    )

    print(
        "STALE_PYG_DATAPIPE_REGISTRY =",
        STALE_PYG_DATAPIPE_REGISTRY,
    )


    assert not STALE_PYG_DATAPIPE_REGISTRY, (
        "STOP: stale PyG DataPipe registration detected while "
        "torch_geometric is not loaded. "
        "Restart the Colab runtime. "
        "Do NOT remove registry entries manually."
    )


    print(
        "PYG_PROCESS_STATE_CLEAN = PASS"
    )


    # =========================================================================
    # 8. INSPECT INSTALLED PYG DISTRIBUTION WITHOUT IMPORTING IT
    # =========================================================================

    try:

        INSTALLED_PYG = (
            importlib.metadata.version(
                "torch-geometric"
            )
        )

    except (
        importlib.metadata.PackageNotFoundError
    ):

        INSTALLED_PYG = None


    print()
    print(
        "INSTALLED_PYG_DISTRIBUTION =",
        INSTALLED_PYG,
    )

    print(
        "TARGET_PYG_VERSION =",
        RECORDED_PYG,
    )


    # =========================================================================
    # 9. INSTALL EXACT B11 PYG VERSION IF NECESSARY
    #
    # --no-deps is intentional:
    # do not allow pip to replace the current PyTorch CPU build.
    # =========================================================================

    if (
        INSTALLED_PYG
        != RECORDED_PYG
    ):

        print(
            "PYG_INSTALL_REQUIRED = YES"
        )


        command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-deps",
            "--force-reinstall",
            f"torch-geometric=={RECORDED_PYG}",
        ]


        print()
        print("COMMAND:")
        print(
            " ".join(
                command
            )
        )


        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )


        print()
        print(
            "PIP_RETURN_CODE =",
            result.returncode,
        )


        if result.stdout:
            print(
                result.stdout
            )

        if result.stderr:
            print(
                result.stderr
            )


        assert (
            result.returncode
            == 0
        ), (
            "STOP: exact PyG installation failed."
        )


        INSTALLED_PYG = (
            importlib.metadata.version(
                "torch-geometric"
            )
        )


        assert (
            INSTALLED_PYG
            == RECORDED_PYG
        ), (
            "STOP: installed PyG distribution "
            "does not match B11."
        )


    else:

        print(
            "PYG_INSTALL_REQUIRED = NO"
        )


    # =========================================================================
    # 10. FIRST AND ONLY PYG IMPORT
    # =========================================================================

    import torch_geometric


    CURRENT_PYG = str(
        torch_geometric.__version__
    )


    print()
    print(
        "Current PyG =",
        CURRENT_PYG,
    )


    assert (
        CURRENT_PYG
        == RECORDED_PYG
    ), (
        "STOP: imported PyG does not match B11."
    )


# =============================================================================
# 11. VERIFY PYTORCH WAS NOT REPLACED DURING PYG INSTALLATION
# =============================================================================

TORCH_AFTER_PYG = str(
    torch.__version__
)

TORCH_AFTER_PYG_BASE = (
    torch_base_version(
        TORCH_AFTER_PYG
    )
)


assert (
    TORCH_AFTER_PYG
    == CURRENT_TORCH
), (
    "STOP: PyTorch changed during PyG recovery."
)

assert (
    TORCH_AFTER_PYG_BASE
    == RECORDED_TORCH_BASE
)


print()
print(
    "PYG_EXACT_MATCH = PASS"
)

print(
    "PYTORCH_UNCHANGED_DURING_RECOVERY = PASS"
)


# =============================================================================
# 12. MINIMAL PYG CPU SANITY
# =============================================================================

from torch_geometric.data import (
    Data,
    Batch,
)


x = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
    ],
    dtype=torch.float32,
)


edge_index = torch.tensor(
    [
        [0, 1],
        [1, 0],
    ],
    dtype=torch.long,
)


test_graph = Data(
    x=x,
    edge_index=edge_index,
)


test_batch = (
    Batch.from_data_list(
        [
            test_graph,
            test_graph,
        ]
    )
)


assert (
    test_batch.num_graphs
    == 2
)

assert (
    int(
        test_batch.x.shape[0]
    )
    == 4
)


print(
    "PYG_BASIC_CPU_SANITY = PASS"
)


# =============================================================================
# 13. NNCONV IMPORT SANITY
#
# This is specifically relevant because the project encoder imports NNConv.
# =============================================================================

from torch_geometric.nn import (
    NNConv,
)


assert NNConv is not None


print(
    "PYG_NNCONV_IMPORT = PASS"
)


# =============================================================================
# 14. RESTORE PROJECT IMPORT PATH
# =============================================================================

assert Path(REPO).is_dir(), (
    f"REPO missing: {REPO}"
)

assert Path(SRC).is_dir(), (
    f"SRC missing: {SRC}"
)


os.chdir(
    REPO
)


for candidate in (
    str(REPO),
    str(SRC),
):

    while candidate in sys.path:

        sys.path.remove(
            candidate
        )


sys.path.insert(
    0,
    str(REPO),
)

sys.path.insert(
    0,
    str(SRC),
)


# Purge project package only.
#
# SAFE:
#   gnn_siamese
#
# DO NOT purge:
#   torch
#   torch_geometric

for name in list(
    sys.modules
):

    if (
        name == "gnn_siamese"
        or
        name.startswith(
            "gnn_siamese."
        )
    ):

        del sys.modules[
            name
        ]


importlib.invalidate_caches()


# =============================================================================
# 15. PROJECT PACKAGE IMPORT
# =============================================================================

import gnn_siamese


expected_init = (
    Path(SRC)
    / "gnn_siamese"
    / "__init__.py"
).resolve()


actual_init = (
    Path(
        gnn_siamese.__file__
    ).resolve()
)


print()
print(
    "gnn_siamese file =",
    actual_init,
)


assert (
    actual_init
    == expected_init
), (
    "STOP: gnn_siamese imported from unexpected location."
)


print(
    "PROJECT_IMPORT = PASS"
)


# =============================================================================
# 16. IMPORT EXACT FUNCTIONS REQUIRED BY FAST-B6
# =============================================================================

from gnn_siamese.training import (
    load_checkpoint,
)

from gnn_siamese.utils.atomic_io import (
    atomic_write_text,
)


assert callable(
    load_checkpoint
)

assert callable(
    atomic_write_text
)


print(
    "FAST_B6_PROJECT_IMPORTS = PASS"
)


# =============================================================================
# 17. OPTIONAL READ-ONLY B11 CHECKPOINT LOAD SANITY
#
# This verifies that the CPU environment can deserialize the productive
# checkpoint using map_location='cpu'.
#
# NO model inference.
# NO training.
# NO modification.
# =============================================================================

B11_RUN_DIR = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
    / "model_b_graph_level_relational"
    / "run_20260914T143850.128514Z-e382b31e"
)


B11_BEST_PT = (
    B11_RUN_DIR
    / "checkpoints"
    / "best.pt"
)


assert B11_BEST_PT.is_file(), (
    f"B11 best.pt missing: {B11_BEST_PT}"
)


checkpoint = load_checkpoint(
    B11_BEST_PT,
    map_location="cpu",
)


assert isinstance(
    checkpoint,
    dict,
)


assert int(
    checkpoint["seed"]
) == 11


print()
print(
    "B11_BEST_PT_CPU_LOAD = PASS"
)

print(
    "B11_CHECKPOINT_SEED =",
    checkpoint["seed"],
)

print(
    "B11_CHECKPOINT_EPOCH =",
    checkpoint.get(
        "epoch_completed",
        "NOT_RECORDED",
    ),
)


# Free the local reference.
del checkpoint


# =============================================================================
# 18. FINAL CLASSIFICATION
# =============================================================================

EXACT_B11_ENVIRONMENT = (
    CURRENT_PYTHON == RECORDED_PYTHON
    and
    CURRENT_TORCH == RECORDED_TORCH
    and
    CURRENT_PYG == RECORDED_PYG
)


CPU_POST_TRAINING_ENVIRONMENT = (
    CURRENT_PYTHON == RECORDED_PYTHON
    and
    CURRENT_TORCH_BASE == RECORDED_TORCH_BASE
    and
    CURRENT_PYG == RECORDED_PYG
)


assert CPU_POST_TRAINING_ENVIRONMENT is True


print()
print("=" * 80)
print("CPU ANALYSIS ENVIRONMENT — FINAL")
print("=" * 80)

print(
    "REFERENCE_PYTHON =",
    RECORDED_PYTHON,
)

print(
    "ANALYSIS_PYTHON =",
    CURRENT_PYTHON,
)

print()

print(
    "REFERENCE_PYTORCH =",
    RECORDED_TORCH,
)

print(
    "ANALYSIS_PYTORCH =",
    CURRENT_TORCH,
)

print()

print(
    "REFERENCE_PYG =",
    RECORDED_PYG,
)

print(
    "ANALYSIS_PYG =",
    CURRENT_PYG,
)

print()

print(
    "EXACT_B11_ENVIRONMENT =",
    EXACT_B11_ENVIRONMENT,
)

print(
    "CPU_POST_TRAINING_ENVIRONMENT = PASS"
)

print(
    "PYTORCH_BASE_VERSION_MATCH = PASS"
)

print(
    "PYG_EXACT_MATCH = PASS"
)

print(
    "PYG_BASIC_CPU_SANITY = PASS"
)

print(
    "PYG_NNCONV_IMPORT = PASS"
)

print(
    "PROJECT_IMPORT = PASS"
)

print(
    "FAST_B6_PROJECT_IMPORTS = PASS"
)

print(
    "B11_BEST_PT_CPU_LOAD = PASS"
)

print()

print(
    "ANALYSIS_DEVICE = cpu"
)

print(
    "POST_TRAINING_ANALYSIS_ONLY = TRUE"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_RESUME_EXECUTED = TRUE"
)

print(
    "NO_NEW_RUN_CREATED = TRUE"
)

print(
    "NO_CHECKPOINT_MODIFIED = TRUE"
)

print(
    "NO_ACCEPTANCE_MODIFIED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print()

print(
    "READY_FOR_FAST_B6 = YES"
)

print("=" * 80)

MODEL B A9 — 0.3-CPU POST-TRAINING ANALYSIS ENVIRONMENT
RUN_PRODUCTIVE_QUEUE = False

REFERENCE_DEPENDENCY_BLOCK = root.environment.dependencies
B11 Python = 3.13.15
B11 PyTorch = 2.11.0+cu128
B11 PyG = 2.8.0.post1

Current Python = 3.13.15
Current PyTorch = 2.11.0+cpu
Current torch CUDA = None
CUDA available = False
ANALYSIS_DEVICE = CPU

PYTHON_EXACT_MATCH = PASS
B11 PyTorch base = 2.11.0
Current PyTorch base = 2.11.0
PYTORCH_BASE_VERSION_MATCH = PASS
PYTORCH_EXACT_BUILD_MATCH = False
PYTORCH_BUILD_DIFFERENCE = DOCUMENTED
REFERENCE_BUILD = 2.11.0+cu128
ANALYSIS_BUILD = 2.11.0+cpu

POST_TRAINING_ANALYSIS_ONLY = TRUE

PYG_ALREADY_LOADED = True
Current PyG = 2.8.0.post1
PYG_REUSED_FROM_CURRENT_PROCESS = YES

PYG_EXACT_MATCH = PASS
PYTORCH_UNCHANGED_DURING_RECOVERY = PASS
PYG_BASIC_CPU_SANITY = PASS
PYG_NNCONV_IMPORT = PASS

gnn_siamese file = /content/model_b_workspace/repo/src/gnn_siamese/__init__.py
PROJECT_IMPORT = PASS
FAST_B6_PROJECT_IMPORTS = PASS

B11_BEST_PT_CPU_LOAD = PASS
B11_

In [ ]:
# =============================================================================
# MODEL B A9 — FAST-B6
# FIVE-SEED AGGREGATION
#
# READ-ONLY with respect to productive runs.
#
# INPUT:
#   accepted B11 / B23 / B37 / B41 / B53
#
# OUTPUT:
#   isolated multiseed aggregation files
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# NO ACCEPTANCE MODIFICATION
# =============================================================================

from pathlib import Path
import csv
import hashlib
import io
import json
import math
import statistics
import subprocess

from gnn_siamese.training import load_checkpoint
from gnn_siamese.utils.atomic_io import atomic_write_text


print("=" * 80)
print("MODEL B A9 — FAST-B6 FIVE-SEED AGGREGATION")
print("=" * 80)

RUN_PRODUCTIVE_QUEUE = False
assert RUN_PRODUCTIVE_QUEUE is False


# =============================================================================
# 1. FROZEN SCIENTIFIC IDENTITY
# =============================================================================

EXPECTED_SHA = (
    "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"
)

PRODUCTIVE_SEEDS = (
    11,
    23,
    37,
    41,
    53,
)

MODEL_B_ARCHITECTURE = (
    "model_b_graph_level_relational"
)


# =============================================================================
# 2. PRODUCTIVE ROOT + EXACT RUN MAP
# =============================================================================

MODEL_B_A9_ROOT = (
    Path(DRIVE_PROJECT_BASE)
    / "model_b"
    / "runs"
    / "model_b_a9"
)

MODEL_B_ARCH_ROOT = (
    MODEL_B_A9_ROOT
    / MODEL_B_ARCHITECTURE
)

RUNS = {
    11: (
        MODEL_B_ARCH_ROOT
        / "run_20260914T143850.128514Z-e382b31e"
    ),

    23: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T095945.494670Z-9a96c769"
    ),

    37: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T114757.375432Z-cf5a1245"
    ),

    41: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T103715.331004Z-a30da1be"
    ),

    53: (
        MODEL_B_ARCH_ROOT
        / "run_20260915T111311.350252Z-628b5469"
    ),
}


# =============================================================================
# 3. FROZEN ACCEPTANCE HASHES
# =============================================================================

EXPECTED_ACCEPTANCE_SHA256 = {

    11:
        "c9285e995fa069ec7fe9622909918f4f"
        "ac024dda8f3eea7ac6b137c08f2641e2",

    23:
        "d47179e2d0746851de4eed4f48e19074"
        "0147006b37140f11dedb1129bca233a6",

    37:
        "29ab42870e53bf2ca584ca70bd32d069c"
        "caf964f122dcfb7a8fbf6b6015edc95",

    41:
        "643feb4e191f042f69d9a42e01a89fa"
        "426a2de5153c31ac505159db0fc5964c9",

    53:
        "ef95e25e90b5a0458aff4a0821bd2084"
        "d006529aacadb5af33be1fcb36068fd9",
}


# =============================================================================
# 4. HELPERS
# =============================================================================

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):

    result = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    assert isinstance(
        result,
        dict,
    )

    return result


def finite_number(value):

    return (
        isinstance(value, (int, float))
        and
        not isinstance(value, bool)
        and
        math.isfinite(float(value))
    )


def summarize(values):

    values = [
        float(x)
        for x in values
    ]

    assert len(values) == 5
    assert all(
        math.isfinite(x)
        for x in values
    )

    return {
        "n":
            len(values),

        "mean":
            statistics.mean(values),

        "sd":
            statistics.stdev(values),

        "median":
            statistics.median(values),

        "min":
            min(values),

        "max":
            max(values),
    }


# =============================================================================
# 5. SCIENTIFIC CHECKOUT
# =============================================================================

head = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

status = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short",
    ],
    text=True,
).strip()

assert head == EXPECTED_SHA
assert status == ""

print(
    "SCIENTIFIC_COMMIT =",
    head,
)

print(
    "SCIENTIFIC_CHECKOUT = PASS"
)


# =============================================================================
# 6. READ AND VERIFY ALL FIVE ACCEPTED RUNS
# =============================================================================

records = []


for seed in PRODUCTIVE_SEEDS:

    print()
    print("-" * 80)
    print(f"AGGREGATION INPUT — B{seed}")
    print("-" * 80)

    run_dir = RUNS[seed]

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    acceptance_path = (
        run_dir
        / "a9_acceptance.json"
    )

    best_path = (
        run_dir
        / "checkpoints"
        / "best.pt"
    )

    last_path = (
        run_dir
        / "checkpoints"
        / "last.pt"
    )


    for path in (
        manifest_path,
        acceptance_path,
        best_path,
        last_path,
    ):

        assert path.is_file(), (
            f"B{seed}: missing {path}"
        )

        assert (
            path.stat().st_size > 0
        )


    # -------------------------------------------------------------------------
    # Acceptance identity
    # -------------------------------------------------------------------------

    actual_acceptance_sha = (
        sha256_file(
            acceptance_path
        )
    )

    assert (
        actual_acceptance_sha
        ==
        EXPECTED_ACCEPTANCE_SHA256[
            seed
        ]
    ), (
        f"B{seed}: acceptance SHA256 changed."
    )


    acceptance = load_json(
        acceptance_path
    )

    assert (
        acceptance["status"]
        == "accepted"
    )

    assert (
        int(
            acceptance[
                "run_seed"
            ]
        )
        == seed
    )

    assert (
        acceptance[
            "architecture"
        ]
        == MODEL_B_ARCHITECTURE
    )

    assert (
        acceptance[
            "automatic_rejection_checks"
        ]
        == "PASS"
    )


    representation_metrics = (
        acceptance[
            "representation_metrics"
        ]
    )

    assert (
        set(
            representation_metrics
        )
        == {
            "r_delta",
            "z_delta",
            "z_instance_pair",
        }
    )


    # -------------------------------------------------------------------------
    # Manifest identity
    # -------------------------------------------------------------------------

    manifest = load_json(
        manifest_path
    )

    assert (
        manifest["status"]
        == "completed"
    )

    assert (
        manifest["architecture"]
        == MODEL_B_ARCHITECTURE
    )

    assert (
        int(
            manifest[
                "configuration"
            ]["seed"]
        )
        == seed
    )

    assert (
        int(
            manifest[
                "configuration"
            ][
                "seed_bundle"
            ]["split"]
        )
        == 42
    )


    training = (
        manifest[
            "training"
        ]
    )


    epochs_completed = int(
        training[
            "epochs_completed"
        ]
    )

    stopped_early = bool(
        training[
            "stopped_early"
        ]
    )

    best_validation_loss = float(
        training[
            "early_stopping"
        ][
            "best_metric"
        ]
    )


    assert epochs_completed >= 1
    assert stopped_early is True
    assert math.isfinite(
        best_validation_loss
    )


    # -------------------------------------------------------------------------
    # Checkpoints
    # -------------------------------------------------------------------------

    best = load_checkpoint(
        best_path,
        map_location="cpu",
    )

    last = load_checkpoint(
        last_path,
        map_location="cpu",
    )


    best_epoch = int(
        best[
            "epoch_completed"
        ]
    )

    last_epoch = int(
        last[
            "epoch_completed"
        ]
    )


    assert (
        last_epoch
        == epochs_completed
    )

    assert (
        best_epoch
        <= last_epoch
    )

    assert (
        int(
            best["seed"]
        )
        == seed
    )

    assert (
        int(
            last["seed"]
        )
        == seed
    )


    # -------------------------------------------------------------------------
    # Representation metrics
    # -------------------------------------------------------------------------

    representation_record = {}


    for name in (
        "r_delta",
        "z_delta",
        "z_instance_pair",
    ):

        metrics = (
            representation_metrics[
                name
            ]
        )

        assert (
            metrics[
                "finite"
            ]
            is True
        )

        assert (
            metrics[
                "exact_total_collapse"
            ]
            is False
        )

        assert (
            int(
                metrics[
                    "exact_unique_rows"
                ]
            )
            == 483
        )

        assert finite_number(
            metrics[
                "effective_rank"
            ]
        )

        assert finite_number(
            metrics[
                "pc1_variance_fraction"
            ]
        )


        representation_record[
            f"{name}_effective_rank"
        ] = float(
            metrics[
                "effective_rank"
            ]
        )

        representation_record[
            f"{name}_pc1_variance_fraction"
        ] = float(
            metrics[
                "pc1_variance_fraction"
            ]
        )


    record = {
        "seed":
            seed,

        "run_dir":
            str(
                run_dir
            ),

        "acceptance_sha256":
            actual_acceptance_sha,

        "best_epoch":
            best_epoch,

        "last_epoch":
            last_epoch,

        "epochs_completed":
            epochs_completed,

        "stopped_early":
            stopped_early,

        "best_validation_loss":
            best_validation_loss,

        **representation_record,
    }


    records.append(
        record
    )


    print(
        "ACCEPTANCE_SHA256 =",
        actual_acceptance_sha,
    )

    print(
        "BEST_EPOCH =",
        best_epoch,
    )

    print(
        "EPOCHS_COMPLETED =",
        epochs_completed,
    )

    print(
        "BEST_VALIDATION_LOSS =",
        best_validation_loss,
    )

    print(
        f"B{seed}_AGGREGATION_INPUT = PASS"
    )


# =============================================================================
# 7. FIVE-SEED IDENTITY
# =============================================================================

assert len(records) == 5

assert {
    int(x["seed"])
    for x in records
} == set(PRODUCTIVE_SEEDS)

assert len({
    x["run_dir"]
    for x in records
}) == 5


print()
print(
    "FIVE_ACCEPTED_RUNS_VERIFIED = PASS"
)


# =============================================================================
# 8. AGGREGATE STATISTICS
# =============================================================================

metrics_to_summarize = [

    "best_epoch",
    "epochs_completed",
    "best_validation_loss",

    "r_delta_effective_rank",
    "r_delta_pc1_variance_fraction",

    "z_delta_effective_rank",
    "z_delta_pc1_variance_fraction",

    "z_instance_pair_effective_rank",
    "z_instance_pair_pc1_variance_fraction",
]


statistics_summary = {}


for metric in metrics_to_summarize:

    statistics_summary[
        metric
    ] = summarize(
        [
            record[
                metric
            ]
            for record in records
        ]
    )


print()
print("=" * 80)
print("FIVE-SEED STATISTICAL SUMMARY")
print("=" * 80)


for metric in metrics_to_summarize:

    s = statistics_summary[
        metric
    ]

    print(
        f"{metric}: "
        f"mean={s['mean']:.8g} "
        f"sd={s['sd']:.8g} "
        f"median={s['median']:.8g} "
        f"range=[{s['min']:.8g}, {s['max']:.8g}]"
    )


# =============================================================================
# 9. BUILD PERSISTENT AGGREGATION PAYLOAD
# =============================================================================

aggregation_payload = {

    "status":
        "PASS",

    "stage":
        "MODEL_B_A9_FAST_B6",

    "architecture":
        MODEL_B_ARCHITECTURE,

    "scientific_commit":
        EXPECTED_SHA,

    "split_seed":
        42,

    "run_seeds":
        list(
            PRODUCTIVE_SEEDS
        ),

    "number_of_runs":
        5,

    "all_runs_completed":
        True,

    "all_runs_accepted":
        True,

    "all_runs_stopped_early":
        all(
            record[
                "stopped_early"
            ]
            for record in records
        ),

    "per_seed":
        records,

    "statistics":
        statistics_summary,

    "representation_acceptance": {
        "r_delta": {
            "shape":
                [483, 640],
            "all_finite":
                True,
            "all_unique_rows":
                483,
            "all_noncollapsed":
                True,
        },

        "z_delta": {
            "shape":
                [483, 128],
            "all_finite":
                True,
            "all_unique_rows":
                483,
            "all_noncollapsed":
                True,
        },

        "z_instance_pair": {
            "shape":
                [483, 64],
            "all_finite":
                True,
            "all_unique_rows":
                483,
            "all_noncollapsed":
                True,
        },
    },

    "scope_note": (
        "FAST-B6 aggregates accepted run-level and "
        "representation-QC metrics. Cross-seed geometric "
        "stability of the full 483xD representations is "
        "not inferred from these summary metrics and will "
        "be evaluated separately from best.pt."
    ),
}


# =============================================================================
# 10. ISOLATED OUTPUT ROOT
# =============================================================================

FAST_B6_ROOT = (
    MODEL_B_A9_ROOT
    / "fast_b6_multiseed_aggregation"
)

FAST_B6_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


JSON_OUT = (
    FAST_B6_ROOT
    / "model_b_a9_multiseed_summary.json"
)

TSV_OUT = (
    FAST_B6_ROOT
    / "model_b_a9_multiseed_per_seed.tsv"
)


# Fail closed: never silently overwrite an existing aggregation.

assert not JSON_OUT.exists(), (
    f"STOP: already exists: {JSON_OUT}"
)

assert not TSV_OUT.exists(), (
    f"STOP: already exists: {TSV_OUT}"
)


# =============================================================================
# 11. WRITE JSON ATOMICALLY
# =============================================================================

json_text = json.dumps(
    aggregation_payload,
    indent=2,
    sort_keys=True,
) + "\n"


atomic_write_text(
    JSON_OUT,
    json_text,
)


# =============================================================================
# 12. WRITE TSV ATOMICALLY
# =============================================================================

fieldnames = [
    "seed",
    "run_dir",
    "acceptance_sha256",
    "best_epoch",
    "last_epoch",
    "epochs_completed",
    "stopped_early",
    "best_validation_loss",

    "r_delta_effective_rank",
    "r_delta_pc1_variance_fraction",

    "z_delta_effective_rank",
    "z_delta_pc1_variance_fraction",

    "z_instance_pair_effective_rank",
    "z_instance_pair_pc1_variance_fraction",
]


buffer = io.StringIO()

writer = csv.DictWriter(
    buffer,
    fieldnames=fieldnames,
    delimiter="\t",
    lineterminator="\n",
)

writer.writeheader()

for record in records:

    writer.writerow(
        {
            key:
                record[key]
            for key in fieldnames
        }
    )


atomic_write_text(
    TSV_OUT,
    buffer.getvalue(),
)


# =============================================================================
# 13. VERIFY PERSISTED OUTPUTS
# =============================================================================

assert JSON_OUT.is_file()
assert TSV_OUT.is_file()

assert JSON_OUT.stat().st_size > 0
assert TSV_OUT.stat().st_size > 0


persisted_json = load_json(
    JSON_OUT
)

assert (
    persisted_json
    == aggregation_payload
)


JSON_SHA256 = sha256_file(
    JSON_OUT
)

TSV_SHA256 = sha256_file(
    TSV_OUT
)


print()
print("=" * 80)
print("PERSISTED FAST-B6 OUTPUTS")
print("=" * 80)

print(
    "JSON_OUT =",
    JSON_OUT,
)

print(
    "JSON_SHA256 =",
    JSON_SHA256,
)

print()

print(
    "TSV_OUT =",
    TSV_OUT,
)

print(
    "TSV_SHA256 =",
    TSV_SHA256,
)


# =============================================================================
# 14. FINAL
# =============================================================================

FAST_B6_AGGREGATION_PASS = True


print()
print("=" * 80)
print("FAST-B6 — FINAL")
print("=" * 80)

print(
    "B11_AGGREGATION = PASS"
)

print(
    "B23_AGGREGATION = PASS"
)

print(
    "B37_AGGREGATION = PASS"
)

print(
    "B41_AGGREGATION = PASS"
)

print(
    "B53_AGGREGATION = PASS"
)

print()

print(
    "FIVE_ACCEPTED_RUNS_VERIFIED = PASS"
)

print(
    "FIVE_SEED_STATISTICAL_AGGREGATION = PASS"
)

print(
    "MULTISEED_JSON_PERSISTED = PASS"
)

print(
    "MULTISEED_TSV_PERSISTED = PASS"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_CHECKPOINT_MODIFIED = TRUE"
)

print(
    "NO_ACCEPTANCE_MODIFIED = TRUE"
)

print(
    "RUN_PRODUCTIVE_QUEUE = False"
)

print()

print(
    "MODEL_B_A9_MULTISEED_BASELINE = CLOSED"
)

print(
    "READY_FOR_CROSS_SEED_REPRESENTATION_STABILITY = YES"
)

print("=" * 80)

MODEL B A9 — FAST-B6 FIVE-SEED AGGREGATION
SCIENTIFIC_COMMIT = ed9dc23d7936a77a33bc40b2ed3400d43f1f9111
SCIENTIFIC_CHECKOUT = PASS

--------------------------------------------------------------------------------
AGGREGATION INPUT — B11
--------------------------------------------------------------------------------
ACCEPTANCE_SHA256 = c9285e995fa069ec7fe9622909918f4fac024dda8f3eea7ac6b137c08f2641e2
BEST_EPOCH = 24
EPOCHS_COMPLETED = 39
BEST_VALIDATION_LOSS = 0.3051567841798831
B11_AGGREGATION_INPUT = PASS

--------------------------------------------------------------------------------
AGGREGATION INPUT — B23
--------------------------------------------------------------------------------
ACCEPTANCE_SHA256 = d47179e2d0746851de4eed4f48e190740147006b37140f11dedb1129bca233a6
BEST_EPOCH = 38
EPOCHS_COMPLETED = 53
BEST_VALIDATION_LOSS = 0.24280673418289575
B23_AGGREGATION_INPUT = PASS

--------------------------------------------------------------------------------
AGGREGATION INPUT — B37


# Estabilidad geométrica multiseed de B

In [ ]:
# =============================================================================
# MODEL B — FINAL FIVE-SEED REEXTRACTION + GEOMETRY
# CPU POST-TRAINING PATH
#
# NO TRAINING
# NO RESUME
# NO MODIFICATION OF PRODUCTIVE RUNS
# =============================================================================

from pathlib import Path
from copy import deepcopy
import csv
import gc
import hashlib
import json
import os
import subprocess
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader


print("=" * 80)
print("MODEL B — FIVE-SEED REPRESENTATION RECOVERY")
print("=" * 80)


# =============================================================================
# 0. CONTRACT
# =============================================================================

SCIENTIFIC_COMMIT = (
    "ed9dc23d7936a77a33bc40b2ed3400d43f1f9111"
)

ANALYSIS_COMMIT = (
    "4af06bd729b5ef04111852482aebe63c9363f543"
)

PRODUCTIVE_SEEDS = (
    11, 23, 37, 41, 53
)

RUN_PRODUCTIVE_QUEUE = False
assert RUN_PRODUCTIVE_QUEUE is False

REPO = Path(REPO)
SRC = REPO / "src"

MUTANTS_HDF5 = Path(MUTANTS_HDF5)
WT_HDF5 = Path(WT_HDF5)

assert REPO.is_dir()
assert SRC.is_dir()
assert MUTANTS_HDF5.is_file()
assert WT_HDF5.is_file()

DEVICE = torch.device("cpu")

print("DEVICE =", DEVICE)
print("PYTORCH =", torch.__version__)
print("MUTANTS =", MUTANTS_HDF5)
print("WT =", WT_HDF5)


# =============================================================================
# 1. VERIFY ANALYSIS BRANCH DID NOT CHANGE SCIENTIFIC SOURCE
# =============================================================================

result = subprocess.run(
    [
        "git",
        "-C", str(REPO),
        "diff",
        "--quiet",
        SCIENTIFIC_COMMIT,
        ANALYSIS_COMMIT,
        "--",
        "src",
        "configs",
        "splits",
    ],
)

assert result.returncode == 0, (
    "Scientific source differs from training commit."
)

print("SCIENTIFIC_SOURCE_IDENTITY = PASS")


# =============================================================================
# 2. IMPORT PATH + PROJECT IMPORTS
# =============================================================================

os.chdir(REPO)

for candidate in (
    str(REPO),
    str(SRC),
):
    while candidate in sys.path:
        sys.path.remove(candidate)

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(SRC))

from gnn_siamese.config import (
    _load_config_unvalidated,
    validate_c1_config,
)

from gnn_siamese.builders import (
    build_dataset_bundle,
    build_model,
)

from gnn_siamese.data import (
    collate_mut_wt_pairs,
)

from gnn_siamese.training import (
    load_checkpoint,
)

from gnn_siamese.training.a9_contract import (
    _audit_representation,
)

from scripts.model_b_multiseed_geometry import (
    geometry_analysis,
)


print("PROJECT_IMPORTS = PASS")


# =============================================================================
# 3. OUTPUT
# =============================================================================

DRIVE_PROJECT_BASE = Path(
    "/content/drive/MyDrive/modelos_proyecto_PKP2"
)

OUTPUT_BASE = (
    DRIVE_PROJECT_BASE
    / "model_b"
    / "analysis"
    / "model_b_a9_multiseed_geometry_final"
)

OUTPUT_DIR = OUTPUT_BASE

suffix = 1

while OUTPUT_DIR.exists():
    OUTPUT_DIR = Path(
        f"{OUTPUT_BASE}_run{suffix}"
    )
    suffix += 1

EMBEDDINGS_DIR = (
    OUTPUT_DIR
    / "01_embeddings"
)

EMBEDDINGS_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

print("OUTPUT_DIR =", OUTPUT_DIR)


# =============================================================================
# 4. HELPERS
# =============================================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def read_json(path):

    obj = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    assert isinstance(obj, dict)

    return obj


# =============================================================================
# 5. IMMUTABLE INPUT HASHES
# =============================================================================

acceptance_hash_before = {}
checkpoint_hash_before = {}

for seed in PRODUCTIVE_SEEDS:

    run_dir = Path(RUNS[seed])

    acceptance = (
        run_dir
        / "a9_acceptance.json"
    )

    best = (
        run_dir
        / "checkpoints"
        / "best.pt"
    )

    assert acceptance.is_file()
    assert best.is_file()

    acceptance_hash_before[seed] = (
        sha256_file(acceptance)
    )

    checkpoint_hash_before[seed] = (
        sha256_file(best)
    )

print("PRODUCTIVE_INPUT_HASHES_CAPTURED = PASS")


# =============================================================================
# 6. FIVE-SEED EXTRACTION
# =============================================================================

reference_ids = None
manifest_records = []
metric_comparison = []


for seed in PRODUCTIVE_SEEDS:

    print()
    print("=" * 80)
    print(f"B{seed}")
    print("=" * 80)

    run_dir = Path(
        RUNS[seed]
    )

    config_path = (
        run_dir
        / "config_resolved.yaml"
    )

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    acceptance_path = (
        run_dir
        / "a9_acceptance.json"
    )

    best_path = (
        run_dir
        / "checkpoints"
        / "best.pt"
    )


    # -------------------------------------------------------------------------
    # Run identity
    # -------------------------------------------------------------------------

    manifest = read_json(
        manifest_path
    )

    acceptance = read_json(
        acceptance_path
    )

    assert manifest["status"] == "completed"

    assert (
        manifest["architecture"]
        == "model_b_graph_level_relational"
    )

    assert (
        int(
            manifest["configuration"]["seed"]
        )
        == seed
    )

    assert acceptance["status"] == "accepted"

    assert (
        int(
            acceptance["run_seed"]
        )
        == seed
    )

    assert (
        acceptance[
            "automatic_rejection_checks"
        ]
        == "PASS"
    )

    print("RUN_IDENTITY = PASS")


    # -------------------------------------------------------------------------
    # Resolved scientific config
    #
    # IMPORTANT:
    # The productive run was executed with training.device=cuda.
    # Read it first WITHOUT runtime validation, verify the original scientific
    # identity, and only then create an ephemeral CPU copy for post-training
    # inference. The original config_resolved.yaml is NEVER modified.
    # -------------------------------------------------------------------------

    original_config = (
        _load_config_unvalidated(
            config_path
        )
    )

    # Verify the exact identity of the productive run before any
    # operational adaptation.
    assert (
        original_config[
            "model"
        ][
            "architecture"
        ]
        == "model_b_graph_level_relational"
    )

    assert (
        int(
            original_config[
                "project"
            ][
                "seed"
            ]
        )
        == seed
    )

    assert (
        int(
            original_config[
                "split"
            ][
                "seed"
            ]
        )
        == 42
    )

    # The productive training run was intentionally CUDA-based.
    assert (
        str(
            original_config[
                "training"
            ][
                "device"
            ]
        ).lower()
        == "cuda"
    )

    print(
        "ORIGINAL_RUN_DEVICE =",
        original_config[
            "training"
        ][
            "device"
        ],
    )

    # -------------------------------------------------------------------------
    # Ephemeral CPU analysis copy
    # -------------------------------------------------------------------------

    config = deepcopy(
        original_config
    )

    # Operational adaptation ONLY.
    # No model architecture, weights, run seed, split seed, loss,
    # hyperparameter, checkpoint or productive artifact is changed.
    config[
        "training"
    ][
        "device"
    ] = "cpu"

    config[
        "paths"
    ][
        "mutants_hdf5"
    ] = str(
        MUTANTS_HDF5
    )

    config[
        "paths"
    ][
        "wt_companion_hdf5"
    ] = str(
        WT_HDF5
    )

    config[
        "split"
    ][
        "persist_path"
    ] = str(
        (
            REPO
            / "splits"
            / "leave_position_out_seed_42.json"
        ).resolve()
    )

    config[
        "split"
    ][
        "allow_create"
    ] = False

    # Validate only the ephemeral CPU analysis copy.
    config = validate_c1_config(
        config
    )

    assert (
        config[
            "training"
        ][
            "device"
        ]
        == "cpu"
    )

    assert (
        config[
            "model"
        ][
            "architecture"
        ]
        == "model_b_graph_level_relational"
    )

    assert (
        int(
            config[
                "project"
            ][
                "seed"
            ]
        )
        == seed
    )

    assert (
        int(
            config[
                "split"
            ][
                "seed"
            ]
        )
        == 42
    )

    assert (
        config[
            "split"
        ][
            "allow_create"
        ]
        is False
    )

    print(
        "POST_TRAINING_ANALYSIS_DEVICE = cpu"
    )

    print(
        "CPU_RUNTIME_CONFIG_ADAPTATION = PASS"
    )


    # -------------------------------------------------------------------------
    # Biological dataset
    # -------------------------------------------------------------------------

    dataset_bundle = (
        build_dataset_bundle(
            config
        )
    )

    dataset = (
        dataset_bundle.dataset
    )

    assert len(dataset.pairs) == 483

    assert (
        len(
            dataset.native_wt_controls
        )
        == 1
    )

    print("DATASET_REBUILD = PASS")


    # -------------------------------------------------------------------------
    # Model
    # -------------------------------------------------------------------------

    model = build_model(
        config,
        dataset,
    )

    assert (
        model.__class__.__name__
        == "ModelBGraphLevelRelationalContrastive"
    )

    model = model.to(DEVICE)


    # -------------------------------------------------------------------------
    # BEST checkpoint
    # -------------------------------------------------------------------------

    checkpoint = load_checkpoint(
        best_path,
        map_location="cpu",
    )

    assert (
        int(
            checkpoint["seed"]
        )
        == seed
    )

    assert (
        checkpoint["architecture"]
        == "model_b_graph_level_relational"
    )

    load_status = (
        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ],
            strict=True,
        )
    )

    assert (
        list(
            load_status.missing_keys
        )
        == []
    )

    assert (
        list(
            load_status.unexpected_keys
        )
        == []
    )

    model.eval()

    print(
        "BEST_EPOCH =",
        checkpoint["epoch_completed"],
    )

    print("STRICT_STATE_DICT_LOAD = PASS")


    # -------------------------------------------------------------------------
    # Same deterministic FAST-B5 loader
    # -------------------------------------------------------------------------

    loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=(
            collate_mut_wt_pairs
        ),
    )

    assert len(loader) == 121


    chunks = {
        "r_delta": [],
        "z_delta": [],
        "z_instance_pair": [],
    }

    variant_ids = []


    with torch.inference_mode():

        for batch in loader:

            variant_ids.extend(
                list(
                    batch.variant_ids
                )
            )

            batch = batch.to(
                DEVICE
            )

            output = (
                model.siamese_model(
                    graph_mut=(
                        batch.graph_mut
                    ),
                    graph_wt=(
                        batch.graph_wt
                    ),
                    allow_trainable_z_delta=True,
                )
            )

            tensors = {
                "r_delta":
                    output.r_delta,

                "z_delta":
                    output.z_delta,

                "z_instance_pair":
                    output.z_instance_pair,
            }

            for name, tensor in tensors.items():

                assert tensor is not None

                chunks[name].append(
                    tensor
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(
                        np.float32,
                        copy=False,
                    )
                )


    representations = {
        name:
            np.concatenate(
                parts,
                axis=0,
            )

        for name, parts
        in chunks.items()
    }


    # -------------------------------------------------------------------------
    # Basic QC
    # -------------------------------------------------------------------------

    assert len(variant_ids) == 483
    assert len(set(variant_ids)) == 483

    if reference_ids is None:

        reference_ids = tuple(
            variant_ids
        )

    else:

        assert (
            tuple(variant_ids)
            == reference_ids
        ), (
            f"B{seed}: variant alignment differs"
        )


    expected_shapes = {
        "r_delta":
            (483, 640),

        "z_delta":
            (483, 128),

        "z_instance_pair":
            (483, 64),
    }


    for name, matrix in representations.items():

        assert (
            matrix.shape
            == expected_shapes[name]
        )

        assert (
            matrix.dtype
            == np.float32
        )

        assert np.isfinite(matrix).all()

        assert (
            np.unique(
                matrix,
                axis=0,
            ).shape[0]
            == 483
        )

        print(
            name,
            matrix.shape,
            "finite=PASS unique=483",
        )


    print("REPRESENTATION_EXTRACTION = PASS")


    # -------------------------------------------------------------------------
    # Compare with existing A9 acceptance.
    #
    # CPU vs original CUDA inference may produce tiny floating differences.
    # We RECORD them descriptively rather than inventing a scientific
    # accept/reject threshold.
    # -------------------------------------------------------------------------

    stored = (
        acceptance[
            "representation_metrics"
        ]
    )


    for name, matrix in representations.items():

        current = (
            _audit_representation(
                name,
                matrix,
            )
        )

        previous = stored[name]

        assert current["finite"] is True

        assert (
            int(
                current[
                    "exact_unique_rows"
                ]
            )
            == 483
        )


        er_diff = abs(
            float(
                current[
                    "effective_rank"
                ]
            )
            -
            float(
                previous[
                    "effective_rank"
                ]
            )
        )

        pc1_diff = abs(
            float(
                current[
                    "pc1_variance_fraction"
                ]
            )
            -
            float(
                previous[
                    "pc1_variance_fraction"
                ]
            )
        )


        metric_comparison.append(
            {
                "seed":
                    seed,

                "representation":
                    name,

                "effective_rank_acceptance":
                    float(
                        previous[
                            "effective_rank"
                        ]
                    ),

                "effective_rank_cpu":
                    float(
                        current[
                            "effective_rank"
                        ]
                    ),

                "effective_rank_abs_diff":
                    er_diff,

                "pc1_acceptance":
                    float(
                        previous[
                            "pc1_variance_fraction"
                        ]
                    ),

                "pc1_cpu":
                    float(
                        current[
                            "pc1_variance_fraction"
                        ]
                    ),

                "pc1_abs_diff":
                    pc1_diff,
            }
        )


        print(
            name,
            "effective_rank_diff =",
            f"{er_diff:.3e}",
            "| PC1_diff =",
            f"{pc1_diff:.3e}",
        )


    # -------------------------------------------------------------------------
    # Persist NPZ for geometry
    # -------------------------------------------------------------------------

    npz_path = (
        EMBEDDINGS_DIR
        / f"seed_{seed}_representations.npz"
    )


    np.savez_compressed(

        npz_path,

        variant_id=np.asarray(
            variant_ids,
            dtype="U",
        ),

        r_delta=(
            representations[
                "r_delta"
            ]
        ),

        z_delta=(
            representations[
                "z_delta"
            ]
        ),

        z_instance_pair=(
            representations[
                "z_instance_pair"
            ]
        ),
    )


    assert npz_path.is_file()


    # -------------------------------------------------------------------------
    # Productive artifact immutability
    # -------------------------------------------------------------------------

    assert (
        sha256_file(
            acceptance_path
        )
        ==
        acceptance_hash_before[
            seed
        ]
    )

    assert (
        sha256_file(
            best_path
        )
        ==
        checkpoint_hash_before[
            seed
        ]
    )


    manifest_records.append(
        {
            "seed":
                seed,

            "best_epoch":
                int(
                    checkpoint[
                        "epoch_completed"
                    ]
                ),

            "run_dir":
                str(run_dir),

            "best_pt_sha256":
                checkpoint_hash_before[
                    seed
                ],

            "acceptance_sha256":
                acceptance_hash_before[
                    seed
                ],

            "npz":
                str(npz_path),

            "npz_sha256":
                sha256_file(
                    npz_path
                ),
        }
    )


    print("PRODUCTIVE_ARTIFACTS_UNCHANGED = PASS")
    print("NPZ_EXPORT = PASS")


    del output
    del representations
    del chunks
    del loader
    del model
    del checkpoint
    del dataset
    del dataset_bundle

    gc.collect()


# =============================================================================
# 7. FIVE-SEED EXTRACTION FINAL
# =============================================================================

assert reference_ids is not None
assert len(reference_ids) == 483

print()
print("=" * 80)
print("FIVE-SEED EXTRACTION — FINAL")
print("=" * 80)

print("FIVE_SEED_REPRESENTATION_EXTRACTION = PASS")
print("REPRESENTATION_ID_ALIGNMENT_GATE = PASS")
print("REPRESENTATION_FINITE_VALUES_GATE = PASS")


# =============================================================================
# 8. PERSIST EXTRACTION AUDIT
# =============================================================================

manifest_out = (
    EMBEDDINGS_DIR
    / "representation_export_manifest.json"
)

manifest_out.write_text(
    json.dumps(
        {
            "status":
                "PASS",

            "scientific_commit":
                SCIENTIFIC_COMMIT,

            "analysis_commit":
                ANALYSIS_COMMIT,

            "device":
                "cpu",

            "run_seeds":
                list(
                    PRODUCTIVE_SEEDS
                ),

            "n_variants":
                483,

            "gates": {
                "BEST_CHECKPOINT_IDENTITY_GATE":
                    "PASS",

                "REPRESENTATION_EXTRACTION_GATE":
                    "PASS",

                "REPRESENTATION_ID_ALIGNMENT_GATE":
                    "PASS",

                "REPRESENTATION_FINITE_VALUES_GATE":
                    "PASS",
            },

            "records":
                manifest_records,
        },
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


metric_out = (
    EMBEDDINGS_DIR
    / "cpu_vs_acceptance_metrics.tsv"
)


with metric_out.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:

    fields = [
        "seed",
        "representation",
        "effective_rank_acceptance",
        "effective_rank_cpu",
        "effective_rank_abs_diff",
        "pc1_acceptance",
        "pc1_cpu",
        "pc1_abs_diff",
    ]

    writer = csv.DictWriter(
        handle,
        fieldnames=fields,
        delimiter="\t",
    )

    writer.writeheader()

    writer.writerows(
        metric_comparison
    )


print("EXTRACTION_AUDIT_PERSISTED = PASS")


# =============================================================================
# 9. GEOMETRIC STABILITY
# =============================================================================

print()
print("=" * 80)
print("GEOMETRIC STABILITY")
print("=" * 80)


geometry_report = geometry_analysis(

    embeddings_dir=(
        EMBEDDINGS_DIR
    ),

    output_dir=(
        OUTPUT_DIR
    ),

    knn_k=(
        5,
        10,
        20,
    ),
)


gates = (
    geometry_report[
        "gates"
    ]
)


for key, value in gates.items():

    print(
        key,
        "=",
        value,
    )


assert (
    gates[
        "GEOMETRIC_EXTRACTION_GATE"
    ]
    == "PASS"
)

assert (
    gates[
        "GEOMETRIC_ID_ALIGNMENT_GATE"
    ]
    == "PASS"
)

assert (
    gates[
        "GEOMETRIC_FINITE_VALUES_GATE"
    ]
    == "PASS"
)

assert (
    gates[
        "GEOMETRIC_STABILITY_GATE"
    ]
    == "PASS_DESCRIPTIVE"
)


# =============================================================================
# 10. FINAL IMMUTABILITY
# =============================================================================

for seed in PRODUCTIVE_SEEDS:

    run_dir = Path(
        RUNS[seed]
    )

    assert (
        sha256_file(
            run_dir
            / "a9_acceptance.json"
        )
        ==
        acceptance_hash_before[
            seed
        ]
    )

    assert (
        sha256_file(
            run_dir
            / "checkpoints"
            / "best.pt"
        )
        ==
        checkpoint_hash_before[
            seed
        ]
    )


print("ALL_ACCEPTANCES_UNCHANGED = PASS")
print("ALL_BEST_CHECKPOINTS_UNCHANGED = PASS")


# =============================================================================
# FINAL
# =============================================================================

print()
print("=" * 80)
print("MODEL B — FINAL")
print("=" * 80)

print(
    "MODEL_B_A9_MULTISEED_BASELINE = CLOSED"
)

print(
    "FIVE_SEED_REPRESENTATION_EXTRACTION = PASS"
)

print(
    "GEOMETRIC_STABILITY_GATE = PASS_DESCRIPTIVE"
)

print(
    "NO_TRAINING_EXECUTED = TRUE"
)

print(
    "NO_RESUME_EXECUTED = TRUE"
)

print(
    "NO_CHECKPOINT_MODIFIED = TRUE"
)

print(
    "NO_ACCEPTANCE_MODIFIED = TRUE"
)

print(
    "OUTPUT_DIR =",
    OUTPUT_DIR,
)

print(
    "READY_FOR_MODEL_A_AND_AB_COMPARISON = YES"
)

print("=" * 80)

KeyboardInterrupt: 

# Guardar notebook